<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## بازبینی مجدد پس از Final Health — بررسی متن اصلی Case 4 از 22

در بازبینی مجدد Human Review مشخص شد سؤال پذیرفته‌شده قبلی درباره «تمرین 8» مستقل و روشن نیست و Target Evidence فعلی نیز فقط شامل عنوان تمرین و یک دستور کوتاه است.

پیش از هر تصمیم REVISE یا RETARGET، باید متن واقعی همان کتاب در اطراف پاراگراف‌های 329 و 330 بررسی شود تا مشخص شود:

- تمرین 8 ادامه چه مبحثی بوده است؛
- هدف آموزشی آن از متن قبل یا بعد قابل استنباط هست یا نه؛
- آیا می‌توان از همان منبع سؤال طبیعی و مفهومی ساخت؛
- یا باید Target Evidence به بخش مناسب‌تری از همان موضوع تغییر کند.

این مرحله فقط Diagnostic است و هیچ Human Decision، Registry، Source، Citation یا Queue را تغییر نمی‌دهد.

</div>

In [1]:
# Code Cell 105

from pathlib import Path
import json
import re


CELL105_PILOT_ID = (
    "question-pilot:"
    "909b695797935246c92c7a66ebe0f773"
)

CELL105_WORK_ID = (
    "book:trainer-training-financial-literacy"
)

CELL105_EDITION_ID = (
    "book:trainer-training-financial-literacy:fa-source"
)

CELL105_TARGET_PARAGRAPHS = {
    329,
    330,
}

CELL105_CONTEXT_START = 315
CELL105_CONTEXT_END = 345


cell105_failures = []


def cell105_check(name, condition):
    condition = bool(condition)

    if not condition:
        cell105_failures.append(
            name
        )

    return condition


def cell105_find_repo_root():
    candidates = []

    if "cell21_registry_path" in globals():
        registry_path = Path(
            cell21_registry_path
        ).resolve()

        candidates.extend(
            [
                registry_path.parent,
                *registry_path.parents,
            ]
        )

    cwd = Path.cwd().resolve()

    candidates.extend(
        [
            cwd,
            *cwd.parents,
        ]
    )

    seen = set()

    for candidate in candidates:
        candidate = candidate.resolve()

        if str(candidate) in seen:
            continue

        seen.add(
            str(candidate)
        )

        if (
            (
                candidate
                /
                "data"
            ).exists()
            and
            (
                candidate
                /
                "notebooks"
            ).exists()
        ):
            return candidate

    return None


def cell105_extract_paragraph_number(record):
    if not isinstance(
        record,
        dict,
    ):
        return None

    direct = record.get(
        "paragraph_number"
    )

    if isinstance(
        direct,
        int,
    ):
        return direct

    locator = record.get(
        "locator"
    )

    if isinstance(
        locator,
        dict,
    ):
        value = locator.get(
            "paragraph_number"
        )

        if isinstance(
            value,
            int,
        ):
            return value

    paragraph_id = (
        record.get(
            "paragraph_id"
        )
        or
        record.get(
            "document_id"
        )
    )

    if isinstance(
        paragraph_id,
        str,
    ):
        match = re.search(
            r":p0*(\d+)$",
            paragraph_id,
        )

        if match:
            return int(
                match.group(
                    1
                )
            )

    return None


def cell105_record_work_id(record):
    if not isinstance(
        record,
        dict,
    ):
        return None

    value = record.get(
        "work_id"
    )

    if isinstance(
        value,
        str,
    ):
        return value

    locator = record.get(
        "locator"
    )

    if isinstance(
        locator,
        dict,
    ):
        value = locator.get(
            "work_id"
        )

        if isinstance(
            value,
            str,
        ):
            return value

    return None


def cell105_record_edition_id(record):
    if not isinstance(
        record,
        dict,
    ):
        return None

    value = (
        record.get(
            "edition_id"
        )
        or
        record.get(
            "source_id"
        )
    )

    if isinstance(
        value,
        str,
    ):
        return value

    locator = record.get(
        "locator"
    )

    if isinstance(
        locator,
        dict,
    ):
        value = locator.get(
            "edition_id"
        )

        if isinstance(
            value,
            str,
        ):
            return value

    return None


def cell105_record_text(record):
    if not isinstance(
        record,
        dict,
    ):
        return None

    for key in [
        "text",
        "normalized_text",
        "content",
        "paragraph_text",
    ]:
        value = record.get(
            key
        )

        if (
            isinstance(
                value,
                str,
            )
            and
            value.strip()
        ):
            return value.strip()

    return None


def cell105_heading_path(record):
    if not isinstance(
        record,
        dict,
    ):
        return None

    value = record.get(
        "heading_path"
    )

    if isinstance(
        value,
        list,
    ):
        return value

    locator = record.get(
        "locator"
    )

    if isinstance(
        locator,
        dict,
    ):
        value = locator.get(
            "heading_path"
        )

        if isinstance(
            value,
            list,
        ):
            return value

    return None


def cell105_walk(obj):
    if isinstance(
        obj,
        dict,
    ):
        yield obj

        for value in obj.values():
            yield from cell105_walk(
                value
            )

    elif isinstance(
        obj,
        list,
    ):
        for item in obj:
            yield from cell105_walk(
                item
            )


cell105_repo_root = (
    cell105_find_repo_root()
)

cell105_check(
    "REPO_ROOT_NOT_FOUND",
    cell105_repo_root
    is not None,
)


cell105_registry_path = (
    (
        Path(
            cell21_registry_path
        ).resolve()
    )
    if "cell21_registry_path" in globals()
    else
    (
        cell105_repo_root
        /
        "data"
        /
        "phase2"
        /
        "human_review"
        /
        "human_review_registry.json"
    )
)


cell105_book_files = [
    cell105_repo_root
    /
    "data"
    /
    "phase2"
    /
    "interim"
    /
    "book_ingested_v1.json",

    cell105_repo_root
    /
    "data"
    /
    "phase2"
    /
    "interim"
    /
    "book_ingested_v2.json",
]


cell105_existing_book_files = [
    path
    for path in cell105_book_files
    if path.exists()
]


cell105_check(
    "BOOK_INGESTED_FILES_NOT_FOUND",
    bool(
        cell105_existing_book_files
    ),
)


cell105_registry = json.loads(
    cell105_registry_path.read_text(
        encoding="utf-8"
    )
)


cell105_case_matches = [
    case
    for case
    in cell105_registry.get(
        "completed_human_review_cases",
        [],
    )
    if (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL105_PILOT_ID
    )
]


cell105_check(
    "PILOT_CASE_NOT_UNIQUE",
    len(
        cell105_case_matches
    )
    ==
    1,
)


cell105_case = (
    cell105_case_matches[
        0
    ]
    if len(
        cell105_case_matches
    )
    ==
    1
    else
    {}
)


cell105_current_question = None

for draft in (
    cell105_case.get(
        "model_drafts",
        []
    )
    or
    []
):
    if (
        isinstance(
            draft,
            dict,
        )
        and
        draft.get(
            "human_decision"
        )
        ==
        "ACCEPT"
    ):
        cell105_current_question = (
            draft.get(
                "question_text"
            )
        )


cell105_current_answer = (
    (
        cell105_case.get(
            "expected_answer_review"
        )
        or
        {}
    ).get(
        "expected_answer_text"
    )
)


cell105_current_evidence = (
    cell105_case.get(
        "source_evidence",
        []
    )
    or
    []
)


cell105_book_records = []


for book_file in cell105_existing_book_files:
    payload = json.loads(
        book_file.read_text(
            encoding="utf-8"
        )
    )

    for record in cell105_walk(
        payload
    ):
        paragraph_number = (
            cell105_extract_paragraph_number(
                record
            )
        )

        text = cell105_record_text(
            record
        )

        if (
            paragraph_number
            is None
            or
            text
            is None
        ):
            continue

        work_id = cell105_record_work_id(
            record
        )

        edition_id = (
            cell105_record_edition_id(
                record
            )
        )

        paragraph_id = (
            record.get(
                "paragraph_id"
            )
            or
            record.get(
                "document_id"
            )
        )

        identity_text = " ".join(
            str(value)
            for value in [
                work_id,
                edition_id,
                paragraph_id,
            ]
            if value
        )

        exact_source_match = any([
            work_id
            ==
            CELL105_WORK_ID,

            edition_id
            ==
            CELL105_EDITION_ID,

            CELL105_EDITION_ID
            in
            identity_text,
        ])

        if not exact_source_match:
            continue

        if not (
            CELL105_CONTEXT_START
            <=
            paragraph_number
            <=
            CELL105_CONTEXT_END
        ):
            continue

        cell105_book_records.append({
            "paragraph_number":
                paragraph_number,

            "text":
                text,

            "heading_path":
                cell105_heading_path(
                    record
                ),

            "paragraph_id":
                paragraph_id,

            "source_file":
                str(
                    book_file
                ),
        })


cell105_unique_records = {}

for record in cell105_book_records:
    key = (
        record[
            "paragraph_number"
        ],
        record[
            "text"
        ],
    )

    if key not in cell105_unique_records:
        cell105_unique_records[
            key
        ] = record


cell105_context_records = sorted(
    cell105_unique_records.values(),
    key=lambda item: (
        item[
            "paragraph_number"
        ],
        item[
            "text"
        ],
    ),
)


cell105_found_numbers = {
    record[
        "paragraph_number"
    ]
    for record in cell105_context_records
}


cell105_check(
    "TARGET_PARAGRAPHS_NOT_FOUND_IN_BOOK",
    CELL105_TARGET_PARAGRAPHS.issubset(
        cell105_found_numbers
    ),
)


cell105_check(
    "NO_CONTEXT_BEFORE_TARGET",
    any(
        number
        <
        min(
            CELL105_TARGET_PARAGRAPHS
        )
        for number
        in cell105_found_numbers
    ),
)


cell105_check(
    "NO_CONTEXT_AFTER_TARGET",
    any(
        number
        >
        max(
            CELL105_TARGET_PARAGRAPHS
        )
        for number
        in cell105_found_numbers
    ),
)


print(
    "=== CASE 4 POST-FINAL SOURCE CONTEXT DIAGNOSTIC ==="
)


print(
    "\n=== CURRENT ACCEPTED CASE ==="
)

print(
    "Pilot:",
    CELL105_PILOT_ID,
)

print(
    "Current Question:",
    cell105_current_question,
)

print(
    "Current Answer:",
    cell105_current_answer,
)

print(
    "Current Target Evidence:",
    cell105_case.get(
        "target_evidence_id"
    ),
)


print(
    "\n=== CURRENT PROTECTED SOURCE EVIDENCE ==="
)

for item in cell105_current_evidence:
    print(
        "\nDocument:",
        item.get(
            "document_id"
        ),
    )

    print(
        "Text:",
        item.get(
            "text"
        ),
    )


print(
    "\n=== EXACT BOOK CONTEXT ==="
)

print(
    "Work ID:",
    CELL105_WORK_ID,
)

print(
    "Edition ID:",
    CELL105_EDITION_ID,
)

print(
    "Requested paragraph window:",
    (
        CELL105_CONTEXT_START,
        CELL105_CONTEXT_END,
    ),
)

print(
    "Found unique records:",
    len(
        cell105_context_records
    ),
)


for record in cell105_context_records:
    print(
        "\n"
        +
        "="
        *
        90
    )

    print(
        "Paragraph:",
        record[
            "paragraph_number"
        ],
    )

    print(
        "Paragraph ID:",
        record[
            "paragraph_id"
        ],
    )

    print(
        "Heading Path:",
        record[
            "heading_path"
        ],
    )

    print(
        "Text:"
    )

    print(
        record[
            "text"
        ]
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    False,
)

print(
    "Human Decision added:",
    False,
)

print(
    "Question changed:",
    False,
)

print(
    "Answer changed:",
    False,
)

print(
    "Target changed:",
    False,
)

print(
    "Citation changed:",
    False,
)

print(
    "Raw Source mutated:",
    False,
)

print(
    "Parser mutated:",
    False,
)

print(
    "Commit / Push:",
    False,
)


cell105_gate = (
    len(
        cell105_failures
    )
    ==
    0
)


print(
    "\n=== FAILURES ==="
)

print(
    cell105_failures
)


print(
    "\nCODE CELL 105 GATE:",
    (
        "PASS"
        if cell105_gate
        else
        "FAIL"
    ),
)


assert cell105_gate, (
    "Case 4 source-context diagnostic failed. "
    "Repair this SAME Code Cell 105 before any re-review decision."
)

=== CASE 4 POST-FINAL SOURCE CONTEXT DIAGNOSTIC ===

=== CURRENT ACCEPTED CASE ===
Pilot: question-pilot:909b695797935246c92c7a66ebe0f773
Current Question: مدیریت پول‌توجیبی از ۵ تا ۱۸ سالگی چگونه تغییر می‌کند و در هر بازه سنی چه تقسیم‌بندی‌ای پیشنهاد شده است؟
Current Answer: در ۵ تا ۷ سالگی کودک ابتدا بیشتر پول را نگه می‌دارد و پس‌انداز می‌کند و به‌تدریج خرج‌کردن را در موقعیت‌های واقعی یاد می‌گیرد. در ۸ تا ۱۲ سالگی پول‌توجیبی به خرج و پس‌انداز تقسیم می‌شود و الگوی پیشنهادی ۵۰ درصد برای هر کدام است. در ۱۲ تا ۱۵ سالگی سرمایه‌گذاری هم اضافه می‌شود و الگوی ۵۰ درصد خرج، ۳۰ درصد پس‌انداز و ۲۰ درصد سرمایه‌گذاری پیشنهاد می‌شود. در ۱۵ تا ۱۸ سالگی هدیه یا قرض به دیگران نیز به بودجه اضافه می‌شود و الگوی ۴۰ درصد خرج، ۳۰ درصد پس‌انداز، ۲۰ درصد سرمایه‌گذاری و ۱۰ درصد هدیه به دیگران مطرح شده است.
Current Target Evidence: target-evidence:63c2f0f041fa8caa0f21a1aed165d7ec

=== CURRENT PROTECTED SOURCE EVIDENCE ===

Document: book:trainer-training-financial-literacy:fa-source:p000323
Text: کودکان بین پن

<div style="direction: rtl; text-align: right; white-space: normal; line-height: 1.8;">

## بازبینی دوم — مورد 4 از 22 — اصلاح کامل Target، سؤال و پاسخ

### 1. وضعیت ثبت‌شده در بازبینی اول

در Human Review اولیه، این Case با Target زیر پذیرفته شده بود:

**Target Evidence قبلی:**

`target-evidence:8d87341f7847558d90ee76e350c64fa3`

**متن منبع قبلی:**

**پاراگراف 329**

تمرین 8

**پاراگراف 330**

هر مقطع سنی دو نفر را انتخاب کنید و درباره میزان هزینه ماهانه آنها مصاحبه کنید. به آن‌ها اطمینان دهید که این اطلاعات محرمانه خواهد بود. نتیجه را تحلیل کنید.

**سؤال پذیرفته‌شده قبلی:**

در تمرین ۸، برای بررسی هزینه‌های ماهانه افراد چه مراحلی باید انجام شود؟

**پاسخ پذیرفته‌شده قبلی:**

باید از هر مقطع سنی دو نفر انتخاب شوند، درباره میزان هزینه ماهانه آن‌ها مصاحبه شود، به آن‌ها اطمینان داده شود که اطلاعاتشان محرمانه خواهد ماند و در پایان نتیجه مصاحبه‌ها تحلیل شود.

این سؤال و پاسخ در بازبینی اول ACCEPT شده بودند، اما در بازبینی دوم مشخص شد که:

- سؤال بدون مشاهده کتاب مستقل و طبیعی نیست؛
- عبارت «تمرین ۸» برای کاربر معنای مستقلی ندارد؛
- Target قبلی فقط عنوان و دستور یک تمرین را شامل می‌شود؛
- بخش اصلی و آموزشی موضوع بودجه‌بندی درست پیش از این تمرین قرار دارد؛
- بنابراین نگه‌داشتن این Case به شکل قبلی برای Golden Dataset مناسب نیست.

تصمیم بازبینی دوم:

**RETARGET**

نسخه قبلی پاک نمی‌شود و به‌عنوان تاریخچه بازبینی اول حفظ خواهد شد.

---

### 2. منبع بازبینی‌شده و جایگزین

در بررسی متن کامل کتاب، پاراگراف‌های 323 تا 327 به‌عنوان Target جایگزین انتخاب شدند.

#### پاراگراف 323

کودکان بین پنج تا هفت سال : اولین پولی که کودک دریافت می‌کند، تنها از آن نگهداری می‌کند و در واقع فقط پس‌انداز می‌کند و کم‌کم با موقعیت‌هایی که به او پول می‌دهیم (مانند فروشگاه یا مسافرت) خرج کردن را یاد می‌گیرد.

#### پاراگراف 324

کودکان بین هشت تا دوازده سال: دومین ردیف بودجه پول‌توجیبی هم یاد داده می‌شود. خرج پول با نظارت والدین است. هر چند به کودکان اجازهٔ اشتباه می‌دهند، اما همچنان کنارشان هستند و خرج پول با نظارت آن‌هاست. حتی اگر با خرید فرزند مخالف‌اند، اجازه می‌دهند بخرد. مهم این است که جلوی اشتباه گرفته نشود، بلکه بعد از اشتباه، خیلی واقعی درباره آن و چرایی اشتباه بودن آن گفتگو شود. در این سن، پول‌توجیبی فقط در دو ردیف خرج و پس‌انداز صرف می‌شود و می‌توان به هر کدام 50 درصد از دریافتی اختصاص یابد.

#### پاراگراف 325

اگر با این روش پیش رفته باشیم، دیگر «پول خودم است» مطرح نیست؛ چون پدر و مادر همواره با او هستند. هیچ وقت احساس نمی‌کند پول خودش است. اگر کودکان بگویند پول خودشان است،‌ می‌گوییم باید پاسخگوی تصمیم‌شان باشند. ممکن است تصمیم‌شان پیامدهایی داشته یا نداشته باشد. اگر پیامدها، فرصت آموزش را فراهم می‌کرد، تصمیم اشتباه و پیامدش را یادآوری می‌کنیم. یادمان باشد هنگام یادگیری راه رفتن، فرزند را برای زمین خوردن مواخذه نمی‌کنیم.

#### پاراگراف 326

نوجوان دوازده تا پانزده سال:در این دوره ردیف سرمایه‌گذاری هم اضافه می‌شود و پول‌توجیبی به سه بخش تقسیم می‌شود: خرج، پس‌انداز و سرمایه‌گذاری. در این حالت به ترتیب الگوی 50 درصد، 30 درصد و 20 درصد می‌تواند به کارگرفته شود.(تفاوت پس‌انداز و سرمایه‌گذاری: مشخص و نامشخص بودن هدف، بازه زمانی کوتاه مدت و بلندمدت). می‌توان در نشست خانوادگی دربارهٔ سرفصل‌های بودجه‌بندی و خرج با آن‌ها به توافق رسید. در این دوره می‌توان به شکل موقعیتی به او، هدیه به دیگران، بخشش یا حتی قرض به دیگران را پیشنهاد کرد.

#### پاراگراف 327

نوجوان پانزده تا هجده سال: در این دوره ردیف دیگری به پول‌توجیبی اضافه می‌شود: هدیه یا قرض به دیگران. تا پیش از این، به شکل موقعیتی با این ردیف آشنا بوده است. اما از اینجا به بعد با تعیین بخش و درصدی به این موضوع به آن ساختار می‌دهیم؛ الگوی 40 درصد خرج، 30 درصد پس‌انداز، 20 درصد سرمایه‌گذاری و 10 درصد هدیه به دیگران در سن مناسب است. نقش نظارتی والدین همچنان هست و آخر ماه از فرزندشان می‌خواهند توضیح دهد که این درصدها را چطور اعمال کرده است. این درصدها به نوعی خط‌کش است تا جلوی رفتارهای هیجانی نوجوان‌ها را بگیرد تا در هیچ کدام افراط و تفریط نداشته باشند؛ حتی در بخشش!

---

### 3. سؤال جایگزین

**مدیریت پول‌توجیبی از ۵ تا ۱۸ سالگی چگونه تغییر می‌کند و در هر بازه سنی چه تقسیم‌بندی‌ای پیشنهاد شده است؟**

Human Reviewer این سؤال را در بازبینی دوم تأیید کرد.

---

### 4. پاسخ جایگزین

**در ۵ تا ۷ سالگی کودک ابتدا بیشتر پول را نگه می‌دارد و پس‌انداز می‌کند و به‌تدریج خرج‌کردن را در موقعیت‌های واقعی یاد می‌گیرد. در ۸ تا ۱۲ سالگی پول‌توجیبی به خرج و پس‌انداز تقسیم می‌شود و الگوی پیشنهادی ۵۰ درصد برای هر کدام است. در ۱۲ تا ۱۵ سالگی سرمایه‌گذاری هم اضافه می‌شود و الگوی ۵۰ درصد خرج، ۳۰ درصد پس‌انداز و ۲۰ درصد سرمایه‌گذاری پیشنهاد می‌شود. در ۱۵ تا ۱۸ سالگی هدیه یا قرض به دیگران نیز به بودجه اضافه می‌شود و الگوی ۴۰ درصد خرج، ۳۰ درصد پس‌انداز، ۲۰ درصد سرمایه‌گذاری و ۱۰ درصد هدیه به دیگران مطرح شده است.**

این پاسخ فقط از پاراگراف‌های 323 تا 327 همان منبع ساخته شده و دانش بیرونی به آن افزوده نشده است.

---

### 5. تصمیم نهایی بازبینی دوم

- تصمیم درباره Target قبلی: **RETARGET**
- Target قبلی: **حفظ در History**
- سؤال قبلی: **حفظ در History**
- پاسخ قبلی: **حفظ در History**
- Source خام: **بدون تغییر**
- Parser Output: **بدون تغییر**
- Target جدید: **پاراگراف‌های 323 تا 327**
- سؤال جدید: **ACCEPT**
- پاسخ جدید: **ACCEPT**
- Citation جدید: **مبتنی بر پاراگراف‌های 323 تا 327**
- Author Final Review: **NOT_STARTED**
- Case پس از این اصلاح همچنان آماده بازبینی دکتر کمیل رودی است.

</div>

In [2]:
# Code Cell 106
# ============================================================
# SECOND HUMAN REVIEW — CASE 4 / 22
#
# COMPLETE RETARGET + QUESTION + ANSWER + CITATION AMENDMENT
#
# Prerequisite:
#   Cell 105 diagnostic must have PASSed.
#
# This cell:
# - preserves the previously accepted target/question/answer/citation;
# - records the second-review RETARGET;
# - activates book paragraphs 323-327 as replacement evidence;
# - records the newly approved Question;
# - records the replacement Expected Answer;
# - rebuilds citation mapping from the replacement evidence;
# - updates BOTH Registry copies of the case;
# - keeps Author Final Review = NOT_STARTED;
# - writes a backup before changing Registry;
# - reloads and verifies the persisted Registry;
# - is idempotent for this exact amendment.
#
# It DOES NOT:
# - modify raw book source;
# - modify parser output;
# - delete first-review history;
# - record any Author decision;
# - commit or push anything.
# ============================================================

from pathlib import Path
from copy import deepcopy
from datetime import datetime, timezone
import hashlib
import json


# ============================================================
# 1) Fixed identity
# ============================================================

CELL106_PILOT_ID = (
    "question-pilot:"
    "909b695797935246c92c7a66ebe0f773"
)

CELL106_OLD_TARGET_ID = (
    "target-evidence:"
    "8d87341f7847558d90ee76e350c64fa3"
)

CELL106_REPLACEMENT_PARAGRAPHS = [
    323,
    324,
    325,
    326,
    327,
]

CELL106_NEW_QUESTION = (
    "مدیریت پول‌توجیبی از ۵ تا ۱۸ سالگی چگونه تغییر می‌کند "
    "و در هر بازه سنی چه تقسیم‌بندی‌ای پیشنهاد شده است؟"
)

CELL106_NEW_ANSWER = (
    "در ۵ تا ۷ سالگی کودک ابتدا بیشتر پول را نگه می‌دارد و "
    "پس‌انداز می‌کند و به‌تدریج خرج‌کردن را در موقعیت‌های واقعی "
    "یاد می‌گیرد. در ۸ تا ۱۲ سالگی پول‌توجیبی به خرج و پس‌انداز "
    "تقسیم می‌شود و الگوی پیشنهادی ۵۰ درصد برای هر کدام است. "
    "در ۱۲ تا ۱۵ سالگی سرمایه‌گذاری هم اضافه می‌شود و الگوی "
    "۵۰ درصد خرج، ۳۰ درصد پس‌انداز و ۲۰ درصد سرمایه‌گذاری "
    "پیشنهاد می‌شود. در ۱۵ تا ۱۸ سالگی هدیه یا قرض به دیگران "
    "نیز به بودجه اضافه می‌شود و الگوی ۴۰ درصد خرج، "
    "۳۰ درصد پس‌انداز، ۲۰ درصد سرمایه‌گذاری و ۱۰ درصد هدیه "
    "به دیگران مطرح شده است."
)

CELL106_HUMAN_REASON = (
    "در بازبینی دوم مشخص شد Target قبلی فقط عنوان و دستور "
    "تمرین 8 را شامل می‌شد و سؤال پذیرفته‌شده قبلی بدون مشاهده "
    "کتاب، مستقل و طبیعی نبود. پس از بررسی متن کامل بخش "
    "بودجه‌بندی، پاراگراف‌های 323 تا 327 محتوای آموزشی کامل‌تر "
    "و مناسب‌تری برای همین Pilot فراهم کردند. Human Reviewer "
    "سؤال و پاسخ جایگزین را تأیید کرد."
)


# ============================================================
# 2) Utilities
# ============================================================

cell106_failures = []


def cell106_check(name, condition, details=None):

    if not bool(condition):

        failure = {
            "reason": name,
        }

        if details is not None:
            failure["details"] = details

        cell106_failures.append(
            failure
        )

    return bool(condition)


def cell106_sha256_text(text):

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def cell106_stable_id(prefix, payload):

    if not isinstance(
        payload,
        str,
    ):
        payload = json.dumps(
            payload,
            ensure_ascii=False,
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
        )

    digest = hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()[:32]

    return (
        prefix
        +
        ":"
        +
        digest
    )


def cell106_now():

    return (
        datetime.now(
            timezone.utc
        )
        .replace(
            microsecond=0
        )
        .isoformat()
    )


# ============================================================
# 3) Cell 105 prerequisite
# ============================================================

cell106_check(
    "CELL105_GATE_NOT_PASS",
    globals().get(
        "cell105_gate"
    )
    is True,
)

cell106_check(
    "CELL105_REGISTRY_PATH_MISSING",
    "cell105_registry_path"
    in globals(),
)

cell106_check(
    "CELL105_CONTEXT_RECORDS_MISSING",
    "cell105_context_records"
    in globals(),
)

cell106_check(
    "CELL105_WORK_ID_MISSING",
    "CELL105_WORK_ID"
    in globals(),
)

cell106_check(
    "CELL105_EDITION_ID_MISSING",
    "CELL105_EDITION_ID"
    in globals(),
)


if cell106_failures:

    print(
        "Prerequisite failures:",
        cell106_failures,
    )

    raise AssertionError(
        "Cell 106 blocked because Cell 105 prerequisites are not valid."
    )


cell106_registry_path = Path(
    cell105_registry_path
).resolve()


cell106_check(
    "REGISTRY_FILE_NOT_FOUND",
    cell106_registry_path.exists(),
    str(
        cell106_registry_path
    ),
)


if cell106_failures:

    raise AssertionError(
        cell106_failures
    )


# ============================================================
# 4) Resolve EXACT replacement paragraphs from Cell 105 output
# ============================================================

cell106_replacement_records = []


for paragraph_number in CELL106_REPLACEMENT_PARAGRAPHS:

    matches = [
        record

        for record
        in cell105_context_records

        if (
            isinstance(
                record,
                dict,
            )
            and
            record.get(
                "paragraph_number"
            )
            ==
            paragraph_number
        )
    ]

    cell106_check(
        (
            "REPLACEMENT_PARAGRAPH_"
            +
            str(
                paragraph_number
            )
            +
            "_NOT_UNIQUE"
        ),
        len(
            matches
        )
        ==
        1,
        {
            "match_count":
                len(
                    matches
                ),
        },
    )

    if len(
        matches
    ) == 1:

        cell106_replacement_records.append(
            deepcopy(
                matches[0]
            )
        )


if cell106_failures:

    print(
        cell106_failures
    )

    raise AssertionError(
        "Replacement source resolution failed."
    )


cell106_check(
    "REPLACEMENT_ORDER_MISMATCH",
    [
        record[
            "paragraph_number"
        ]

        for record
        in cell106_replacement_records
    ]
    ==
    CELL106_REPLACEMENT_PARAGRAPHS,
)


cell106_replacement_text = "\n\n".join(
    record[
        "text"
    ]

    for record
    in cell106_replacement_records
)


# ============================================================
# 5) Source-grounding sanity checks
# ============================================================

CELL106_REQUIRED_SOURCE_CUES = [
    "پنج تا هفت",
    "هشت تا دوازده",
    "خرج و پس‌انداز",
    "50 درصد",
    "دوازده تا پانزده",
    "سرمایه‌گذاری",
    "30 درصد",
    "20 درصد",
    "پانزده تا هجده",
    "40 درصد",
    "10 درصد",
    "هدیه",
]


for cue in CELL106_REQUIRED_SOURCE_CUES:

    cell106_check(
        (
            "REQUIRED_SOURCE_CUE_MISSING:"
            +
            cue
        ),
        cue
        in
        cell106_replacement_text,
    )


if cell106_failures:

    print(
        cell106_failures
    )

    raise AssertionError(
        "Replacement evidence does not support the approved Question/Answer."
    )


# ============================================================
# 6) Build normalized replacement source evidence
# ============================================================

cell106_source_evidence = []


for part_index, record in enumerate(
    cell106_replacement_records
):

    paragraph_number = (
        record[
            "paragraph_number"
        ]
    )

    document_id = (
        record[
            "paragraph_id"
        ]
    )

    text = (
        record[
            "text"
        ]
    )

    heading_path = (
        record.get(
            "heading_path"
        )
        or
        [
            "بودجه‌بندی"
        ]
    )

    cell106_source_evidence.append({
        "part_index":
            part_index,

        "role":
            "PROTECTED_REGION_MEMBER",

        "document_id":
            document_id,

        "text":
            text,

        "text_sha256":
            cell106_sha256_text(
                text
            ),

        "char_start":
            0,

        "char_end":
            len(
                text
            ),

        "locator": {
            "work_id":
                CELL105_WORK_ID,

            "edition_id":
                CELL105_EDITION_ID,

            "paragraph_id":
                document_id,

            "paragraph_number":
                paragraph_number,

            "heading_path":
                deepcopy(
                    heading_path
                ),
        },
    })


# ============================================================
# 7) Deterministic replacement IDs
# ============================================================

cell106_target_identity_payload = {
    "pilot_case_id":
        CELL106_PILOT_ID,

    "replacement_paragraphs":
        CELL106_REPLACEMENT_PARAGRAPHS,

    "replacement_documents": [
        {
            "document_id":
                item[
                    "document_id"
                ],

            "text_sha256":
                item[
                    "text_sha256"
                ],
        }

        for item
        in cell106_source_evidence
    ],
}


cell106_new_target_id = (
    cell106_stable_id(
        "target-evidence",
        cell106_target_identity_payload,
    )
)


cell106_new_question_id = (
    cell106_stable_id(
        "question-draft",
        {
            "pilot_case_id":
                CELL106_PILOT_ID,

            "target_evidence_id":
                cell106_new_target_id,

            "question":
                CELL106_NEW_QUESTION,
        },
    )
)


cell106_new_answer_id = (
    cell106_stable_id(
        "expected-answer-draft",
        {
            "pilot_case_id":
                CELL106_PILOT_ID,

            "question_draft_id":
                cell106_new_question_id,

            "target_evidence_id":
                cell106_new_target_id,

            "answer":
                CELL106_NEW_ANSWER,
        },
    )
)


cell106_new_citation_id = (
    cell106_stable_id(
        "citation-mapping",
        {
            "pilot_case_id":
                CELL106_PILOT_ID,

            "expected_answer_draft_id":
                cell106_new_answer_id,

            "target_evidence_id":
                cell106_new_target_id,

            "documents": [
                item[
                    "document_id"
                ]

                for item
                in cell106_source_evidence
            ],
        },
    )
)


cell106_amendment_id = (
    cell106_stable_id(
        "second-human-review",
        {
            "pilot_case_id":
                CELL106_PILOT_ID,

            "old_target_evidence_id":
                CELL106_OLD_TARGET_ID,

            "new_target_evidence_id":
                cell106_new_target_id,

            "new_question_draft_id":
                cell106_new_question_id,

            "new_expected_answer_draft_id":
                cell106_new_answer_id,
        },
    )
)


# ============================================================
# 8) Load Registry fresh from disk
# ============================================================

cell106_registry_before_text = (
    cell106_registry_path.read_text(
        encoding="utf-8"
    )
)


cell106_registry = json.loads(
    cell106_registry_before_text
)


cell106_registry_before_sha256 = (
    hashlib.sha256(
        cell106_registry_before_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# 9) Locate Case in BOTH durable Registry copies
# ============================================================

cell106_completed_cases = (
    cell106_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell106_author_review_root = (
    cell106_registry.get(
        "author_review",
        {}
    )
    or
    {}
)


cell106_author_cases = (
    cell106_author_review_root.get(
        "cases_ready_for_review",
        [],
    )
)


def cell106_find_case(
    case_list,
    location_name,
):

    matches = [
        case

        for case
        in case_list

        if (
            isinstance(
                case,
                dict,
            )
            and
            case.get(
                "pilot_case_id"
            )
            ==
            CELL106_PILOT_ID
        )
    ]

    cell106_check(
        (
            location_name
            +
            "_CASE_NOT_UNIQUE"
        ),
        len(
            matches
        )
        ==
        1,
        {
            "match_count":
                len(
                    matches
                ),
        },
    )

    if len(
        matches
    ) == 1:

        return matches[0]

    return None


cell106_completed_case = (
    cell106_find_case(
        cell106_completed_cases,
        "COMPLETED",
    )
)


cell106_author_case = (
    cell106_find_case(
        cell106_author_cases,
        "AUTHOR_READY",
    )
)


if cell106_failures:

    print(
        cell106_failures
    )

    raise AssertionError(
        "Case 4 could not be resolved uniquely in Registry."
    )


# ============================================================
# 10) Helper: resolve currently active Question
# ============================================================

def cell106_active_question(case):

    final_question_id = (
        case.get(
            "final_question_draft_id"
        )
    )

    for draft in (
        case.get(
            "model_drafts",
            []
        )
        or
        []
    ):

        if (
            isinstance(
                draft,
                dict,
            )
            and
            draft.get(
                "question_draft_id"
            )
            ==
            final_question_id
        ):

            return draft.get(
                "question_text"
            )

    question_review = (
        case.get(
            "question_review"
        )
        or
        {}
    )

    return question_review.get(
        "question_text"
    )


# ============================================================
# 11) Validate current / historical state before mutation
# ============================================================

CELL106_OLD_QUESTION = (
    "در تمرین ۸، برای بررسی هزینه‌های ماهانه افراد "
    "چه مراحلی باید انجام شود؟"
)


for location_name, case in [
    (
        "COMPLETED",
        cell106_completed_case,
    ),
    (
        "AUTHOR_READY",
        cell106_author_case,
    ),
]:

    existing_second_review = (
        case.get(
            "second_human_review"
        )
        or
        {}
    )

    existing_amendment_id = (
        existing_second_review.get(
            "amendment_id"
        )
    )

    exact_amendment_already_active = (
        existing_amendment_id
        ==
        cell106_amendment_id
        and
        case.get(
            "target_evidence_id"
        )
        ==
        cell106_new_target_id
        and
        cell106_active_question(
            case
        )
        ==
        CELL106_NEW_QUESTION
        and
        (
            case.get(
                "expected_answer_review"
            )
            or
            {}
        ).get(
            "expected_answer_text"
        )
        ==
        CELL106_NEW_ANSWER
    )

    if exact_amendment_already_active:

        continue

    cell106_check(
        (
            location_name
            +
            "_UNEXPECTED_EXISTING_SECOND_REVIEW"
        ),
        existing_amendment_id
        in
        [
            None,
            cell106_amendment_id,
        ],
        existing_amendment_id,
    )

    cell106_check(
        (
            location_name
            +
            "_OLD_TARGET_MISMATCH"
        ),
        case.get(
            "target_evidence_id"
        )
        ==
        CELL106_OLD_TARGET_ID,
        case.get(
            "target_evidence_id"
        ),
    )

    cell106_check(
        (
            location_name
            +
            "_OLD_QUESTION_MISMATCH"
        ),
        cell106_active_question(
            case
        )
        ==
        CELL106_OLD_QUESTION,
        cell106_active_question(
            case
        ),
    )

    author_final_review = (
        case.get(
            "author_final_review"
        )
        or
        {}
    )

    cell106_check(
        (
            location_name
            +
            "_AUTHOR_REVIEW_ALREADY_STARTED"
        ),
        author_final_review.get(
            "status"
        )
        ==
        "NOT_STARTED",
        author_final_review,
    )


if cell106_failures:

    print(
        "\n=== PRE-WRITE FAILURES ==="
    )

    print(
        cell106_failures
    )

    raise AssertionError(
        "Registry current state does not match the expected Case 4 preconditions."
    )


# ============================================================
# 12) Reuse existing attribution — DO NOT invent attribution
# ============================================================

cell106_existing_citation_entries = (
    (
        cell106_completed_case.get(
            "citation_review"
        )
        or
        {}
    ).get(
        "citation_entries"
    )
    or
    []
)


cell106_attribution_template = None


for entry in cell106_existing_citation_entries:

    if (
        isinstance(
            entry,
            dict,
        )
        and
        isinstance(
            entry.get(
                "attribution"
            ),
            dict,
        )
    ):

        cell106_attribution_template = (
            deepcopy(
                entry[
                    "attribution"
                ]
            )
        )

        break


cell106_check(
    "EXISTING_ATTRIBUTION_TEMPLATE_NOT_FOUND",
    isinstance(
        cell106_attribution_template,
        dict,
    ),
)


if cell106_failures:

    raise AssertionError(
        cell106_failures
    )


# ============================================================
# 13) Build new Citation entries from exact replacement evidence
# ============================================================

cell106_new_citation_entries = []


for citation_index, item in enumerate(
    cell106_source_evidence
):

    cell106_new_citation_entries.append({
        "citation_index":
            citation_index,

        "document_id":
            item[
                "document_id"
            ],

        "source_id":
            CELL105_EDITION_ID,

        "target_evidence_id":
            cell106_new_target_id,

        "role":
            item[
                "role"
            ],

        "char_start":
            item[
                "char_start"
            ],

        "char_end":
            item[
                "char_end"
            ],

        "char_count":
            (
                item[
                    "char_end"
                ]
                -
                item[
                    "char_start"
                ]
            ),

        "locator":
            deepcopy(
                item[
                    "locator"
                ]
            ),

        "locator_origin":
            "COPIED_FROM_SECOND_HUMAN_REVIEW_REPLACEMENT_EVIDENCE",

        "attribution":
            deepcopy(
                cell106_attribution_template
            ),
    })


# ============================================================
# 14) Apply exact amendment to one Registry Case
# ============================================================

cell106_review_timestamp = (
    cell106_now()
)


def cell106_apply_amendment(
    case,
    registry_location,
):

    existing_second_review = (
        case.get(
            "second_human_review"
        )
        or
        {}
    )

    # --------------------------------------------------------
    # Idempotent no-op if this exact amendment is already active
    # --------------------------------------------------------

    if (
        existing_second_review.get(
            "amendment_id"
        )
        ==
        cell106_amendment_id
        and
        case.get(
            "target_evidence_id"
        )
        ==
        cell106_new_target_id
        and
        cell106_active_question(
            case
        )
        ==
        CELL106_NEW_QUESTION
        and
        (
            case.get(
                "expected_answer_review"
            )
            or
            {}
        ).get(
            "expected_answer_text"
        )
        ==
        CELL106_NEW_ANSWER
    ):

        return (
            "ALREADY_APPLIED"
        )


    # --------------------------------------------------------
    # Preserve exact first-review accepted state
    # --------------------------------------------------------

    previous_final_question_id = (
        case.get(
            "final_question_draft_id"
        )
    )

    previous_question_text = (
        cell106_active_question(
            case
        )
    )

    previous_expected_answer_review = (
        deepcopy(
            case.get(
                "expected_answer_review"
            )
        )
    )

    previous_citation_review = (
        deepcopy(
            case.get(
                "citation_review"
            )
        )
    )

    previous_source_evidence = (
        deepcopy(
            case.get(
                "source_evidence"
            )
        )
    )

    previous_target_id = (
        case.get(
            "target_evidence_id"
        )
    )


    history = case.setdefault(
        "post_final_human_review_history",
        [],
    )


    existing_history_ids = {
        item.get(
            "amendment_id"
        )

        for item
        in history

        if isinstance(
            item,
            dict,
        )
    }


    if (
        cell106_amendment_id
        not in
        existing_history_ids
    ):

        history.append({
            "amendment_id":
                cell106_amendment_id,

            "review_record_version":
                "second_human_review_amendment_v1",

            "review_stage":
                "SECOND_HUMAN_REVIEW",

            "review_timestamp_utc":
                cell106_review_timestamp,

            "registry_location":
                registry_location,

            "human_decision":
                "RETARGET_AND_ACCEPT_REPLACEMENT",

            "human_review_reason":
                CELL106_HUMAN_REASON,

            "previous_state_preserved":
                True,

            "superseded_accepted_qa": {
                "target_evidence_id":
                    previous_target_id,

                "source_evidence":
                    previous_source_evidence,

                "question_draft_id":
                    previous_final_question_id,

                "question_text":
                    previous_question_text,

                "expected_answer_review":
                    previous_expected_answer_review,

                "citation_review":
                    previous_citation_review,
            },

            "replacement_state": {
                "target_evidence_id":
                    cell106_new_target_id,

                "replacement_paragraphs":
                    deepcopy(
                        CELL106_REPLACEMENT_PARAGRAPHS
                    ),

                "question_draft_id":
                    cell106_new_question_id,

                "question_text":
                    CELL106_NEW_QUESTION,

                "expected_answer_draft_id":
                    cell106_new_answer_id,

                "expected_answer_text":
                    CELL106_NEW_ANSWER,

                "citation_mapping_id":
                    cell106_new_citation_id,
            },

            "raw_source_mutated":
                False,

            "parser_output_mutated":
                False,

            "author_review_recorded":
                False,
        })


    # --------------------------------------------------------
    # Mark old active draft as historically accepted but superseded
    # WITHOUT changing the old human_decision.
    # --------------------------------------------------------

    model_drafts = (
        case.get(
            "model_drafts",
            []
        )
        or
        []
    )


    for draft in model_drafts:

        if (
            isinstance(
                draft,
                dict,
            )
            and
            draft.get(
                "question_draft_id"
            )
            ==
            previous_final_question_id
        ):

            draft[
                "second_review_status"
            ] = (
                "SUPERSEDED_BY_RETARGET"
            )

            draft[
                "superseded_by_question_draft_id"
            ] = (
                cell106_new_question_id
            )


    existing_draft_ids = {
        draft.get(
            "question_draft_id"
        )

        for draft
        in model_drafts

        if isinstance(
            draft,
            dict,
        )
    }


    if (
        cell106_new_question_id
        not in
        existing_draft_ids
    ):

        previous_revisions = [
            int(
                draft.get(
                    "revision",
                    0,
                )
                or
                0
            )

            for draft
            in model_drafts

            if isinstance(
                draft,
                dict,
            )
        ]

        new_revision = (
            max(
                previous_revisions
                or
                [0]
            )
            +
            1
        )

        model_drafts.append({
            "question_draft_id":
                cell106_new_question_id,

            "question_text":
                CELL106_NEW_QUESTION,

            "revision":
                new_revision,

            "machine_validation":
                "SECOND_HUMAN_REVIEW_LOCAL_GATES_PASS",

            "human_decision":
                "ACCEPT",

            "review_origin":
                "SECOND_HUMAN_REVIEW_RETARGET",

            "target_evidence_id":
                cell106_new_target_id,
        })


    case[
        "model_drafts"
    ] = (
        model_drafts
    )


    # --------------------------------------------------------
    # Append second-review Human Review records
    # --------------------------------------------------------

    human_reviews = (
        case.get(
            "human_reviews",
            []
        )
        or
        []
    )


    cell106_retarget_review_event_id = (
        cell106_stable_id(
            "human-review",
            {
                "amendment_id":
                    cell106_amendment_id,

                "event":
                    "RETARGET",
            },
        )
    )


    cell106_question_accept_event_id = (
        cell106_stable_id(
            "human-review",
            {
                "amendment_id":
                    cell106_amendment_id,

                "event":
                    "QUESTION_ACCEPT",
            },
        )
    )


    existing_review_event_ids = {
        review.get(
            "review_event_id"
        )

        for review
        in human_reviews

        if isinstance(
            review,
            dict,
        )
    }


    if (
        cell106_retarget_review_event_id
        not in
        existing_review_event_ids
    ):

        human_reviews.append({
            "review_event_id":
                cell106_retarget_review_event_id,

            "review_record_version":
                "second_human_review_record_v1",

            "review_stage":
                "SECOND_HUMAN_REVIEW_RETARGET",

            "pilot_case_id":
                CELL106_PILOT_ID,

            "human_decision":
                "RETARGET",

            "previous_target_evidence_id":
                previous_target_id,

            "previous_question_draft_id":
                previous_final_question_id,

            "replacement_target_evidence_id":
                cell106_new_target_id,

            "replacement_question_draft_id":
                cell106_new_question_id,

            "previous_review_preserved":
                True,

            "human_review_reason":
                CELL106_HUMAN_REASON,

            "raw_source_mutated":
                False,

            "parser_output_mutated":
                False,
        })


    if (
        cell106_question_accept_event_id
        not in
        existing_review_event_ids
    ):

        human_reviews.append({
            "review_event_id":
                cell106_question_accept_event_id,

            "review_record_version":
                "second_human_question_review_record_v1",

            "review_stage":
                "SECOND_HUMAN_REVIEW_QUESTION",

            "pilot_case_id":
                CELL106_PILOT_ID,

            "target_evidence_id":
                cell106_new_target_id,

            "question_draft_id":
                cell106_new_question_id,

            "question_text":
                CELL106_NEW_QUESTION,

            "human_decision":
                "ACCEPT",

            "human_note":
                (
                    "Human Reviewer explicitly approved the "
                    "replacement Question after reviewing the "
                    "full source context."
                ),

            "external_knowledge_used":
                False,
        })


    case[
        "human_reviews"
    ] = (
        human_reviews
    )


    # --------------------------------------------------------
    # Activate replacement Target + Question
    # --------------------------------------------------------

    case[
        "target_evidence_id"
    ] = (
        cell106_new_target_id
    )

    case[
        "source_evidence"
    ] = deepcopy(
        cell106_source_evidence
    )

    case[
        "final_question_draft_id"
    ] = (
        cell106_new_question_id
    )


    # --------------------------------------------------------
    # Activate replacement Expected Answer
    # --------------------------------------------------------

    case[
        "expected_answer_review"
    ] = {
        "expected_answer_draft_id":
            cell106_new_answer_id,

        "expected_answer_text":
            CELL106_NEW_ANSWER,

        "human_decision":
            "ACCEPT",

        "machine_validation_status":
            "SECOND_HUMAN_REVIEW_LOCAL_GATES_PASS",

        "source_evidence":
            deepcopy(
                cell106_source_evidence
            ),

        "human_review": {
            "review_record_version":
                "second_human_expected_answer_review_record_v1",

            "review_stage":
                "SECOND_HUMAN_REVIEW_EXPECTED_ANSWER",

            "pilot_case_id":
                CELL106_PILOT_ID,

            "question_draft_id":
                cell106_new_question_id,

            "expected_answer_draft_id":
                cell106_new_answer_id,

            "target_evidence_id":
                cell106_new_target_id,

            "human_decision":
                "ACCEPT",

            "expected_answer_text":
                CELL106_NEW_ANSWER,

            "external_knowledge_used":
                False,

            "human_context_used_as_evidence":
                False,

            "human_note":
                (
                    "Human Reviewer accepted the replacement "
                    "Expected Answer after reviewing the exact "
                    "book paragraphs 323-327."
                ),
        },
    }


    # --------------------------------------------------------
    # Activate replacement Citation Mapping
    # --------------------------------------------------------

    case[
        "citation_review"
    ] = {
        "citation_mapping_id":
            cell106_new_citation_id,

        "citation_entries":
            deepcopy(
                cell106_new_citation_entries
            ),

        "human_decision":
            "ACCEPT",

        "machine_validation_status":
            "SECOND_HUMAN_REVIEW_LOCAL_GATES_PASS",

        "human_review": {
            "review_record_version":
                "second_human_citation_review_record_v1",

            "review_stage":
                "SECOND_HUMAN_REVIEW_CITATION",

            "pilot_case_id":
                CELL106_PILOT_ID,

            "question_draft_id":
                cell106_new_question_id,

            "expected_answer_draft_id":
                cell106_new_answer_id,

            "target_evidence_id":
                cell106_new_target_id,

            "citation_mapping_id":
                cell106_new_citation_id,

            "human_decision":
                "ACCEPT",

            "external_locator_lookup_used":
                False,

            "human_note":
                (
                    "Citation Mapping uses only the exact "
                    "replacement book paragraphs 323-327 "
                    "resolved in Cell 105."
                ),
        },
    }


    # --------------------------------------------------------
    # Explicit second-review current status
    # --------------------------------------------------------

    case[
        "second_human_review"
    ] = {
        "amendment_id":
            cell106_amendment_id,

        "status":
            "COMPLETE",

        "decision":
            "RETARGET_AND_ACCEPT_REPLACEMENT",

        "review_timestamp_utc":
            cell106_review_timestamp,

        "previous_target_evidence_id":
            previous_target_id,

        "replacement_target_evidence_id":
            cell106_new_target_id,

        "replacement_question_draft_id":
            cell106_new_question_id,

        "replacement_expected_answer_draft_id":
            cell106_new_answer_id,

        "replacement_citation_mapping_id":
            cell106_new_citation_id,

        "previous_state_preserved":
            True,

        "raw_source_mutated":
            False,

        "parser_output_mutated":
            False,

        "author_review_status":
            "NOT_STARTED",
    }


    # --------------------------------------------------------
    # Keep author review untouched / pending
    # --------------------------------------------------------

    author_final_review = (
        case.get(
            "author_final_review"
        )
        or
        {}
    )

    author_final_review.setdefault(
        "reviewer",
        "AUTHOR",
    )

    author_final_review[
        "status"
    ] = (
        "NOT_STARTED"
    )

    author_final_review[
        "decision"
    ] = None

    author_final_review[
        "notes"
    ] = None

    case[
        "author_final_review"
    ] = (
        author_final_review
    )


    case[
        "final_status"
    ] = (
        "HUMAN_REVIEW_COMPLETE_"
        "AUTHOR_FINAL_REVIEW_PENDING"
    )


    return (
        "APPLIED"
    )


# ============================================================
# 15) Apply to BOTH durable copies
# ============================================================

cell106_completed_apply_status = (
    cell106_apply_amendment(
        cell106_completed_case,
        "completed_human_review_cases",
    )
)


cell106_author_apply_status = (
    cell106_apply_amendment(
        cell106_author_case,
        "author_review.cases_ready_for_review",
    )
)


# ============================================================
# 16) In-memory validation BEFORE write
# ============================================================

def cell106_validate_final_case(
    case,
    location_name,
):

    failures_before = len(
        cell106_failures
    )

    cell106_check(
        location_name
        +
        "_ACTIVE_TARGET_MISMATCH",
        case.get(
            "target_evidence_id"
        )
        ==
        cell106_new_target_id,
    )

    cell106_check(
        location_name
        +
        "_ACTIVE_QUESTION_ID_MISMATCH",
        case.get(
            "final_question_draft_id"
        )
        ==
        cell106_new_question_id,
    )

    cell106_check(
        location_name
        +
        "_ACTIVE_QUESTION_TEXT_MISMATCH",
        cell106_active_question(
            case
        )
        ==
        CELL106_NEW_QUESTION,
    )

    cell106_check(
        location_name
        +
        "_ACTIVE_ANSWER_MISMATCH",
        (
            case.get(
                "expected_answer_review"
            )
            or
            {}
        ).get(
            "expected_answer_text"
        )
        ==
        CELL106_NEW_ANSWER,
    )

    cell106_check(
        location_name
        +
        "_ANSWER_NOT_ACCEPTED",
        (
            case.get(
                "expected_answer_review"
            )
            or
            {}
        ).get(
            "human_decision"
        )
        ==
        "ACCEPT",
    )

    cell106_check(
        location_name
        +
        "_CITATION_NOT_ACCEPTED",
        (
            case.get(
                "citation_review"
            )
            or
            {}
        ).get(
            "human_decision"
        )
        ==
        "ACCEPT",
    )

    cell106_check(
        location_name
        +
        "_CITATION_TARGET_MISMATCH",
        all(
            entry.get(
                "target_evidence_id"
            )
            ==
            cell106_new_target_id

            for entry
            in (
                (
                    case.get(
                        "citation_review"
                    )
                    or
                    {}
                ).get(
                    "citation_entries"
                )
                or
                []
            )
        ),
    )

    cell106_check(
        location_name
        +
        "_REPLACEMENT_EVIDENCE_COUNT_MISMATCH",
        len(
            case.get(
                "source_evidence",
                []
            )
            or
            []
        )
        ==
        5,
    )

    cell106_check(
        location_name
        +
        "_SECOND_REVIEW_NOT_COMPLETE",
        (
            case.get(
                "second_human_review"
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "COMPLETE",
    )

    cell106_check(
        location_name
        +
        "_AMENDMENT_ID_MISMATCH",
        (
            case.get(
                "second_human_review"
            )
            or
            {}
        ).get(
            "amendment_id"
        )
        ==
        cell106_amendment_id,
    )

    cell106_check(
        location_name
        +
        "_HISTORY_MISSING",
        any(
            (
                isinstance(
                    record,
                    dict,
                )
                and
                record.get(
                    "amendment_id"
                )
                ==
                cell106_amendment_id
            )

            for record
            in (
                case.get(
                    "post_final_human_review_history",
                    []
                )
                or
                []
            )
        ),
    )

    cell106_check(
        location_name
        +
        "_AUTHOR_REVIEW_STATUS_CHANGED",
        (
            case.get(
                "author_final_review"
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "NOT_STARTED",
    )

    return (
        len(
            cell106_failures
        )
        ==
        failures_before
    )


cell106_validate_final_case(
    cell106_completed_case,
    "COMPLETED",
)


cell106_validate_final_case(
    cell106_author_case,
    "AUTHOR_READY",
)


cell106_check(
    "AUTHOR_REVIEW_CASE_COUNT_CHANGED",
    cell106_author_review_root.get(
        "case_count"
    )
    ==
    22,
    cell106_author_review_root.get(
        "case_count"
    ),
)


if cell106_failures:

    print(
        "\n=== IN-MEMORY VALIDATION FAILURES ==="
    )

    print(
        cell106_failures
    )

    raise AssertionError(
        "Second Review amendment failed before Registry write."
    )


# ============================================================
# 17) Write only if bytes actually need to change
# ============================================================

cell106_registry_after_text = (
    json.dumps(
        cell106_registry,
        ensure_ascii=False,
        indent=2,
    )
    +
    "\n"
)


cell106_change_required = (
    cell106_registry_after_text
    !=
    cell106_registry_before_text
)


cell106_backup_path = None


if cell106_change_required:

    timestamp_for_filename = (
        datetime.now(
            timezone.utc
        ).strftime(
            "%Y%m%dT%H%M%SZ"
        )
    )

    cell106_backup_path = (
        cell106_registry_path.parent
        /
        (
            cell106_registry_path.stem
            +
            ".before_case4_second_review_"
            +
            timestamp_for_filename
            +
            cell106_registry_path.suffix
        )
    )

    cell106_backup_path.write_text(
        cell106_registry_before_text,
        encoding="utf-8",
    )


    cell106_temp_path = (
        cell106_registry_path.parent
        /
        (
            cell106_registry_path.name
            +
            ".cell106.tmp"
        )
    )


    cell106_temp_path.write_text(
        cell106_registry_after_text,
        encoding="utf-8",
    )


    # Validate temporary JSON before replacing Registry.
    json.loads(
        cell106_temp_path.read_text(
            encoding="utf-8"
        )
    )


    cell106_temp_path.replace(
        cell106_registry_path
    )


# ============================================================
# 18) Reload exact persisted Registry
# ============================================================

cell106_persisted_text = (
    cell106_registry_path.read_text(
        encoding="utf-8"
    )
)


cell106_persisted_registry = (
    json.loads(
        cell106_persisted_text
    )
)


cell106_registry_after_sha256 = (
    hashlib.sha256(
        cell106_persisted_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# 19) Resolve persisted copies independently
# ============================================================

cell106_persisted_completed_matches = [
    case

    for case
    in cell106_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )

    if (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL106_PILOT_ID
    )
]


cell106_persisted_author_matches = [
    case

    for case
    in (
        (
            cell106_persisted_registry.get(
                "author_review",
                {}
            )
            or
            {}
        ).get(
            "cases_ready_for_review",
            [],
        )
    )

    if (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL106_PILOT_ID
    )
]


cell106_check(
    "PERSISTED_COMPLETED_CASE_NOT_UNIQUE",
    len(
        cell106_persisted_completed_matches
    )
    ==
    1,
)


cell106_check(
    "PERSISTED_AUTHOR_CASE_NOT_UNIQUE",
    len(
        cell106_persisted_author_matches
    )
    ==
    1,
)


if not cell106_failures:

    cell106_persisted_completed_case = (
        cell106_persisted_completed_matches[
            0
        ]
    )

    cell106_persisted_author_case = (
        cell106_persisted_author_matches[
            0
        ]
    )

    cell106_validate_final_case(
        cell106_persisted_completed_case,
        "PERSISTED_COMPLETED",
    )

    cell106_validate_final_case(
        cell106_persisted_author_case,
        "PERSISTED_AUTHOR_READY",
    )


# ============================================================
# 20) Cross-copy canonical equality
# ============================================================

if not cell106_failures:

    cell106_check(
        "CROSS_COPY_TARGET_MISMATCH",
        cell106_persisted_completed_case.get(
            "target_evidence_id"
        )
        ==
        cell106_persisted_author_case.get(
            "target_evidence_id"
        ),
    )

    cell106_check(
        "CROSS_COPY_QUESTION_MISMATCH",
        cell106_active_question(
            cell106_persisted_completed_case
        )
        ==
        cell106_active_question(
            cell106_persisted_author_case
        ),
    )

    cell106_check(
        "CROSS_COPY_ANSWER_MISMATCH",
        (
            cell106_persisted_completed_case.get(
                "expected_answer_review"
            )
            or
            {}
        ).get(
            "expected_answer_text"
        )
        ==
        (
            cell106_persisted_author_case.get(
                "expected_answer_review"
            )
            or
            {}
        ).get(
            "expected_answer_text"
        ),
    )

    cell106_check(
        "CROSS_COPY_CITATION_ID_MISMATCH",
        (
            cell106_persisted_completed_case.get(
                "citation_review"
            )
            or
            {}
        ).get(
            "citation_mapping_id"
        )
        ==
        (
            cell106_persisted_author_case.get(
                "citation_review"
            )
            or
            {}
        ).get(
            "citation_mapping_id"
        ),
    )


# ============================================================
# 21) Final gate
# ============================================================

cell106_gate = (
    len(
        cell106_failures
    )
    ==
    0
)


print(
    "=== CASE 4 SECOND HUMAN REVIEW — COMPLETE RETARGET ==="
)


print(
    "\nPilot:",
    CELL106_PILOT_ID,
)


print(
    "\nPrevious Target:",
    CELL106_OLD_TARGET_ID,
)


print(
    "Replacement Target:",
    cell106_new_target_id,
)


print(
    "Replacement paragraphs:",
    CELL106_REPLACEMENT_PARAGRAPHS,
)


print(
    "\nReplacement Question:"
)

print(
    CELL106_NEW_QUESTION
)


print(
    "\nReplacement Answer:"
)

print(
    CELL106_NEW_ANSWER
)


print(
    "\nQuestion Draft ID:",
    cell106_new_question_id,
)


print(
    "Expected Answer Draft ID:",
    cell106_new_answer_id,
)


print(
    "Citation Mapping ID:",
    cell106_new_citation_id,
)


print(
    "Second Review Amendment ID:",
    cell106_amendment_id,
)


print(
    "\nCompleted copy apply status:",
    cell106_completed_apply_status,
)


print(
    "Author-ready copy apply status:",
    cell106_author_apply_status,
)


print(
    "\nRegistry write required:",
    cell106_change_required,
)


print(
    "Backup path:",
    (
        str(
            cell106_backup_path
        )
        if cell106_backup_path
        is not None
        else
        "NOT_NEEDED_ALREADY_APPLIED"
    ),
)


print(
    "\nRegistry SHA256 before:",
    cell106_registry_before_sha256,
)


print(
    "Registry SHA256 after:",
    cell106_registry_after_sha256,
)


print(
    "\n=== HISTORY / SAFETY ==="
)


print(
    "Previous Target preserved:",
    True,
)


print(
    "Previous Question preserved:",
    True,
)


print(
    "Previous Answer preserved:",
    True,
)


print(
    "Previous Citation preserved:",
    True,
)


print(
    "Raw Source mutated:",
    False,
)


print(
    "Parser Output mutated:",
    False,
)


print(
    "Author Review recorded:",
    False,
)


print(
    "Author Review status:",
    "NOT_STARTED",
)


print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)


print(
    cell106_failures
)


print(
    "\nCODE CELL 106 GATE:",
    (
        "PASS"
        if cell106_gate
        else
        "FAIL"
    ),
)


assert cell106_gate, (
    "Case 4 Second Human Review amendment failed. "
    "Do not continue to another Case until this SAME Cell 106 is repaired."
)

=== CASE 4 SECOND HUMAN REVIEW — COMPLETE RETARGET ===

Pilot: question-pilot:909b695797935246c92c7a66ebe0f773

Previous Target: target-evidence:8d87341f7847558d90ee76e350c64fa3
Replacement Target: target-evidence:63c2f0f041fa8caa0f21a1aed165d7ec
Replacement paragraphs: [323, 324, 325, 326, 327]

Replacement Question:
مدیریت پول‌توجیبی از ۵ تا ۱۸ سالگی چگونه تغییر می‌کند و در هر بازه سنی چه تقسیم‌بندی‌ای پیشنهاد شده است؟

Replacement Answer:
در ۵ تا ۷ سالگی کودک ابتدا بیشتر پول را نگه می‌دارد و پس‌انداز می‌کند و به‌تدریج خرج‌کردن را در موقعیت‌های واقعی یاد می‌گیرد. در ۸ تا ۱۲ سالگی پول‌توجیبی به خرج و پس‌انداز تقسیم می‌شود و الگوی پیشنهادی ۵۰ درصد برای هر کدام است. در ۱۲ تا ۱۵ سالگی سرمایه‌گذاری هم اضافه می‌شود و الگوی ۵۰ درصد خرج، ۳۰ درصد پس‌انداز و ۲۰ درصد سرمایه‌گذاری پیشنهاد می‌شود. در ۱۵ تا ۱۸ سالگی هدیه یا قرض به دیگران نیز به بودجه اضافه می‌شود و الگوی ۴۰ درصد خرج، ۳۰ درصد پس‌انداز، ۲۰ درصد سرمایه‌گذاری و ۱۰ درصد هدیه به دیگران مطرح شده است.

Question Draft ID: question-draft:a0

<div style="direction: rtl; text-align: right; white-space: normal; line-height: 1.8;">

## پایلوت‌های مشتق‌شده از بازخورد واقعی دکتر در دموی فاز یک

در این مرحله، ۶ مورد بازخورد واقعی دکتر کمیل رودی از دموی فاز یک به‌عنوان Candidate Pilot جدید وارد بازبینی دوم می‌شوند.

این ۶ مورد با ۲۲ پایلوت قبلی تفاوت منشأ دارند:

- ۲۲ پایلوت قبلی از Corpus فاز دو و Human Review ساخته شده‌اند.
- این ۶ مورد از سؤال‌های واقعی دموی فاز یک و بازخورد مستقیم دکتر کمیل رودی استخراج شده‌اند.
- متن بازخورد دکتر به‌عنوان شواهد مستقیم مؤلف حفظ می‌شود.
- جواب مدل فاز یک به‌عنوان Ground Truth در نظر گرفته نمی‌شود.
- هیچ متن تاریخی حذف یا بازنویسی نمی‌شود.
- سؤال و پاسخ پیشنهادی جدید به‌صورت Artifact مستقل ثبت می‌شود.
- این ۶ مورد هنوز به صف ۲۲تایی Author Review اضافه نمی‌شوند.
- پس از بررسی این Batch، هر ۶ مورد در یک مرحله مستقل به همراه ۲۲ مورد قبلی برای بازبینی نهایی دکتر آماده خواهند شد.

---

### مورد 23 — تعریف سواد مالی

**سؤال اصلی دموی فاز یک:**

سواد مالی درباره پول دار شدن دیگه؟

**بازخورد مستقیم دکتر:**

نه ، ببین سواد مالی درباره مدیریت پول است. نه درباره میزان پول

**سؤال پیشنهادی پایلوت:**

سواد مالی درباره میزان پولی است که داریم یا درباره مدیریت پول؟

**پاسخ پیشنهادی:**

سواد مالی درباره مدیریت پول است، نه درباره میزان پول.

**نوع پایلوت:**

AUTHOR_FEEDBACK_CONTENT

---

### مورد 24 — ارزیابی دانش، نگرش و مهارت مالی

**سؤال اصلی دموی فاز یک:**

ایا من از نظر مالی وضعیت قابل قبولی دارم؟ از نظر دانشی و مهارتی یا حتی نگرشی؟

**بازخورد مستقیم دکتر:**

مدیریت مالی شخصی نیازمند دانش و نگرش و مهارت های خاصی است، برا یاینکه بدانید شما به اندازه کافی وضعیت مالی خوبی دارید ، لازم است این پرسشنامه را تکمیل کنید.

**سؤال پیشنهادی پایلوت:**

برای اینکه بفهمیم از نظر دانش، نگرش و مهارت‌های مالی وضعیت قابل قبولی داریم، چه باید کرد؟

**پاسخ پیشنهادی:**

مدیریت مالی شخصی به دانش، نگرش و مهارت‌های خاص نیاز دارد. برای اینکه بدانیم وضعیت مالی ما از این جنبه‌ها به اندازه کافی مناسب است، باید پرسشنامه مربوط را تکمیل کنیم.

**نوع پایلوت:**

AUTHOR_FEEDBACK_CONTENT

---

### مورد 25 — تداوم گفت‌وگو در سؤال پیگیری

**سؤال اولیه کاربر:**

نمی دانم پولم کجا می رود ؟ انگار پولم را گم می کنم.

**پاسخ اول مدل:**

پول شما ممکن است به دلیل عدم توجه به جزئیات هزینه‌ها و انجام خریدهایی بدون نظارت، به طور غیر منظمی خرج شود. بهترین راه برای کشف این موضوع، ثبت دقیق هزینه‌هاست تا بدانیم چه چیزی را کجاها خرج کرده‌اید. این کار به شما کمک می‌کند تا بهترین تصمیم‌ها را بگیرید و از هدررفت پول جلوگیری کنید.

**سؤال پیگیری کاربر:**

این جواب خوبی بود. ولی کوتاه است بیشتر توضیح می دهی.

**پاسخ دوم مدل:**

مدل به‌اشتباه موضوع را عوض کرد و درباره قرض گرفتن پاسخ داد.

**بازخورد مستقیم دکتر:**

این خیلی حواب بی ربطی بود.

**وظیفه پایلوت:**

اگر کاربر بعد از پاسخ درباره پیدا کردن مسیر خرج پول بگوید «بیشتر توضیح می‌دهی»، پاسخ باید همان موضوع قبلی را ادامه و بسط دهد و نباید به موضوع نامرتبطی مانند قرض گرفتن تغییر مسیر دهد.

**نوع پایلوت:**

MULTI_TURN_CONTEXT_CONTINUITY

این مورد یک Golden Content Question معمولی نیست؛ یک Regression Pilot برای حفظ Context مکالمه است.

---

### مورد 26 — الگوهای خریدی که باعث هدررفت پول می‌شوند

**زمینه گفت‌وگوی فاز یک:**

کاربر درباره هزینه‌های غیرضروری پرسید و سپس خواست چند نمونه ببیند.

**بازخورد مستقیم دکتر:**

آنچه باعث می شود پول هایم هدر روند ، الگوی خرید نامناسب است مانند

1. خرید با تنوع و حجم زیاد
2. خرید به خاطر تنش عاطفی
3. خرید به خاطر هیجان زود گذر
4. خرید به خاطر پذیرفته شدن در گروه های مرجع
5. خرید به خاطر پذیرش تبلیغات و بازاریابی
6. خرید به خاطر یک دوره سختی و احتیاج به پرداختن به خود
7. خرید به خاطر فداکاری

**سؤال پیشنهادی پایلوت:**

چه الگوهای خریدی می‌توانند باعث هدررفت پول شوند؟

**پاسخ پیشنهادی:**

الگوهای خریدی که می‌توانند باعث هدررفت پول شوند شامل این موارد هستند:

1. خرید با تنوع و حجم زیاد
2. خرید به خاطر تنش عاطفی
3. خرید به خاطر هیجان زودگذر
4. خرید به خاطر پذیرفته شدن در گروه‌های مرجع
5. خرید به خاطر پذیرش تبلیغات و بازاریابی
6. خرید به خاطر یک دوره سختی و نیاز به پرداختن به خود
7. خرید به خاطر فداکاری

**نوع پایلوت:**

AUTHOR_FEEDBACK_CONTENT

---

### مورد 27 — روش‌های افزایش درآمد

**سؤال اصلی دموی فاز یک:**

روش های افزایش درآمد را توضیح بده

**بازخورد مستقیم دکتر:**

ترتیب آن مناسب نیست.

1. فروش وسایلی که واقعا نیازی نداریم.
2. پیشنهاد پروژه و کار بیشتر به کارفرمایی که الان برایش کار می کنیم.
3. خرید دارایی درآمد زا و سرمایه گذاری دوباره سود آن
4. ایجاد درآمد منفعل یعنی درآمدی که وابسته به فعالیت شغلی من نباشد

**سؤال پیشنهادی پایلوت:**

چه روش‌هایی برای افزایش درآمد پیشنهاد می‌شود؟

**پاسخ پیشنهادی:**

روش‌های پیشنهادی برای افزایش درآمد به این ترتیب‌اند:

1. فروش وسایلی که واقعاً به آن‌ها نیاز نداریم.
2. پیشنهاد پروژه یا کار بیشتر به کارفرمای فعلی.
3. خرید دارایی درآمدزا و سرمایه‌گذاری دوباره سود آن.
4. ایجاد درآمد منفعل؛ یعنی درآمدی که وابسته به فعالیت شغلی فرد نباشد.

**نوع پایلوت:**

AUTHOR_FEEDBACK_CONTENT

---

### مورد 28 — احساس فقر با وجود داشتن درآمد

**سؤال اصلی دموی فاز یک:**

من پول در می آورم اما هنوز احساس می کنم فقیر هستم!

**بازخورد مستقیم دکتر:**

جواب خوبی است که مساله فقر مساله پول درآوردن و خرج کردن نیست. مساله رابطه با پول است . پول به معنای چیزی است که نداریم و تلاش می کنیم آن چیز را با پول جبران کنیم. در نتیجه مثلا اعتماد به نفس را به پول به دست بیارویم.
ضمن اینکه همه احساس فقر می کنند چون فقر به معنی نرسیدن به چیزی است که می خواهیم و همه چیزهایی می خواهند که ندارند و در نتیجه فقیر هستند.

**سؤال پیشنهادی پایلوت:**

چرا ممکن است فردی با وجود درآمد داشتن همچنان احساس فقر کند؟

**پاسخ پیشنهادی:**

احساس فقر فقط به میزان پول درآوردن یا خرج کردن مربوط نیست و می‌تواند به رابطه فرد با پول مربوط باشد. ممکن است فرد تلاش کند با پول چیزی را که احساس می‌کند ندارد جبران کند؛ برای مثال، اعتمادبه‌نفس را از طریق پول به دست آورد. همچنین در این دیدگاه، فقر به معنای نرسیدن به چیزی است که فرد می‌خواهد و چون انسان‌ها همواره چیزهایی می‌خواهند که ندارند، احساس فقر می‌تواند حتی با وجود داشتن درآمد باقی بماند.

**نوع پایلوت:**

AUTHOR_FEEDBACK_CONTENT

---

### وضعیت Batch

در پایان Cell بعدی باید دقیقاً ۶ Candidate Pilot جدید با شماره‌های 23 تا 28 ثبت شده باشند.

این مرحله نباید:

- ۲۲ پایلوت قبلی را تغییر دهد؛
- Author Review قبلی را تغییر دهد؛
- Case Count فعلی ۲۲تایی Author Review را به ۲۸ تغییر دهد؛
- Feedback خام دکتر را بازنویسی کند؛
- Commit یا Push انجام دهد.

پس از PASS این Batch، مرحله بعدی فقط یک Promotion Batch خواهد بود که مجموعه بازبینی مؤلف را از ۲۲ به ۲۸ مورد ارتقا می‌دهد.

</div>

In [3]:
# Code Cell 107
# ============================================================
# PHASE-1 AUTHOR FEEDBACK -> DERIVED PILOTS 23-28
#
# Batch import + pilot construction for ALL SIX cases.
#
# This cell is intentionally self-contained.
#
# It:
# - loads the durable Human Review Registry;
# - preserves the exact historical Phase-1 demo material;
# - creates six deterministic derived Pilot records;
# - classifies five as AUTHOR_FEEDBACK_CONTENT;
# - classifies one as MULTI_TURN_CONTEXT_CONTINUITY;
# - records the proposed final Question/Answer or expected behavior;
# - stores all six in a separate second-review Registry collection;
# - prepares them for later Author Review promotion;
# - verifies that the original 22 Author Review cases remain untouched;
# - creates a Registry backup before writing;
# - reloads and validates the exact persisted state;
# - is idempotent.
#
# It DOES NOT:
# - append these six to the existing 22-case Author Review queue;
# - modify any of the 22 existing completed Human Review cases;
# - treat Phase-1 model answers as authoritative evidence;
# - alter Dr. Roudi's raw historical feedback;
# - commit or push.
# ============================================================

from pathlib import Path
from copy import deepcopy
from datetime import datetime, timezone
import hashlib
import json


# ============================================================
# 1) Constants
# ============================================================

CELL107_BATCH_VERSION = (
    "phase1_author_feedback_derived_pilots_v1"
)

CELL107_ORIGIN = (
    "PHASE1_DEMO_AUTHOR_FEEDBACK"
)

CELL107_AUTHORITY = (
    "PRIMARY_DIRECT_AUTHOR_FEEDBACK"
)

CELL107_EXPECTED_EXISTING_AUTHOR_CASE_COUNT = 22

CELL107_EXPECTED_NEW_CASE_COUNT = 6

CELL107_FUTURE_TOTAL_REFERENCE_PILOTS = 28


# ============================================================
# 2) Utilities
# ============================================================

cell107_failures = []


def cell107_check(
    name,
    condition,
    details=None,
):

    if not bool(
        condition
    ):

        failure = {
            "reason":
                name,
        }

        if details is not None:

            failure[
                "details"
            ] = details

        cell107_failures.append(
            failure
        )

    return bool(
        condition
    )


def cell107_stable_id(
    prefix,
    payload,
):

    if not isinstance(
        payload,
        str,
    ):

        payload = json.dumps(
            payload,
            ensure_ascii=False,
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
        )

    digest = hashlib.sha256(
        payload.encode(
            "utf-8"
        )
    ).hexdigest()[:32]

    return (
        prefix
        +
        ":"
        +
        digest
    )


def cell107_sha256_text(
    text,
):

    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


def cell107_now():

    return (
        datetime.now(
            timezone.utc
        )
        .replace(
            microsecond=0
        )
        .isoformat()
    )


def cell107_find_repo_root():

    candidates = []

    if (
        "cell105_registry_path"
        in globals()
    ):

        registry_path = Path(
            cell105_registry_path
        ).resolve()

        candidates.extend(
            [
                registry_path.parent,
                *registry_path.parents,
            ]
        )

    cwd = Path.cwd().resolve()

    candidates.extend(
        [
            cwd,
            *cwd.parents,
        ]
    )

    seen = set()

    for candidate in candidates:

        candidate = (
            candidate.resolve()
        )

        if str(
            candidate
        ) in seen:

            continue

        seen.add(
            str(
                candidate
            )
        )

        expected_registry = (
            candidate
            /
            "data"
            /
            "phase2"
            /
            "human_review"
            /
            "human_review_registry.json"
        )

        if expected_registry.exists():

            return candidate

    return None


# ============================================================
# 3) Resolve Registry independently
# ============================================================

cell107_repo_root = (
    cell107_find_repo_root()
)


cell107_check(
    "REPO_ROOT_NOT_FOUND",
    cell107_repo_root
    is not None,
)


if cell107_failures:

    raise AssertionError(
        cell107_failures
    )


cell107_registry_path = (
    cell107_repo_root
    /
    "data"
    /
    "phase2"
    /
    "human_review"
    /
    "human_review_registry.json"
)


cell107_check(
    "REGISTRY_NOT_FOUND",
    cell107_registry_path.exists(),
    str(
        cell107_registry_path
    ),
)


if cell107_failures:

    raise AssertionError(
        cell107_failures
    )


# ============================================================
# 4) Exact historical Phase-1 records
#
# IMPORTANT:
# Raw questions, model answers and author feedback are preserved
# as historical evidence. Derived Q/A are separate fields.
# ============================================================

CELL107_CASES = [


    # --------------------------------------------------------
    # Pilot 23
    # --------------------------------------------------------

    {
        "reference_pilot_number":
            23,

        "feedback_case_number":
            1,

        "pilot_type":
            "AUTHOR_FEEDBACK_CONTENT",

        "raw_conversation": [
            {
                "speaker":
                    "HUMAN",

                "text":
                    "سواد مالی درباره پول دار شدن دیگه؟",
            },

            {
                "speaker":
                    "PHASE1_MODEL",

                "text":
                    (
                        "سواد مالی به رابطۀ انسان با پول می‌پردازد، "
                        "نه فقط به خود پول. پول در سواد مالی، یک "
                        "پدیده‌ای بیرونی نیست، بلکه احساسات و رفتارهای "
                        "انسان نسبت به پول است. سواد مالی به دانش و "
                        "مهارت‌های لازم برای مدیریت بهتر پول و "
                        "تصمیم‌گیری‌های مالی اشاره دارد."
                    ),
            },
        ],

        "raw_author_feedback":
            (
                "نه ، ببین سواد مالی درباره مدیریت پول است. "
                "نه درباره میزان پول"
            ),

        "derived_question":
            (
                "سواد مالی درباره میزان پولی است که داریم "
                "یا درباره مدیریت پول؟"
            ),

        "derived_expected_answer":
            (
                "سواد مالی درباره مدیریت پول است، "
                "نه درباره میزان پول."
            ),
    },


    # --------------------------------------------------------
    # Pilot 24
    # --------------------------------------------------------

    {
        "reference_pilot_number":
            24,

        "feedback_case_number":
            2,

        "pilot_type":
            "AUTHOR_FEEDBACK_CONTENT",

        "raw_conversation": [
            {
                "speaker":
                    "HUMAN",

                "text":
                    (
                        "ایا من از نظر مالی وضعیت قابل قبولی دارم؟ "
                        "از نظر دانشی و مهارتی یا حتی نگرشی؟"
                    ),
            },

            {
                "speaker":
                    "PHASE1_MODEL",

                "text":
                    (
                        "پاسخ شما در منابع موجود پاسخ کافی پیدا نشد."
                    ),
            },
        ],

        "raw_author_feedback":
            (
                "مدیریت مالی شخصی نیازمند دانش و نگرش و "
                "مهارت های خاصی است، برا یاینکه بدانید شما "
                "به اندازه کافی وضعیت مالی خوبی دارید ، لازم "
                "است این پرسشنامه را تکمیل کنید."
            ),

        "derived_question":
            (
                "برای اینکه بفهمیم از نظر دانش، نگرش و "
                "مهارت‌های مالی وضعیت قابل قبولی داریم، "
                "چه باید کرد؟"
            ),

        "derived_expected_answer":
            (
                "مدیریت مالی شخصی به دانش، نگرش و مهارت‌های "
                "خاص نیاز دارد. برای اینکه بدانیم وضعیت مالی "
                "ما از این جنبه‌ها به اندازه کافی مناسب است، "
                "باید پرسشنامه مربوط را تکمیل کنیم."
            ),
    },


    # --------------------------------------------------------
    # Pilot 25
    # --------------------------------------------------------

    {
        "reference_pilot_number":
            25,

        "feedback_case_number":
            3,

        "pilot_type":
            "MULTI_TURN_CONTEXT_CONTINUITY",

        "raw_conversation": [
            {
                "speaker":
                    "HUMAN",

                "turn":
                    1,

                "text":
                    (
                        "نمی دانم پولم کجا می رود ؟ "
                        "انگار پولم را گم می کنم."
                    ),
            },

            {
                "speaker":
                    "PHASE1_MODEL",

                "turn":
                    1,

                "text":
                    (
                        "پول شما ممکن است به دلیل عدم توجه به جزئیات "
                        "هزینه‌ها و انجام خریدهایی بدون نظارت، به طور "
                        "غیر منظمی خرج شود. بهترین راه برای کشف این "
                        "موضوع، ثبت دقیق هزینه‌هاست تا بدانیم چه چیزی "
                        "را کجاها خرج کرده‌اید. این کار به شما کمک "
                        "می‌کند تا بهترین تصمیم‌ها را بگیرید و از "
                        "هدررفت پول جلوگیری کنید."
                    ),
            },

            {
                "speaker":
                    "HUMAN",

                "turn":
                    2,

                "text":
                    (
                        "این جواب خوبی بود. ولی کوتاه است "
                        "بیشتر توضیح می دهی."
                    ),
            },

            {
                "speaker":
                    "PHASE1_MODEL",

                "turn":
                    2,

                "text":
                    (
                        "قرض گرفتن یک راه حلی است که ممکن است در "
                        "موقعیت‌هایی مانند نیاز به پول سریع یا سرمایه "
                        "گذاری، مفید باشد. اما اگر به طور مداوم و بدون "
                        "نیت مشخص انجام شود، می‌تواند به یک معضل تبدیل "
                        "شود. قرض گرفتن بهتر است زمانی باشد که بتوانیم "
                        "آن را به خوبی بازپرداخت کنیم و از آن برای "
                        "هزینه‌های ضروری یا سرمایه‌گذاری استفاده کنیم. "
                        "اما اگر به صورت مداوم و بدون برنامه‌ریزی، "
                        "انجام شود، می‌تواند به دلیل افزایش بدهی‌ها و "
                        "مشکلات مالی، به یک معضل تبدیل شود. بهتر است "
                        "از قرض به‌عنوان یک ابزار کمکی استفاده کنیم، "
                        "نه یک راه حل دائمی."
                    ),
            },
        ],

        "raw_author_feedback":
            "این خیلی حواب بی ربطی بود.",

        "derived_question":
            (
                "این جواب خوبی بود. ولی کوتاه است "
                "بیشتر توضیح می‌دهی؟"
            ),

        "derived_expected_answer":
            None,

        "expected_behavior":
            (
                "پاسخ باید همان موضوع Turn قبلی، یعنی ثبت و بررسی "
                "هزینه‌ها برای فهمیدن اینکه پول کجا می‌رود، را "
                "ادامه و بسط دهد. پاسخ نباید بدون ارتباط با Context "
                "به موضوع دیگری مانند قرض گرفتن تغییر مسیر دهد."
            ),

        "golden_content_case":
            False,
    },


    # --------------------------------------------------------
    # Pilot 26
    # --------------------------------------------------------

    {
        "reference_pilot_number":
            26,

        "feedback_case_number":
            4,

        "pilot_type":
            "AUTHOR_FEEDBACK_CONTENT",

        "raw_conversation": [
            {
                "speaker":
                    "HUMAN",

                "turn":
                    1,

                "text":
                    (
                        "چند تا راهکار بده که بتونم پس انداز کنم"
                    ),
            },

            {
                "speaker":
                    "PHASE1_MODEL",

                "turn":
                    1,

                "text":
                    (
                        "سه راهکار برای پس‌انداز عبارتند از:\n"
                        "1. تعیین اهداف مالی مشخص و پس‌اندازی برای "
                        "رسیدن به آن‌ها.\n"
                        "2. کاهش هزینه‌های غیر ضروری و جایگزینی آن‌ها "
                        "با اموری که باعث افزایش ثروت می‌شوند.\n"
                        "3. سرمایه‌گذاری پس‌اندازها در گزینه‌هایی که "
                        "حداقل به اندازۀ تورم بازدهی داشته باشند."
                    ),
            },

            {
                "speaker":
                    "HUMAN",

                "turn":
                    2,

                "text":
                    (
                        "هزینه غیرضروری ندارم. همه اش ضروریه . "
                        "یعنی چی هزینه غیرضروری؟"
                    ),
            },

            {
                "speaker":
                    "PHASE1_MODEL",

                "turn":
                    2,

                "text":
                    (
                        "هزینه غیرضروری به معنای خرج کردن پول برای "
                        "چیزهایی است که لازم نیستند یا بهبود کیفیت "
                        "زندگی ما را فراهم نمی‌کنند. به عبارت دیگر، "
                        "چیزهایی که به طور مداوم یا بدون نیاز به "
                        "خریداری می‌شوند و ممکن است باعث هدررفت پول "
                        "شوند. به طور کلی، هزینه‌های غیرضروری، "
                        "هزینه‌هایی هستند که به طور غیرمتعارف یا "
                        "بدون نیتی در حساب کشیده می‌شوند."
                    ),
            },

            {
                "speaker":
                    "HUMAN",

                "turn":
                    3,

                "text":
                    "چند نمونه برام بگو",
            },

            {
                "speaker":
                    "PHASE1_MODEL",

                "turn":
                    3,

                "text":
                    (
                        "چند نمونه از گزینه‌های سرمایه‌گذاری عبارتند از:\n"
                        "- سرمایه‌گذاری در اوراق بهادار\n"
                        "- سرمایه‌گذاری در املاک و مستغلات\n"
                        "- سرمایه‌گذاری در صندوق‌های سرمایه\n"
                        "- سرمایه‌گذاری در بانک‌ها با سود بالا\n"
                        "- سرمایه‌گذاری در طلا یا سایر فلزات گرانبها."
                    ),
            },
        ],

        "raw_author_feedback":
            (
                "آنچه باعث می شود پول هایم هدر روند ، "
                "الگوی خرید نامناسب است مانند \n"
                "1. خرید با تنوع و حجم زیاد\n"
                "2. خرید به خاطر تنش عاطفی \n"
                "3. خرید به خاطر هیجان زود گذر \n"
                "4. خرید به خاطر پذیرفته شدن در گروه های مرجع \n"
                "5. خرید به خاطر پذیرش تبلیغات و بازاریابی \n"
                "6. خرید به خاطر یک دوره سختی و احتیاج به پرداختن به خود \n"
                "7. خرید به خاطر فداکاری"
            ),

        "derived_question":
            (
                "چه الگوهای خریدی می‌توانند باعث هدررفت پول شوند؟"
            ),

        "derived_expected_answer":
            (
                "الگوهای خریدی که می‌توانند باعث هدررفت پول شوند "
                "شامل این موارد هستند:\n"
                "1. خرید با تنوع و حجم زیاد\n"
                "2. خرید به خاطر تنش عاطفی\n"
                "3. خرید به خاطر هیجان زودگذر\n"
                "4. خرید به خاطر پذیرفته شدن در گروه‌های مرجع\n"
                "5. خرید به خاطر پذیرش تبلیغات و بازاریابی\n"
                "6. خرید به خاطر یک دوره سختی و نیاز به پرداختن به خود\n"
                "7. خرید به خاطر فداکاری"
            ),
    },


    # --------------------------------------------------------
    # Pilot 27
    # --------------------------------------------------------

    {
        "reference_pilot_number":
            27,

        "feedback_case_number":
            5,

        "pilot_type":
            "AUTHOR_FEEDBACK_CONTENT",

        "raw_conversation": [
            {
                "speaker":
                    "HUMAN",

                "text":
                    "روش های افزایش درآمد را توضیح بده",
            },

            {
                "speaker":
                    "PHASE1_MODEL",

                "text":
                    (
                        "روش‌های افزایش درآمد عبارتند از:\n"
                        "1. قیمت‌گذاری مناسب محصولات\n"
                        "2. سرمایه‌گذاری درست\n"
                        "3. خرید دارایی‌های درآمدزا\n"
                        "4. تغییر نقشه کسب‌وکار"
                    ),
            },
        ],

        "raw_author_feedback":
            (
                "ترتیب آن مناسب نیست. \n\n"
                "1. فروش وسایلی که واقعا نیازی نداریم.\n"
                "2. پیشنهاد پروژه و کار بیشتر به کارفرمایی که "
                "الان برایش کار می کنیم. \n"
                "3. خرید دارایی درآمد زا و سرمایه گذاری دوباره سود آن \n"
                "4. ایجاد درآمد منفعل یعنی درآمدی که وابسته به "
                "فعالیت شغلی من نباشد"
            ),

        "derived_question":
            (
                "چه روش‌هایی برای افزایش درآمد پیشنهاد می‌شود؟"
            ),

        "derived_expected_answer":
            (
                "روش‌های پیشنهادی برای افزایش درآمد به این ترتیب‌اند:\n"
                "1. فروش وسایلی که واقعاً به آن‌ها نیاز نداریم.\n"
                "2. پیشنهاد پروژه یا کار بیشتر به کارفرمای فعلی.\n"
                "3. خرید دارایی درآمدزا و سرمایه‌گذاری دوباره سود آن.\n"
                "4. ایجاد درآمد منفعل؛ یعنی درآمدی که وابسته به "
                "فعالیت شغلی فرد نباشد."
            ),
    },


    # --------------------------------------------------------
    # Pilot 28
    # --------------------------------------------------------

    {
        "reference_pilot_number":
            28,

        "feedback_case_number":
            6,

        "pilot_type":
            "AUTHOR_FEEDBACK_CONTENT",

        "raw_conversation": [
            {
                "speaker":
                    "HUMAN",

                "text":
                    (
                        "من پول در می آورم اما هنوز احساس می کنم "
                        "فقیر هستم!"
                    ),
            },

            {
                "speaker":
                    "PHASE1_MODEL",

                "text":
                    (
                        "شما ممکن است به دلیل ارتباط نادرست با پول "
                        "و ترس از نداشتن پول، احساس فقر کنید. پول تنها "
                        "ابزاری برای ثابت کردن خود و امنیت نیست، بلکه "
                        "رابطه‌ای است که با آن سروکار داریم. اگر "
                        "تصمیم‌های نادرستی را در گذشته گرفته باشید، "
                        "ممکن است این احساس فقر را ایجاد کرده باشد. "
                        "بهتر است به جای توجه به مقدار پول، به رابطه‌ای "
                        "که با پول دارید نیز توجه کنید. همچنین، پول را "
                        "باید به کار برد تا ارزشش حفظ شود. بهترین راه "
                        "برای جلوگیری از احساس فقر، توجه به سواد مالی "
                        "و تصمیم‌گیری‌های صحیح است."
                    ),
            },
        ],

        "raw_author_feedback":
            (
                "جواب خوبی است که مساله فقر مساله پول درآوردن و "
                "خرج کردن نیست. مساله رابطه با پول است . پول به "
                "معنای چیزی است که نداریم و تلاش می کنیم آن چیز را "
                "با پول جبران کنیم. در نتیجه مثلا اعتماد به نفس را "
                "به پول به دست بیارویم.\n"
                "ضمن اینکه همه احساس فقر می کنند چون فقر به معنی "
                "نرسیدن به چیزی است که می خواهیم و همه چیزهایی "
                "می خواهند که ندارند و در نتیجه فقیر هستند."
            ),

        "derived_question":
            (
                "چرا ممکن است فردی با وجود درآمد داشتن "
                "همچنان احساس فقر کند؟"
            ),

        "derived_expected_answer":
            (
                "احساس فقر فقط به میزان پول درآوردن یا خرج کردن "
                "مربوط نیست و می‌تواند به رابطه فرد با پول مربوط "
                "باشد. ممکن است فرد تلاش کند با پول چیزی را که "
                "احساس می‌کند ندارد جبران کند؛ برای مثال، "
                "اعتمادبه‌نفس را از طریق پول به دست آورد. همچنین "
                "در این دیدگاه، فقر به معنای نرسیدن به چیزی است "
                "که فرد می‌خواهد و چون انسان‌ها همواره چیزهایی "
                "می‌خواهند که ندارند، احساس فقر می‌تواند حتی با "
                "وجود داشتن درآمد باقی بماند."
            ),
    },
]


# ============================================================
# 5) Batch structural validation BEFORE touching Registry
# ============================================================

cell107_check(
    "INPUT_CASE_COUNT_MISMATCH",
    len(
        CELL107_CASES
    )
    ==
    CELL107_EXPECTED_NEW_CASE_COUNT,
)


cell107_numbers = [
    item[
        "reference_pilot_number"
    ]

    for item
    in CELL107_CASES
]


cell107_check(
    "REFERENCE_NUMBERS_MISMATCH",
    cell107_numbers
    ==
    [
        23,
        24,
        25,
        26,
        27,
        28,
    ],
)


cell107_check(
    "REFERENCE_NUMBERS_NOT_UNIQUE",
    len(
        set(
            cell107_numbers
        )
    )
    ==
    6,
)


cell107_feedback_numbers = [
    item[
        "feedback_case_number"
    ]

    for item
    in CELL107_CASES
]


cell107_check(
    "FEEDBACK_NUMBERS_MISMATCH",
    cell107_feedback_numbers
    ==
    [
        1,
        2,
        3,
        4,
        5,
        6,
    ],
)


cell107_check(
    "CONTENT_CASE_COUNT_MISMATCH",
    sum(
        1

        for item
        in CELL107_CASES

        if (
            item[
                "pilot_type"
            ]
            ==
            "AUTHOR_FEEDBACK_CONTENT"
        )
    )
    ==
    5,
)


cell107_check(
    "CONTINUITY_CASE_COUNT_MISMATCH",
    sum(
        1

        for item
        in CELL107_CASES

        if (
            item[
                "pilot_type"
            ]
            ==
            "MULTI_TURN_CONTEXT_CONTINUITY"
        )
    )
    ==
    1,
)


for item in CELL107_CASES:

    cell107_check(
        (
            "EMPTY_AUTHOR_FEEDBACK_CASE_"
            +
            str(
                item[
                    "feedback_case_number"
                ]
            )
        ),
        bool(
            item.get(
                "raw_author_feedback"
            )
        ),
    )

    cell107_check(
        (
            "EMPTY_DERIVED_QUESTION_CASE_"
            +
            str(
                item[
                    "feedback_case_number"
                ]
            )
        ),
        bool(
            item.get(
                "derived_question"
            )
        ),
    )

    if (
        item[
            "pilot_type"
        ]
        ==
        "AUTHOR_FEEDBACK_CONTENT"
    ):

        cell107_check(
            (
                "EMPTY_EXPECTED_ANSWER_CASE_"
                +
                str(
                    item[
                        "feedback_case_number"
                    ]
                )
            ),
            bool(
                item.get(
                    "derived_expected_answer"
                )
            ),
        )

    else:

        cell107_check(
            "CONTINUITY_EXPECTED_BEHAVIOR_MISSING",
            bool(
                item.get(
                    "expected_behavior"
                )
            ),
        )


if cell107_failures:

    print(
        cell107_failures
    )

    raise AssertionError(
        "Cell 107 input batch failed structural validation."
    )


# ============================================================
# 6) Build immutable derived Pilot records
# ============================================================

cell107_timestamp = (
    cell107_now()
)


cell107_derived_pilots = []


for source_case in CELL107_CASES:

    feedback_case_number = (
        source_case[
            "feedback_case_number"
        ]
    )

    reference_number = (
        source_case[
            "reference_pilot_number"
        ]
    )

    raw_feedback = (
        source_case[
            "raw_author_feedback"
        ]
    )

    feedback_source_id = (
        cell107_stable_id(
            "author-feedback-source",
            {
                "origin":
                    CELL107_ORIGIN,

                "feedback_case_number":
                    feedback_case_number,

                "raw_author_feedback":
                    raw_feedback,
            },
        )
    )


    pilot_case_id = (
        cell107_stable_id(
            "question-pilot",
            {
                "origin":
                    CELL107_ORIGIN,

                "feedback_case_number":
                    feedback_case_number,

                "reference_pilot_number":
                    reference_number,

                "pilot_type":
                    source_case[
                        "pilot_type"
                    ],

                "feedback_source_id":
                    feedback_source_id,
            },
        )
    )


    question_draft_id = (
        cell107_stable_id(
            "question-draft",
            {
                "pilot_case_id":
                    pilot_case_id,

                "question":
                    source_case[
                        "derived_question"
                    ],
            },
        )
    )


    expected_answer_draft_id = None

    citation_mapping_id = None


    if (
        source_case[
            "pilot_type"
        ]
        ==
        "AUTHOR_FEEDBACK_CONTENT"
    ):

        expected_answer_draft_id = (
            cell107_stable_id(
                "expected-answer-draft",
                {
                    "pilot_case_id":
                        pilot_case_id,

                    "question_draft_id":
                        question_draft_id,

                    "answer":
                        source_case[
                            "derived_expected_answer"
                        ],
                },
            )
        )


        citation_mapping_id = (
            cell107_stable_id(
                "citation-mapping",
                {
                    "pilot_case_id":
                        pilot_case_id,

                    "expected_answer_draft_id":
                        expected_answer_draft_id,

                    "feedback_source_id":
                        feedback_source_id,
                },
            )
        )


    else:

        citation_mapping_id = (
            cell107_stable_id(
                "behavior-mapping",
                {
                    "pilot_case_id":
                        pilot_case_id,

                    "question_draft_id":
                        question_draft_id,

                    "expected_behavior":
                        source_case[
                            "expected_behavior"
                        ],

                    "feedback_source_id":
                        feedback_source_id,
                },
            )
        )


    record = {
        "record_schema":
            "AUTHOR_FEEDBACK_DERIVED_PILOT_V1",

        "batch_version":
            CELL107_BATCH_VERSION,

        "reference_pilot_number":
            reference_number,

        "pilot_case_id":
            pilot_case_id,

        "feedback_case_number":
            feedback_case_number,

        "origin":
            CELL107_ORIGIN,

        "pilot_type":
            source_case[
                "pilot_type"
            ],

        "authority": {
            "authority_class":
                CELL107_AUTHORITY,

            "primary_author":
                "Dr. Komeil Roudi",

            "phase1_model_answer_authoritative":
                False,

            "raw_author_feedback_authoritative":
                True,
        },

        "historical_phase1_demo": {
            "raw_conversation":
                deepcopy(
                    source_case[
                        "raw_conversation"
                    ]
                ),

            "raw_author_feedback":
                raw_feedback,

            "raw_author_feedback_sha256":
                cell107_sha256_text(
                    raw_feedback
                ),

            "raw_feedback_preserved_verbatim":
                True,
        },

        "feedback_source": {
            "feedback_source_id":
                feedback_source_id,

            "source_type":
                "DIRECT_AUTHOR_FEEDBACK",

            "source_scope":
                "PHASE1_DEMO",

            "source_mapping_to_phase2_corpus":
                "NOT_REQUIRED_FOR_AUTHOR_FEEDBACK_AUTHORITY",

            "optional_corpus_cross_mapping_status":
                "NOT_STARTED",
        },

        "derived_question": {
            "question_draft_id":
                question_draft_id,

            "question_text":
                source_case[
                    "derived_question"
                ],

            "review_origin":
                "SECOND_HUMAN_REVIEW_PHASE1_AUTHOR_FEEDBACK",

            "human_decision":
                "ACCEPT",

            "external_knowledge_used":
                False,
        },

        "second_human_review": {
            "status":
                "COMPLETE",

            "decision":
                "ACCEPT_DERIVED_PILOT",

            "review_origin":
                "PHASE1_AUTHOR_FEEDBACK",

            "review_timestamp_utc":
                cell107_timestamp,

            "raw_feedback_preserved":
                True,

            "phase1_model_answer_used_as_ground_truth":
                False,

            "author_review_status":
                "NOT_STARTED",
        },

        "author_final_review": {
            "status":
                "NOT_STARTED",

            "reviewer":
                "AUTHOR",

            "decision":
                None,

            "notes":
                None,
        },

        "promotion": {
            "status":
                "READY_FOR_AUTHOR_REVIEW",

            "currently_in_22_case_author_queue":
                False,

            "future_reference_pilot_number":
                reference_number,

            "future_target_total":
                CELL107_FUTURE_TOTAL_REFERENCE_PILOTS,
        },
    }


    if (
        source_case[
            "pilot_type"
        ]
        ==
        "AUTHOR_FEEDBACK_CONTENT"
    ):

        record[
            "expected_answer_review"
        ] = {
            "expected_answer_draft_id":
                expected_answer_draft_id,

            "expected_answer_text":
                source_case[
                    "derived_expected_answer"
                ],

            "human_decision":
                "ACCEPT",

            "grounding_source":
                "DIRECT_AUTHOR_FEEDBACK",

            "external_knowledge_used":
                False,
        }


        record[
            "citation_review"
        ] = {
            "citation_mapping_id":
                citation_mapping_id,

            "human_decision":
                "ACCEPT",

            "citation_entries": [
                {
                    "citation_index":
                        0,

                    "source_type":
                        "DIRECT_AUTHOR_FEEDBACK",

                    "feedback_source_id":
                        feedback_source_id,

                    "raw_author_feedback_sha256":
                        cell107_sha256_text(
                            raw_feedback
                        ),

                    "authority_class":
                        CELL107_AUTHORITY,
                },
            ],
        }


        record[
            "golden_content_case"
        ] = True


    else:

        record[
            "expected_behavior_review"
        ] = {
            "behavior_mapping_id":
                citation_mapping_id,

            "expected_behavior":
                source_case[
                    "expected_behavior"
                ],

            "human_decision":
                "ACCEPT",

            "grounding_source":
                "DIRECT_AUTHOR_FEEDBACK_PLUS_CONVERSATION_CONTEXT",

            "external_knowledge_used":
                False,
        }


        record[
            "golden_content_case"
        ] = False


        record[
            "regression_role"
        ] = (
            "MULTI_TURN_CONTEXT_CONTINUITY"
        )


    cell107_derived_pilots.append(
        record
    )


# ============================================================
# 7) Deterministic-ID validation
# ============================================================

cell107_pilot_ids = [
    item[
        "pilot_case_id"
    ]

    for item
    in cell107_derived_pilots
]


cell107_check(
    "DERIVED_PILOT_ID_COUNT_MISMATCH",
    len(
        cell107_pilot_ids
    )
    ==
    6,
)


cell107_check(
    "DERIVED_PILOT_IDS_NOT_UNIQUE",
    len(
        set(
            cell107_pilot_ids
        )
    )
    ==
    6,
)


if cell107_failures:

    raise AssertionError(
        cell107_failures
    )


# ============================================================
# 8) Load Registry fresh
# ============================================================

cell107_registry_before_text = (
    cell107_registry_path.read_text(
        encoding="utf-8"
    )
)


cell107_registry = json.loads(
    cell107_registry_before_text
)


cell107_registry_before_sha256 = (
    hashlib.sha256(
        cell107_registry_before_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# 9) Protect the existing 22-case collections
# ============================================================

cell107_completed_before = deepcopy(
    cell107_registry.get(
        "completed_human_review_cases",
        []
    )
)


cell107_author_review_before = deepcopy(
    cell107_registry.get(
        "author_review",
        {}
    )
)


cell107_existing_author_cases_before = (
    (
        cell107_author_review_before
        or
        {}
    ).get(
        "cases_ready_for_review",
        []
    )
    or
    []
)


cell107_check(
    "PREEXISTING_AUTHOR_QUEUE_NOT_22",
    len(
        cell107_existing_author_cases_before
    )
    ==
    CELL107_EXPECTED_EXISTING_AUTHOR_CASE_COUNT,
    len(
        cell107_existing_author_cases_before
    ),
)


if cell107_failures:

    raise AssertionError(
        cell107_failures
    )


# ============================================================
# 10) Create / update separate Second Review Batch
# ============================================================

cell107_second_review_root = (
    cell107_registry.setdefault(
        "second_human_review",
        {},
    )
)


cell107_feedback_batch = (
    cell107_second_review_root.setdefault(
        "phase1_author_feedback_derived_pilots",
        {},
    )
)


existing_records = (
    cell107_feedback_batch.get(
        "cases",
        []
    )
    or
    []
)


existing_by_id = {
    item.get(
        "pilot_case_id"
    ):
        item

    for item
    in existing_records

    if isinstance(
        item,
        dict,
    )
}


for new_record in cell107_derived_pilots:

    pilot_id = (
        new_record[
            "pilot_case_id"
        ]
    )

    if pilot_id in existing_by_id:

        cell107_check(
            (
                "EXISTING_RECORD_CONTENT_MISMATCH:"
                +
                pilot_id
            ),
            existing_by_id[
                pilot_id
            ]
            ==
            new_record,
        )

    else:

        existing_records.append(
            deepcopy(
                new_record
            )
        )


if cell107_failures:

    print(
        cell107_failures
    )

    raise AssertionError(
        "Existing Cell 107 batch conflicts with the deterministic rebuild."
    )


# Keep only this Batch's six deterministic records in sorted order.
cell107_batch_ids = set(
    cell107_pilot_ids
)


cell107_exact_batch_records = [
    record

    for record
    in existing_records

    if (
        isinstance(
            record,
            dict,
        )
        and
        record.get(
            "pilot_case_id"
        )
        in
        cell107_batch_ids
    )
]


cell107_exact_batch_records = sorted(
    cell107_exact_batch_records,
    key=lambda item: (
        item[
            "reference_pilot_number"
        ]
    ),
)


cell107_check(
    "PERSISTED_BATCH_CASE_COUNT_NOT_6",
    len(
        cell107_exact_batch_records
    )
    ==
    6,
)


cell107_feedback_batch.update({
    "batch_version":
        CELL107_BATCH_VERSION,

    "origin":
        CELL107_ORIGIN,

    "status":
        "COMPLETE_READY_FOR_AUTHOR_REVIEW_PROMOTION",

    "case_count":
        6,

    "reference_pilot_range":
        [
            23,
            28,
        ],

    "content_pilot_count":
        5,

    "continuity_regression_pilot_count":
        1,

    "future_total_reference_pilots":
        28,

    "currently_promoted_to_author_review":
        False,

    "author_review_queue_count_before_promotion":
        22,

    "cases":
        cell107_exact_batch_records,
})


# ============================================================
# 11) Confirm original 22 collections were not mutated
# ============================================================

cell107_check(
    "COMPLETED_22_CASES_MUTATED",
    cell107_registry.get(
        "completed_human_review_cases",
        []
    )
    ==
    cell107_completed_before,
)


cell107_check(
    "AUTHOR_REVIEW_ROOT_MUTATED",
    cell107_registry.get(
        "author_review",
        {}
    )
    ==
    cell107_author_review_before,
)


if cell107_failures:

    print(
        cell107_failures
    )

    raise AssertionError(
        "Cell 107 attempted to alter the protected existing 22-case state."
    )


# ============================================================
# 12) Serialize / decide whether write is necessary
# ============================================================

cell107_registry_after_text = (
    json.dumps(
        cell107_registry,
        ensure_ascii=False,
        indent=2,
    )
    +
    "\n"
)


cell107_change_required = (
    cell107_registry_after_text
    !=
    cell107_registry_before_text
)


cell107_backup_path = None


# ============================================================
# 13) Backup + atomic-ish write
# ============================================================

if cell107_change_required:

    timestamp_for_filename = (
        datetime.now(
            timezone.utc
        ).strftime(
            "%Y%m%dT%H%M%SZ"
        )
    )


    cell107_backup_path = (
        cell107_registry_path.parent
        /
        (
            cell107_registry_path.stem
            +
            ".before_phase1_author_feedback_batch_"
            +
            timestamp_for_filename
            +
            cell107_registry_path.suffix
        )
    )


    cell107_backup_path.write_text(
        cell107_registry_before_text,
        encoding="utf-8",
    )


    cell107_temp_path = (
        cell107_registry_path.parent
        /
        (
            cell107_registry_path.name
            +
            ".cell107.tmp"
        )
    )


    cell107_temp_path.write_text(
        cell107_registry_after_text,
        encoding="utf-8",
    )


    # Parse temporary bytes before replacement.
    json.loads(
        cell107_temp_path.read_text(
            encoding="utf-8"
        )
    )


    cell107_temp_path.replace(
        cell107_registry_path
    )


# ============================================================
# 14) Reload exact persisted Registry
# ============================================================

cell107_persisted_text = (
    cell107_registry_path.read_text(
        encoding="utf-8"
    )
)


cell107_persisted_registry = (
    json.loads(
        cell107_persisted_text
    )
)


cell107_registry_after_sha256 = (
    hashlib.sha256(
        cell107_persisted_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# 15) Persisted validation
# ============================================================

cell107_persisted_completed = (
    cell107_persisted_registry.get(
        "completed_human_review_cases",
        []
    )
)


cell107_persisted_author_review = (
    cell107_persisted_registry.get(
        "author_review",
        {}
    )
)


cell107_persisted_author_cases = (
    (
        cell107_persisted_author_review
        or
        {}
    ).get(
        "cases_ready_for_review",
        []
    )
    or
    []
)


cell107_persisted_batch = (
    (
        cell107_persisted_registry.get(
            "second_human_review",
            {}
        )
        or
        {}
    ).get(
        "phase1_author_feedback_derived_pilots",
        {}
    )
    or
    {}
)


cell107_persisted_batch_cases = (
    cell107_persisted_batch.get(
        "cases",
        []
    )
    or
    []
)


cell107_check(
    "PERSISTED_COMPLETED_CASES_CHANGED",
    cell107_persisted_completed
    ==
    cell107_completed_before,
)


cell107_check(
    "PERSISTED_AUTHOR_QUEUE_COUNT_CHANGED",
    len(
        cell107_persisted_author_cases
    )
    ==
    22,
    len(
        cell107_persisted_author_cases
    ),
)


cell107_check(
    "PERSISTED_AUTHOR_REVIEW_ROOT_CHANGED",
    cell107_persisted_author_review
    ==
    cell107_author_review_before,
)


cell107_check(
    "PERSISTED_FEEDBACK_BATCH_COUNT_MISMATCH",
    len(
        cell107_persisted_batch_cases
    )
    ==
    6,
)


cell107_check(
    "PERSISTED_FEEDBACK_BATCH_STATUS_MISMATCH",
    cell107_persisted_batch.get(
        "status"
    )
    ==
    "COMPLETE_READY_FOR_AUTHOR_REVIEW_PROMOTION",
)


cell107_check(
    "PERSISTED_FEEDBACK_BATCH_PROMOTED_TOO_EARLY",
    cell107_persisted_batch.get(
        "currently_promoted_to_author_review"
    )
    is False,
)


cell107_persisted_ids = [
    item.get(
        "pilot_case_id"
    )

    for item
    in cell107_persisted_batch_cases
]


cell107_check(
    "PERSISTED_PILOT_IDS_MISMATCH",
    cell107_persisted_ids
    ==
    cell107_pilot_ids,
)


for record in cell107_persisted_batch_cases:

    cell107_check(
        (
            "AUTHOR_REVIEW_STARTED_TOO_EARLY:"
            +
            record.get(
                "pilot_case_id",
                "UNKNOWN",
            )
        ),
        (
            record.get(
                "author_final_review",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "NOT_STARTED",
    )

    cell107_check(
        (
            "PROMOTION_STATUS_MISMATCH:"
            +
            record.get(
                "pilot_case_id",
                "UNKNOWN",
            )
        ),
        (
            record.get(
                "promotion",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "READY_FOR_AUTHOR_REVIEW",
    )


# ============================================================
# 16) Final gate
# ============================================================

cell107_gate = (
    len(
        cell107_failures
    )
    ==
    0
)


print(
    "=== PHASE-1 AUTHOR FEEDBACK -> PILOTS 23-28 ==="
)


print(
    "\nBatch version:",
    CELL107_BATCH_VERSION,
)


print(
    "Origin:",
    CELL107_ORIGIN,
)


print(
    "\nNew Pilot count:",
    len(
        cell107_persisted_batch_cases
    ),
)


print(
    "Content Pilots:",
    sum(
        1

        for item
        in cell107_persisted_batch_cases

        if (
            item.get(
                "pilot_type"
            )
            ==
            "AUTHOR_FEEDBACK_CONTENT"
        )
    ),
)


print(
    "Continuity Regression Pilots:",
    sum(
        1

        for item
        in cell107_persisted_batch_cases

        if (
            item.get(
                "pilot_type"
            )
            ==
            "MULTI_TURN_CONTEXT_CONTINUITY"
        )
    ),
)


print(
    "\n=== DERIVED PILOTS ==="
)


for record in cell107_persisted_batch_cases:

    print(
        "\n"
        +
        "="
        *
        90
    )

    print(
        "Reference Pilot:",
        record[
            "reference_pilot_number"
        ],
    )

    print(
        "Pilot ID:",
        record[
            "pilot_case_id"
        ],
    )

    print(
        "Type:",
        record[
            "pilot_type"
        ],
    )

    print(
        "Question:"
    )

    print(
        record[
            "derived_question"
        ][
            "question_text"
        ]
    )

    if record.get(
        "golden_content_case"
    ):

        print(
            "Expected Answer:"
        )

        print(
            (
                record.get(
                    "expected_answer_review"
                )
                or
                {}
            ).get(
                "expected_answer_text"
            )
        )

    else:

        print(
            "Expected Behavior:"
        )

        print(
            (
                record.get(
                    "expected_behavior_review"
                )
                or
                {}
            ).get(
                "expected_behavior"
            )
        )

    print(
        "Promotion:",
        (
            record.get(
                "promotion"
            )
            or
            {}
        ).get(
            "status"
        ),
    )


print(
    "\n=== PROTECTED EXISTING STATE ==="
)


print(
    "Existing completed Human Review cases changed:",
    False,
)


print(
    "Existing Author Review queue count:",
    len(
        cell107_persisted_author_cases
    ),
)


print(
    "New six already promoted:",
    False,
)


print(
    "Future intended total after promotion:",
    28,
)


print(
    "\n=== WRITE / SAFETY ==="
)


print(
    "Registry write required:",
    cell107_change_required,
)


print(
    "Backup path:",
    (
        str(
            cell107_backup_path
        )
        if cell107_backup_path
        is not None
        else
        "NOT_NEEDED_ALREADY_APPLIED"
    ),
)


print(
    "Registry SHA256 before:",
    cell107_registry_before_sha256,
)


print(
    "Registry SHA256 after:",
    cell107_registry_after_sha256,
)


print(
    "Raw Phase-1 Author Feedback mutated:",
    False,
)


print(
    "Phase-1 model answers treated as authority:",
    False,
)


print(
    "Author Review recorded:",
    False,
)


print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)


print(
    cell107_failures
)


print(
    "\nCODE CELL 107 GATE:",
    (
        "PASS"
        if cell107_gate
        else
        "FAIL"
    ),
)


assert cell107_gate, (
    "Phase-1 Author Feedback Batch failed. "
    "Repair this SAME Code Cell 107 before promotion to the 28-case Author Review queue."
)

=== PHASE-1 AUTHOR FEEDBACK -> PILOTS 23-28 ===

Batch version: phase1_author_feedback_derived_pilots_v1
Origin: PHASE1_DEMO_AUTHOR_FEEDBACK

New Pilot count: 6
Content Pilots: 5
Continuity Regression Pilots: 1

=== DERIVED PILOTS ===

Reference Pilot: 23
Pilot ID: question-pilot:a0f61a7f0c946fe1d79095dbb96a9497
Type: AUTHOR_FEEDBACK_CONTENT
Question:
سواد مالی درباره میزان پولی است که داریم یا درباره مدیریت پول؟
Expected Answer:
سواد مالی درباره مدیریت پول است، نه درباره میزان پول.
Promotion: READY_FOR_AUTHOR_REVIEW

Reference Pilot: 24
Pilot ID: question-pilot:70628caeefebbfba48300176fafddc96
Type: AUTHOR_FEEDBACK_CONTENT
Question:
برای اینکه بفهمیم از نظر دانش، نگرش و مهارت‌های مالی وضعیت قابل قبولی داریم، چه باید کرد؟
Expected Answer:
مدیریت مالی شخصی به دانش، نگرش و مهارت‌های خاص نیاز دارد. برای اینکه بدانیم وضعیت مالی ما از این جنبه‌ها به اندازه کافی مناسب است، باید پرسشنامه مربوط را تکمیل کنیم.
Promotion: READY_FOR_AUTHOR_REVIEW

Reference Pilot: 25
Pilot ID: question-pilot:77d8

<div style="direction: rtl; text-align: right; white-space: normal; line-height: 1.8;">

## بازبینی منبع پایلوت‌های 23 تا 28 — بازخوردهای فاز یک

شش پایلوت 23 تا 28 از بازخوردهای واقعی دکتر کمیل رودی در دموی فاز یک ساخته شده‌اند.

پیش از اینکه این شش مورد هم‌سطح ۲۲ پایلوت قبلی تلقی شوند، باید بررسی شود که سؤال و پاسخ پیشنهادی آن‌ها با Sourceهای واقعی پذیرفته‌شده پروژه سازگار است یا خیر.

### محدوده این مرحله

از آنجا که دموی فاز یک مورد بحث فقط از مقالات دکتر استفاده می‌کرد، Source Mapping در این مرحله ابتدا فقط روی:

**Website + Direct Author / Primary Author**

انجام می‌شود.

Source of Truth:

`data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl`

برای هر یک از پایلوت‌های 23 تا 28 باید:

1. سؤال فعلی خوانده شود.
2. پاسخ/رفتار مورد انتظار خوانده شود.
3. بازخورد تاریخی دکتر به‌عنوان Query Signal استفاده شود.
4. مقالات Direct Author دکتر در Accepted Corpus جستجو شوند.
5. چند Source Candidate واقعی همراه با متن Evidence نمایش داده شوند.
6. میزان پوشش واژگان کلیدی گزارش شود.
7. هنوز هیچ `ACCEPT / REVISE / RETARGET / REJECT` در Registry ثبت نشود.

### مورد خاص Pilot 25

Pilot 25 یک سؤال محتوایی معمولی نیست و نوع آن:

`MULTI_TURN_CONTEXT_CONTINUITY`

است.

برای این مورد، Source Search فقط بررسی می‌کند که موضوع اصلی گفت‌وگو یعنی:

`پول کجا می‌رود / ثبت و بررسی هزینه‌ها`

در Corpus چه پشتوانه‌ای دارد.

خود الزام «حفظ Context مکالمه» از Source مقاله استخراج نمی‌شود و باید به‌عنوان Regression Behavior مستقل باقی بماند.

### ممنوعیت‌ها

این Cell:

- Registry را تغییر نمی‌دهد.
- سؤال یا جواب جدید تولید نمی‌کند.
- Source را خودکار ACCEPT نمی‌کند.
- Author Review Decision ثبت نمی‌کند.
- Target Evidence را تغییر نمی‌دهد.
- ۲۲ پایلوت قبلی را تغییر نمی‌دهد.
- فایل Markdown بازبینی مؤلف را تغییر نمی‌دهد.
- Commit یا Push انجام نمی‌دهد.

خروجی این Cell مبنای تصمیم انسانی مرحله بعد خواهد بود.

</div>

In [6]:
# Code Cell 108
# ============================================================
# SOURCE MAPPING REVIEW — PILOTS 23-28
#
# READ-ONLY diagnostic.
#
# Purpose:
# - Load the frozen Phase-2 Accepted Corpus.
# - Resolve the six Phase-1 Author-Feedback-derived Pilots.
# - Search ONLY Website Direct-Author / Primary-Author sources.
# - Show real candidate documents + evidence windows.
# - Produce a review report in memory.
#
# NO Registry mutation.
# NO ACCEPT / REJECT decision.
# NO target/citation mutation.
# NO Author decision.
# NO Commit / Push.
# ============================================================

from pathlib import Path
from collections import Counter
import hashlib
import json
import math
import re
import unicodedata


# ============================================================
# 1) Constants
# ============================================================

CELL108_EXPECTED_CORPUS_SHA256 = (
    "9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40"
)

CELL108_EXPECTED_CORPUS_RECORDS = 19456
CELL108_EXPECTED_DIRECT_AUTHOR_WEBSITE = 76

CELL108_NEW_PILOT_NUMBERS = [
    23,
    24,
    25,
    26,
    27,
    28,
]

CELL108_TOP_K = 5


# Search terms are derived ONLY from:
# - actual Phase-1 questions,
# - actual Dr. Roudi feedback,
# - the already-created derived pilot.
#
# No external knowledge is introduced.
CELL108_SEARCH_QUERIES = {

    23: (
        "سواد مالی مدیریت پول میزان پول "
        "پولدار شدن مدیریت بهتر پول"
    ),

    24: (
        "مدیریت مالی شخصی دانش نگرش مهارت "
        "پرسشنامه وضعیت مالی ارزیابی"
    ),

    25: (
        "پولم کجا می رود ثبت هزینه ثبت مخارج "
        "مدیریت هزینه خرج هزینه ها"
    ),

    26: (
        "الگوی خرید نامناسب هدررفت پول "
        "تنش عاطفی هیجان زودگذر گروه مرجع "
        "تبلیغات بازاریابی فداکاری خرید زیاد"
    ),

    27: (
        "افزایش درآمد فروش وسایل کار بیشتر "
        "کارفرما پروژه دارایی درآمدزا "
        "سرمایه گذاری سود درآمد منفعل"
    ),

    28: (
        "احساس فقر رابطه با پول "
        "اعتماد به نفس پول نرسیدن خواسته "
        "احساس فقیر بودن"
    ),
}


# ============================================================
# 2) Utilities
# ============================================================

cell108_failures = []


def cell108_check(name, condition, details=None):

    if not bool(condition):

        failure = {
            "reason": name,
        }

        if details is not None:
            failure["details"] = details

        cell108_failures.append(
            failure
        )

    return bool(condition)


def cell108_sha256_file(path):

    h = hashlib.sha256()

    with path.open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def cell108_find_repo_root():

    candidates = []

    # Previous notebook variables are optional.
    for variable_name in [
        "cell107_registry_path",
        "cell108_registry_path",
        "cell105_registry_path",
    ]:

        if variable_name in globals():

            try:
                p = Path(
                    globals()[variable_name]
                ).resolve()

                candidates.extend(
                    [
                        p.parent,
                        *p.parents,
                    ]
                )

            except Exception:
                pass

    cwd = Path.cwd().resolve()

    candidates.extend(
        [
            cwd,
            *cwd.parents,
        ]
    )

    seen = set()

    for candidate in candidates:

        candidate = candidate.resolve()

        if str(candidate) in seen:
            continue

        seen.add(
            str(candidate)
        )

        registry_path = (
            candidate
            /
            "data"
            /
            "phase2"
            /
            "human_review"
            /
            "human_review_registry.json"
        )

        corpus_path = (
            candidate
            /
            "data"
            /
            "phase2"
            /
            "processed"
            /
            "documents_dedup_quality_accepted_v1.jsonl"
        )

        if (
            registry_path.exists()
            and
            corpus_path.exists()
        ):
            return candidate

    return None


# ============================================================
# 3) Persian normalization
# ============================================================

CELL108_STOPWORDS = {
    "از", "به", "در", "با", "برای", "را",
    "که", "و", "یا", "این", "آن", "یک",
    "است", "هست", "هستم", "هستند", "بود",
    "شد", "شود", "می", "شود", "کرد", "کنیم",
    "چه", "چرا", "چگونه", "اگر", "اما",
    "هم", "روی", "خود", "ما", "من", "شما",
    "فرد", "افراد", "مورد", "درباره",
}


def cell108_normalize(text):

    text = "" if text is None else str(text)

    text = unicodedata.normalize(
        "NFKC",
        text,
    )

    replacements = {
        "ي": "ی",
        "ك": "ک",
        "ۀ": "ه",
        "ة": "ه",
        "ؤ": "و",
        "إ": "ا",
        "أ": "ا",
        "ٱ": "ا",
        "\u200c": " ",
        "\u200f": " ",
        "\u200e": " ",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(
        r"[^\w\u0600-\u06FF]+",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text


def cell108_tokens(text):

    tokens = [
        token

        for token
        in cell108_normalize(
            text
        ).split()

        if (
            len(token) >= 2
            and
            token not in CELL108_STOPWORDS
        )
    ]

    return tokens


# ============================================================
# 4) Flexible metadata helpers
# ============================================================

def cell108_json_blob(value):

    try:
        return json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
        ).lower()

    except Exception:
        return str(
            value
        ).lower()


def cell108_is_direct_author_website(record):

    if (
        str(
            record.get(
                "source_type",
                ""
            )
        ).lower()
        !=
        "website"
    ):
        return False

    # Inspect only authority-related metadata,
    # NOT the article text itself.
    authority_blob = cell108_json_blob({
        "attribution":
            record.get(
                "attribution"
            ),

        "content_origin":
            record.get(
                "content_origin"
            ),

        "source_metadata":
            record.get(
                "source_metadata"
            ),
    })

    positive_markers = [
        '"primary_author"',
        '"direct_author"',
        "primary_author",
        "direct_author",
    ]

    return any(
        marker in authority_blob

        for marker
        in positive_markers
    )


def cell108_get_url(record):

    candidates = []

    locator = (
        record.get(
            "locator"
        )
        or
        {}
    )

    source_metadata = (
        record.get(
            "source_metadata"
        )
        or
        {}
    )

    if isinstance(locator, dict):

        for key in [
            "canonical_url",
            "url",
            "source_url",
        ]:

            if locator.get(key):
                candidates.append(
                    locator.get(key)
                )

    if isinstance(source_metadata, dict):

        for key in [
            "canonical_url",
            "url",
            "source_url",
        ]:

            if source_metadata.get(key):
                candidates.append(
                    source_metadata.get(key)
                )

    return (
        candidates[0]
        if candidates
        else None
    )


def cell108_get_title(record):

    title = record.get(
        "title"
    )

    if title:
        return str(title)

    metadata = (
        record.get(
            "source_metadata"
        )
        or
        {}
    )

    if isinstance(metadata, dict):

        for key in [
            "title",
            "article_title",
            "page_title",
        ]:

            if metadata.get(key):
                return str(
                    metadata.get(key)
                )

    return "NO_TITLE"


# ============================================================
# 5) Resolve repository and frozen artifacts
# ============================================================

cell108_repo_root = (
    cell108_find_repo_root()
)


cell108_check(
    "REPO_ROOT_NOT_FOUND",
    cell108_repo_root is not None,
)


if cell108_failures:
    raise AssertionError(
        cell108_failures
    )


cell108_registry_path = (
    cell108_repo_root
    /
    "data"
    /
    "phase2"
    /
    "human_review"
    /
    "human_review_registry.json"
)


cell108_corpus_path = (
    cell108_repo_root
    /
    "data"
    /
    "phase2"
    /
    "processed"
    /
    "documents_dedup_quality_accepted_v1.jsonl"
)


cell108_check(
    "REGISTRY_NOT_FOUND",
    cell108_registry_path.exists(),
    str(cell108_registry_path),
)


cell108_check(
    "ACCEPTED_CORPUS_NOT_FOUND",
    cell108_corpus_path.exists(),
    str(cell108_corpus_path),
)


if cell108_failures:
    raise AssertionError(
        cell108_failures
    )


# ============================================================
# 6) Verify Accepted Corpus identity
# ============================================================

cell108_corpus_sha256 = (
    cell108_sha256_file(
        cell108_corpus_path
    )
)


cell108_check(
    "ACCEPTED_CORPUS_SHA256_MISMATCH",
    cell108_corpus_sha256
    ==
    CELL108_EXPECTED_CORPUS_SHA256,
    {
        "expected":
            CELL108_EXPECTED_CORPUS_SHA256,

        "actual":
            cell108_corpus_sha256,
    },
)


if cell108_failures:

    print(
        cell108_failures
    )

    raise AssertionError(
        "Cell 108 stopped Fail-Closed: frozen corpus identity mismatch."
    )


# ============================================================
# 7) Load Accepted Corpus
# ============================================================

cell108_corpus = []


with cell108_corpus_path.open(
    "r",
    encoding="utf-8",
) as f:

    for line_number, line in enumerate(
        f,
        start=1,
    ):

        line = line.strip()

        if not line:
            continue

        try:

            record = json.loads(
                line
            )

        except Exception as exc:

            cell108_check(
                "CORPUS_JSON_PARSE_FAILURE",
                False,
                {
                    "line":
                        line_number,

                    "error":
                        str(exc),
                },
            )

            break

        cell108_corpus.append(
            record
        )


cell108_check(
    "CORPUS_RECORD_COUNT_MISMATCH",
    len(cell108_corpus)
    ==
    CELL108_EXPECTED_CORPUS_RECORDS,
    len(cell108_corpus),
)


if cell108_failures:
    raise AssertionError(
        cell108_failures
    )


# ============================================================
# 8) Select Direct-Author Website corpus
# ============================================================

cell108_primary_website_records = [
    record

    for record
    in cell108_corpus

    if cell108_is_direct_author_website(
        record
    )
]


cell108_check(
    "DIRECT_AUTHOR_WEBSITE_COUNT_MISMATCH",
    len(
        cell108_primary_website_records
    )
    ==
    CELL108_EXPECTED_DIRECT_AUTHOR_WEBSITE,
    {
        "expected":
            CELL108_EXPECTED_DIRECT_AUTHOR_WEBSITE,

        "actual":
            len(
                cell108_primary_website_records
            ),
    },
)


if cell108_failures:

    print(
        cell108_failures
    )

    raise AssertionError(
        "Cell 108 stopped Fail-Closed: Direct-Author Website scope mismatch."
    )


# ============================================================
# 9) Load six derived Pilots from Registry
# ============================================================

cell108_registry = json.loads(
    cell108_registry_path.read_text(
        encoding="utf-8"
    )
)


cell108_second_review = (
    cell108_registry.get(
        "second_human_review",
        {}
    )
    or
    {}
)


cell108_feedback_batch = (
    cell108_second_review.get(
        "phase1_author_feedback_derived_pilots",
        {}
    )
    or
    {}
)


cell108_feedback_cases = (
    cell108_feedback_batch.get(
        "cases",
        []
    )
    or
    []
)


cell108_cases_by_number = {
    case.get(
        "reference_pilot_number"
    ):
        case

    for case
    in cell108_feedback_cases

    if isinstance(
        case,
        dict,
    )
}


cell108_check(
    "DERIVED_PILOT_BATCH_NOT_6",
    len(
        cell108_feedback_cases
    )
    ==
    6,
    len(
        cell108_feedback_cases
    ),
)


cell108_check(
    "DERIVED_PILOT_NUMBERS_MISSING",
    sorted(
        cell108_cases_by_number.keys()
    )
    ==
    CELL108_NEW_PILOT_NUMBERS,
    sorted(
        cell108_cases_by_number.keys()
    ),
)


if cell108_failures:
    raise AssertionError(
        cell108_failures
    )


# ============================================================
# 10) Build BM25 index over the 76 real Direct-Author articles
# ============================================================

cell108_doc_tokens = []
cell108_doc_term_counts = []
cell108_document_frequency = Counter()


for record in (
    cell108_primary_website_records
):

    text = (
        record.get(
            "text"
        )
        or
        ""
    )

    tokens = cell108_tokens(
        text
    )

    counts = Counter(
        tokens
    )

    cell108_doc_tokens.append(
        tokens
    )

    cell108_doc_term_counts.append(
        counts
    )

    for token in set(tokens):

        cell108_document_frequency[
            token
        ] += 1


cell108_doc_lengths = [
    len(tokens)

    for tokens
    in cell108_doc_tokens
]


cell108_average_doc_length = (
    sum(
        cell108_doc_lengths
    )
    /
    max(
        len(
            cell108_doc_lengths
        ),
        1,
    )
)


def cell108_bm25_score(
    query_tokens,
    doc_index,
    k1=1.5,
    b=0.75,
):

    counts = (
        cell108_doc_term_counts[
            doc_index
        ]
    )

    doc_length = (
        cell108_doc_lengths[
            doc_index
        ]
    )

    N = len(
        cell108_primary_website_records
    )

    score = 0.0

    for term in set(
        query_tokens
    ):

        tf = counts.get(
            term,
            0,
        )

        if tf <= 0:
            continue

        df = (
            cell108_document_frequency.get(
                term,
                0,
            )
        )

        idf = math.log(
            1
            +
            (
                N
                -
                df
                +
                0.5
            )
            /
            (
                df
                +
                0.5
            )
        )

        denominator = (
            tf
            +
            k1
            *
            (
                1
                -
                b
                +
                b
                *
                doc_length
                /
                max(
                    cell108_average_doc_length,
                    1,
                )
            )
        )

        score += (
            idf
            *
            (
                tf
                *
                (
                    k1
                    +
                    1
                )
            )
            /
            denominator
        )

    return score


# ============================================================
# 11) Evidence-window selection
# ============================================================

def cell108_best_window(
    text,
    query_tokens,
    max_chars=700,
):

    text = str(
        text
        or
        ""
    ).strip()

    if not text:
        return ""

    # Preserve actual source text.
    blocks = [
        block.strip()

        for block
        in re.split(
            r"\n\s*\n",
            text,
        )

        if block.strip()
    ]

    if not blocks:
        blocks = [text]

    query_set = set(
        query_tokens
    )

    best_block = None
    best_score = -1

    for block in blocks:

        block_tokens = set(
            cell108_tokens(
                block
            )
        )

        overlap = (
            query_set
            &
            block_tokens
        )

        score = len(
            overlap
        )

        if score > best_score:

            best_score = score
            best_block = block

    if best_block is None:
        best_block = text

    # If the best block is short enough, show it whole.
    if len(
        best_block
    ) <= max_chars:

        return best_block

    normalized_block = (
        cell108_normalize(
            best_block
        )
    )

    first_position = None

    for token in query_tokens:

        position = normalized_block.find(
            token
        )

        if (
            position >= 0
            and
            (
                first_position is None
                or
                position < first_position
            )
        ):
            first_position = position

    # Approximate a useful source window.
    if first_position is None:

        return (
            best_block[:max_chars]
            +
            " ..."
        )

    raw_lower = best_block

    # Find a raw occurrence of one matching term.
    raw_position = None

    for token in query_tokens:

        p = raw_lower.find(
            token
        )

        if p >= 0:

            raw_position = p

            break

    if raw_position is None:
        raw_position = 0

    start = max(
        0,
        raw_position
        -
        200,
    )

    end = min(
        len(
            best_block
        ),
        start
        +
        max_chars,
    )

    excerpt = (
        best_block[
            start:end
        ].strip()
    )

    if start > 0:
        excerpt = "... " + excerpt

    if end < len(
        best_block
    ):
        excerpt = excerpt + " ..."

    return excerpt


# ============================================================
# 12) Search all six Pilots
# ============================================================

cell108_source_mapping_report = []


for pilot_number in (
    CELL108_NEW_PILOT_NUMBERS
):

    case = (
        cell108_cases_by_number[
            pilot_number
        ]
    )

    query_text = (
        CELL108_SEARCH_QUERIES[
            pilot_number
        ]
    )

    query_tokens = (
        cell108_tokens(
            query_text
        )
    )

    query_unique = set(
        query_tokens
    )

    ranked = []


    for doc_index, record in enumerate(
        cell108_primary_website_records
    ):

        score = (
            cell108_bm25_score(
                query_tokens,
                doc_index,
            )
        )

        doc_token_set = set(
            cell108_doc_tokens[
                doc_index
            ]
        )

        matched_terms = sorted(
            query_unique
            &
            doc_token_set
        )

        coverage = (
            len(
                matched_terms
            )
            /
            max(
                len(
                    query_unique
                ),
                1,
            )
        )

        # Small transparent bonus for broader query coverage.
        combined_score = (
            score
            *
            (
                1.0
                +
                coverage
            )
        )

        ranked.append({
            "record":
                record,

            "score":
                combined_score,

            "bm25":
                score,

            "coverage":
                coverage,

            "matched_terms":
                matched_terms,

            "doc_index":
                doc_index,
        })


    ranked.sort(
        key=lambda item: (
            -item[
                "score"
            ],
            str(
                item[
                    "record"
                ].get(
                    "document_id",
                    "",
                )
            ),
        )
    )


    top_candidates = []


    for rank, result in enumerate(
        ranked[
            :CELL108_TOP_K
        ],
        start=1,
    ):

        record = result[
            "record"
        ]

        candidate = {
            "rank":
                rank,

            "document_id":
                record.get(
                    "document_id"
                ),

            "source_id":
                record.get(
                    "source_id"
                ),

            "title":
                cell108_get_title(
                    record
                ),

            "url":
                cell108_get_url(
                    record
                ),

            "score":
                round(
                    result[
                        "score"
                    ],
                    4,
                ),

            "bm25":
                round(
                    result[
                        "bm25"
                    ],
                    4,
                ),

            "query_coverage":
                round(
                    result[
                        "coverage"
                    ],
                    4,
                ),

            "matched_terms":
                result[
                    "matched_terms"
                ],

            "evidence_window":
                cell108_best_window(
                    record.get(
                        "text",
                        "",
                    ),
                    query_tokens,
                ),
        }

        top_candidates.append(
            candidate
        )


    top = (
        top_candidates[0]
        if top_candidates
        else None
    )


    # This is ONLY a retrieval signal.
    # It is explicitly NOT a Human Review decision.
    if top is None:

        retrieval_signal = (
            "NO_CANDIDATE"
        )

    elif (
        top[
            "query_coverage"
        ]
        >=
        0.50
    ):

        retrieval_signal = (
            "STRONG_SOURCE_CANDIDATE"
        )

    elif (
        top[
            "query_coverage"
        ]
        >=
        0.30
    ):

        retrieval_signal = (
            "POSSIBLE_SOURCE_CANDIDATE"
        )

    else:

        retrieval_signal = (
            "WEAK_SOURCE_MATCH_REVIEW_REQUIRED"
        )


    report_item = {
        "reference_pilot_number":
            pilot_number,

        "pilot_case_id":
            case.get(
                "pilot_case_id"
            ),

        "pilot_type":
            case.get(
                "pilot_type"
            ),

        "derived_question":
            (
                (
                    case.get(
                        "derived_question",
                        {}
                    )
                    or
                    {}
                ).get(
                    "question_text"
                )
            ),

        "expected_answer":
            (
                (
                    case.get(
                        "expected_answer_review",
                        {}
                    )
                    or
                    {}
                ).get(
                    "expected_answer_text"
                )
            ),

        "expected_behavior":
            (
                (
                    case.get(
                        "expected_behavior_review",
                        {}
                    )
                    or
                    {}
                ).get(
                    "expected_behavior"
                )
            ),

        "search_query":
            query_text,

        "mapping_purpose":
            (
                "CONTENT_GROUNDING"
                if (
                    case.get(
                        "pilot_type"
                    )
                    ==
                    "AUTHOR_FEEDBACK_CONTENT"
                )
                else
                "CONVERSATION_TOPIC_GROUNDING_ONLY"
            ),

        "retrieval_signal":
            retrieval_signal,

        "top_candidates":
            top_candidates,

        "human_source_decision":
            "NOT_STARTED",
    }


    cell108_source_mapping_report.append(
        report_item
    )


# ============================================================
# 13) Coverage / safety validation
# ============================================================

cell108_check(
    "SOURCE_MAPPING_REPORT_COUNT_NOT_6",
    len(
        cell108_source_mapping_report
    )
    ==
    6,
)


for item in (
    cell108_source_mapping_report
):

    cell108_check(
        (
            "NO_TOP_CANDIDATES_FOR_PILOT_"
            +
            str(
                item[
                    "reference_pilot_number"
                ]
            )
        ),
        len(
            item[
                "top_candidates"
            ]
        )
        ==
        CELL108_TOP_K,
    )


if cell108_failures:
    raise AssertionError(
        cell108_failures
    )


# ============================================================
# 14) Human-readable output
# ============================================================

print(
    "=== CELL 108 — SOURCE MAPPING REVIEW: PILOTS 23-28 ==="
)


print(
    "\n=== FROZEN SOURCE ==="
)


print(
    "Corpus:",
    str(
        cell108_corpus_path
    ),
)


print(
    "Corpus SHA256:",
    cell108_corpus_sha256,
)


print(
    "Accepted Corpus records:",
    len(
        cell108_corpus
    ),
)


print(
    "Direct-Author Website records:",
    len(
        cell108_primary_website_records
    ),
)


print(
    "\nSearch scope:",
    "WEBSITE / DIRECT_AUTHOR ONLY",
)


for item in (
    cell108_source_mapping_report
):

    print(
        "\n"
        +
        "="
        *
        110
    )

    print(
        "PILOT:",
        item[
            "reference_pilot_number"
        ],
    )

    print(
        "Pilot ID:",
        item[
            "pilot_case_id"
        ],
    )

    print(
        "Type:",
        item[
            "pilot_type"
        ],
    )

    print(
        "Mapping purpose:",
        item[
            "mapping_purpose"
        ],
    )

    print(
        "\nQuestion:"
    )

    print(
        item[
            "derived_question"
        ]
    )


    if item[
        "expected_answer"
    ]:

        print(
            "\nCurrent Expected Answer:"
        )

        print(
            item[
                "expected_answer"
            ]
        )


    if item[
        "expected_behavior"
    ]:

        print(
            "\nCurrent Expected Behavior:"
        )

        print(
            item[
                "expected_behavior"
            ]
        )


    print(
        "\nSearch query:"
    )

    print(
        item[
            "search_query"
        ]
    )


    print(
        "\nRETRIEVAL SIGNAL:",
        item[
            "retrieval_signal"
        ],
    )


    print(
        "\n--- TOP REAL SOURCE CANDIDATES ---"
    )


    for candidate in (
        item[
            "top_candidates"
        ]
    ):

        print(
            "\n"
            +
            "-"
            *
            90
        )

        print(
            "Rank:",
            candidate[
                "rank"
            ],
        )

        print(
            "Document ID:",
            candidate[
                "document_id"
            ],
        )

        print(
            "Title:",
            candidate[
                "title"
            ],
        )

        print(
            "URL:",
            candidate[
                "url"
            ],
        )

        print(
            "Score:",
            candidate[
                "score"
            ],
        )

        print(
            "Query coverage:",
            candidate[
                "query_coverage"
            ],
        )

        print(
            "Matched terms:",
            candidate[
                "matched_terms"
            ],
        )

        print(
            "\nEvidence window:"
        )

        print(
            candidate[
                "evidence_window"
            ]
        )


    print(
        "\nHUMAN SOURCE DECISION:",
        "NOT_STARTED"
    )


# ============================================================
# 15) Safety / gate
# ============================================================

cell108_gate = (
    len(
        cell108_failures
    )
    ==
    0
)


print(
    "\n"
    +
    "="
    *
    110
)


print(
    "=== SAFETY ==="
)


print(
    "Registry write:",
    False,
)


print(
    "Author Review mutation:",
    False,
)


print(
    "Human ACCEPT/REJECT recorded:",
    False,
)


print(
    "Question changed:",
    False,
)


print(
    "Expected Answer changed:",
    False,
)


print(
    "Target Evidence changed:",
    False,
)


print(
    "Citation changed:",
    False,
)


print(
    "Approved 22 cases changed:",
    False,
)


print(
    "Markdown document changed:",
    False,
)


print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)


print(
    cell108_failures
)


print(
    "\nCODE CELL 108 GATE:",
    (
        "PASS"
        if cell108_gate
        else
        "FAIL"
    ),
)


assert cell108_gate, (
    "Pilot 23-28 Source Mapping diagnostic failed. "
    "Repair this SAME Code Cell 108 before Human Source Review."
)

=== CELL 108 — SOURCE MAPPING REVIEW: PILOTS 23-28 ===

=== FROZEN SOURCE ===
Corpus: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl
Corpus SHA256: 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40
Accepted Corpus records: 19456
Direct-Author Website records: 76

Search scope: WEBSITE / DIRECT_AUTHOR ONLY

PILOT: 23
Pilot ID: question-pilot:a0f61a7f0c946fe1d79095dbb96a9497
Type: AUTHOR_FEEDBACK_CONTENT
Mapping purpose: CONTENT_GROUNDING

Question:
سواد مالی درباره میزان پولی است که داریم یا درباره مدیریت پول؟

Current Expected Answer:
سواد مالی درباره مدیریت پول است، نه درباره میزان پول.

Search query:
سواد مالی مدیریت پول میزان پول پولدار شدن مدیریت بهتر پول

RETRIEVAL SIGNAL: STRONG_SOURCE_CANDIDATE

--- TOP REAL SOURCE CANDIDATES ---

------------------------------------------------------------------------------------------
Rank: 1
Document ID: website:3A675BF6
Title: دختران خوب

<div style="direction: rtl; text-align: right; white-space: normal; line-height: 1.8;">

## تصمیم نهایی بازبینی منبع برای پایلوت‌های 23 تا 28

در Cell قبل، هر شش پایلوت مشتق‌شده از بازخورد فاز یک روی Accepted Corpus واقعی Phase 2 و فقط در محدوده Website / Direct Author بررسی شدند.

در این مرحله تصمیم انسانی Source Review ثبت می‌شود.

### تصمیم‌های تأییدشده

#### Pilot 23 — ACCEPT

سؤال و پاسخ فعلی حفظ می‌شوند.

Source اصلی Corpus:

`website:7C71884B — ابزارهای سواد مالی برای بهبود مدیریت هزینه`

این Source به‌صورت مستقیم بیان می‌کند که سواد مالی درباره میزان پول نیست، بلکه درباره درست مدیریت کردن پول است.

---

#### Pilot 24 — ACCEPT

سؤال و پاسخ فعلی حفظ می‌شوند.

Grounding ترکیبی:

- `website:662BDA7F — سواد مالی خانم‌ها؛ وضعیت چقدر بحرانی است؟`
- `website:E2311705 — آموزش سواد مالی در اروپا`

یکی بخش پرسشنامه و ارزیابی وضعیت را پشتیبانی می‌کند و دیگری ابعاد دانش، نگرش و مهارت را.

---

#### Pilot 25 — REVISE

این مورد دیگر به‌عنوان `MULTI_TURN_CONTEXT_CONTINUITY` فعال باقی نمی‌ماند.

علت:

در دموی فاز یک، Turnها Context مشترک قابل اتکایی نداشتند؛ بنابراین پاسخ بی‌ربط مدل نباید به‌عنوان یک Golden Regression Case محتوایی وارد مجموعه مرجع شود.

اما موضوع اصلی سؤال با Corpus واقعی پشتیبانی می‌شود.

Source اصلی:

`website:3E26E509 — سواد مالی خانم‌ها؛ پول‌ها کجا می‌روند؟`

سؤال جدید:

**وقتی نمی‌دانیم پولمان کجا می‌رود و در پایان ماه نمی‌دانیم آن را صرف چه چیزهایی کرده‌ایم، چه کاری باید انجام دهیم؟**

پاسخ جدید:

**باید هزینه‌ها را ثبت کنیم تا بدانیم پول را کجا و برای چه چیزهایی خرج کرده‌ایم. آگاهی از هزینه‌ها امکان مدیریت آن‌ها را فراهم می‌کند و ثبت هزینه‌ها می‌تواند به پیشگیری از ولخرجی و مدیریت بهتر مخارج کمک کند.**

نوع فعال جدید:

`AUTHOR_FEEDBACK_CONTENT`

تاریخچه حالت قبلی حذف نمی‌شود و به‌عنوان Superseded History حفظ خواهد شد.

---

#### Pilot 26 — REVISE / GROUNDING ONLY

سؤال و پاسخ فعلی تغییر نمی‌کنند.

بازخورد مستقیم دکتر همچنان Grounding اصلی هفت الگوی خرید است.

Corpus پشتیبانی تکمیلی ارائه می‌دهد:

- `website:9673DF51 — الگوهای نادرست خرید و راهکارهای پرهیز از آن`
- `website:BC0342B6 — سواد مالی خانم‌ها؛ تصمیم‌های خرید را خودمان می‌گیریم یا عادت‌های‌مان؟`

مقالات مفهوم الگوهای نادرست خرید، عوامل عاطفی و اثر تبلیغات/بازاریابی را پشتیبانی می‌کنند، اما تمام هفت مورد را مستقل و عیناً پوشش نمی‌دهند.

بنابراین:

`DIRECT AUTHOR FEEDBACK = PRIMARY GROUNDING`

`WEBSITE CORPUS = PARTIAL CORROBORATION`

---

#### Pilot 27 — REVISE / GROUNDING ONLY

سؤال و پاسخ فعلی تغییر نمی‌کنند.

چهار روش و ترتیب آن‌ها از بازخورد مستقیم دکتر آمده است.

Corpus فقط بخشی از مفاهیم مرتبط با افزایش درآمد، دارایی درآمدزا، سرمایه‌گذاری و سود را پشتیبانی می‌کند.

Sources تکمیلی:

- `website:3A675BF6 — دختران خوب ثروتمند می‌شوند اگر ...`
- `website:C042183F — «پول، پول می‌آورد» اما چگونه؟`

بنابراین:

`DIRECT AUTHOR FEEDBACK = PRIMARY GROUNDING`

`WEBSITE CORPUS = PARTIAL CORROBORATION`

---

#### Pilot 28 — ACCEPT

سؤال و پاسخ فعلی حفظ می‌شوند.

Source مهم Corpus:

`website:2F8F15FB — چگونه کمتر نگران پول باشیم؟`

این مقاله بخش اصلی مفهوم رابطه با پول، احساسات و تلاش برای جبران چیزهایی که فرد احساس می‌کند ندارد را پشتیبانی می‌کند.

بخش گسترده‌تر پاسخ درباره احساس فقر و نرسیدن به خواسته‌ها از بازخورد مستقیم دکتر حفظ می‌شود.

بنابراین Grounding ترکیبی است:

`DIRECT AUTHOR FEEDBACK + WEBSITE DIRECT AUTHOR`

---

### نتیجه مورد انتظار

پس از اجرای Cell بعد:

- 23 = ACCEPT
- 24 = ACCEPT
- 25 = REVISE → AUTHOR_FEEDBACK_CONTENT
- 26 = REVISE / GROUNDING_ONLY
- 27 = REVISE / GROUNDING_ONLY
- 28 = ACCEPT

هر دو Copy موجود در Registry باید Sync شوند:

1. `second_human_review.phase1_author_feedback_derived_pilots`
2. `author_review.cases_ready_for_review`

همچنان:

- Author Final Review = NOT_STARTED
- ۲۲ مورد قبلی تغییر نمی‌کنند.
- Raw Phase-1 Author Feedback تغییر نمی‌کند.
- History قبلی Pilot 25 حذف نمی‌شود.
- هیچ Commit یا Push انجام نمی‌شود.
- فایل Markdown ۲۸تایی در این Cell تغییر نمی‌کند.

</div>

In [7]:
# Code Cell 109
# ============================================================
# APPLY HUMAN SOURCE REVIEW — PILOTS 23-28
#
# Decisions approved after Cell 108 Source Mapping:
#
# 23 -> ACCEPT
# 24 -> ACCEPT
# 25 -> REVISE:
#       MULTI_TURN_CONTEXT_CONTINUITY
#       -> AUTHOR_FEEDBACK_CONTENT
# 26 -> REVISE / GROUNDING_ONLY
# 27 -> REVISE / GROUNDING_ONLY
# 28 -> ACCEPT
#
# This cell:
# - verifies the frozen Accepted Corpus;
# - verifies all chosen source documents really exist;
# - applies the Source Review to BOTH Registry copies;
# - preserves raw Phase-1 conversation and Author Feedback;
# - preserves Pilot 25's superseded continuity history;
# - creates a new accepted Q/A for Pilot 25;
# - updates citation/source mappings;
# - keeps Author Final Review NOT_STARTED;
# - protects the original 22 cases;
# - creates a Registry backup;
# - reloads and validates persisted state;
# - is idempotent.
#
# It DOES NOT:
# - change the 28-case Markdown file;
# - record any decision on behalf of Dr. Roudi;
# - commit or push.
# ============================================================

from pathlib import Path
from copy import deepcopy
from datetime import datetime, timezone
import hashlib
import json


# ============================================================
# 1) Constants
# ============================================================

CELL109_EXPECTED_CORPUS_SHA256 = (
    "9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40"
)

CELL109_EXPECTED_COMPLETED_CASES = 22
CELL109_EXPECTED_AUTHOR_QUEUE = 28
CELL109_EXPECTED_NEW_CASES = 6

CELL109_BATCH_KEY = (
    "phase1_author_feedback_derived_pilots"
)

CELL109_REVIEW_VERSION = (
    "phase1_author_feedback_source_review_v1"
)

CELL109_REFERENCE_NUMBERS = [
    23,
    24,
    25,
    26,
    27,
    28,
]


# ============================================================
# 2) Approved Human Review decisions
# ============================================================

CELL109_DECISIONS = {

    # --------------------------------------------------------
    # Pilot 23
    # --------------------------------------------------------

    23: {
        "decision":
            "ACCEPT",

        "revision_scope":
            "NONE",

        "grounding_mode":
            "DIRECT_AUTHOR_FEEDBACK_PLUS_CORPUS_DIRECT_SUPPORT",

        "sources": [
            {
                "document_id":
                    "website:7C71884B",

                "role":
                    "PRIMARY_CORPUS_GROUNDING",

                "coverage":
                    "DIRECT_EXPLICIT",

                "review_note":
                    (
                        "The article explicitly states that financial "
                        "literacy is not about the amount of money, "
                        "but about managing money correctly."
                    ),
            },
        ],
    },


    # --------------------------------------------------------
    # Pilot 24
    # --------------------------------------------------------

    24: {
        "decision":
            "ACCEPT",

        "revision_scope":
            "NONE",

        "grounding_mode":
            "DIRECT_AUTHOR_FEEDBACK_PLUS_COMBINED_CORPUS_SUPPORT",

        "sources": [
            {
                "document_id":
                    "website:662BDA7F",

                "role":
                    "ASSESSMENT_AND_QUESTIONNAIRE_GROUNDING",

                "coverage":
                    "DIRECT_SUPPORT",

                "review_note":
                    (
                        "Supports questionnaire-based assessment "
                        "and interpretation of financial condition."
                    ),
            },

            {
                "document_id":
                    "website:E2311705",

                "role":
                    "KNOWLEDGE_ATTITUDE_SKILL_GROUNDING",

                "coverage":
                    "DIRECT_SUPPORT",

                "review_note":
                    (
                        "Supports financial literacy as a combination "
                        "of knowledge, attitudes and abilities/skills."
                    ),
            },
        ],
    },


    # --------------------------------------------------------
    # Pilot 25
    # --------------------------------------------------------

    25: {
        "decision":
            "REVISE",

        "revision_scope":
            "QUESTION_ANSWER_AND_PILOT_TYPE",

        "grounding_mode":
            "CORPUS_DIRECT_SUPPORT",

        "new_pilot_type":
            "AUTHOR_FEEDBACK_CONTENT",

        "new_question":
            (
                "وقتی نمی‌دانیم پولمان کجا می‌رود و در پایان ماه "
                "نمی‌دانیم آن را صرف چه چیزهایی کرده‌ایم، "
                "چه کاری باید انجام دهیم؟"
            ),

        "new_expected_answer":
            (
                "باید هزینه‌ها را ثبت کنیم تا بدانیم پول را کجا و "
                "برای چه چیزهایی خرج کرده‌ایم. آگاهی از هزینه‌ها "
                "امکان مدیریت آن‌ها را فراهم می‌کند و ثبت هزینه‌ها "
                "می‌تواند به پیشگیری از ولخرجی و مدیریت بهتر مخارج "
                "کمک کند."
            ),

        "sources": [
            {
                "document_id":
                    "website:3E26E509",

                "role":
                    "PRIMARY_CORPUS_GROUNDING",

                "coverage":
                    "DIRECT_EXPLICIT",

                "review_note":
                    (
                        "Directly supports the problem of not knowing "
                        "where money went and the value of recording "
                        "expenses for awareness and better management."
                    ),
            },

            {
                "document_id":
                    "website:7C71884B",

                "role":
                    "CORROBORATING_CORPUS_SOURCE",

                "coverage":
                    "SUPPORTING",

                "review_note":
                    (
                        "Supports expense recording and using that "
                        "experience for better expense management "
                        "and budgeting."
                    ),
            },
        ],
    },


    # --------------------------------------------------------
    # Pilot 26
    # --------------------------------------------------------

    26: {
        "decision":
            "REVISE",

        "revision_scope":
            "GROUNDING_ONLY",

        "grounding_mode":
            "DIRECT_AUTHOR_FEEDBACK_PRIMARY_WITH_PARTIAL_CORPUS_CORROBORATION",

        "sources": [
            {
                "document_id":
                    "website:9673DF51",

                "role":
                    "CORROBORATING_CORPUS_SOURCE",

                "coverage":
                    "PARTIAL",

                "review_note":
                    (
                        "Supports incorrect purchasing patterns, "
                        "emotional drivers, and advertising/marketing "
                        "influence, but does not independently enumerate "
                        "all seven Author-feedback items."
                    ),
            },

            {
                "document_id":
                    "website:BC0342B6",

                "role":
                    "CORROBORATING_CORPUS_SOURCE",

                "coverage":
                    "PARTIAL",

                "review_note":
                    (
                        "Supports emotional/psychological purchasing "
                        "patterns and broader incorrect buying behavior."
                    ),
            },
        ],

        "grounding_constraint":
            (
                "The seven-item expected answer remains primarily "
                "grounded in Dr. Roudi's direct historical feedback. "
                "Website evidence is corroborating, not an independent "
                "full source for all seven items."
            ),
    },


    # --------------------------------------------------------
    # Pilot 27
    # --------------------------------------------------------

    27: {
        "decision":
            "REVISE",

        "revision_scope":
            "GROUNDING_ONLY",

        "grounding_mode":
            "DIRECT_AUTHOR_FEEDBACK_PRIMARY_WITH_PARTIAL_CORPUS_CORROBORATION",

        "sources": [
            {
                "document_id":
                    "website:3A675BF6",

                "role":
                    "CORROBORATING_CORPUS_SOURCE",

                "coverage":
                    "PARTIAL",

                "review_note":
                    (
                        "Supports increasing income, work/income "
                        "capacity and income-producing assets, but "
                        "does not independently reproduce the complete "
                        "four-step Author sequence."
                    ),
            },

            {
                "document_id":
                    "website:C042183F",

                "role":
                    "CORROBORATING_CORPUS_SOURCE",

                "coverage":
                    "PARTIAL",

                "review_note":
                    (
                        "Supports the concept of investing money in "
                        "productive assets and generating profit."
                    ),
            },
        ],

        "grounding_constraint":
            (
                "The four methods and their ordering remain primarily "
                "grounded in Dr. Roudi's direct historical feedback. "
                "Website sources provide only partial corroboration."
            ),
    },


    # --------------------------------------------------------
    # Pilot 28
    # --------------------------------------------------------

    28: {
        "decision":
            "ACCEPT",

        "revision_scope":
            "NONE",

        "grounding_mode":
            "DIRECT_AUTHOR_FEEDBACK_PLUS_CORPUS_DIRECT_SUPPORT",

        "sources": [
            {
                "document_id":
                    "website:2F8F15FB",

                "role":
                    "PRIMARY_CORPUS_CORROBORATION",

                "coverage":
                    "STRONG_PARTIAL",

                "review_note":
                    (
                        "Strongly supports the relationship-with-money "
                        "concept and the idea that money can represent "
                        "things a person feels are missing, while the "
                        "broader poverty interpretation remains grounded "
                        "in direct Author feedback."
                    ),
            },
        ],
    },
}


# ============================================================
# 3) Utilities
# ============================================================

cell109_failures = []


def cell109_check(
    name,
    condition,
    details=None,
):

    if not bool(condition):

        failure = {
            "reason": name,
        }

        if details is not None:
            failure["details"] = details

        cell109_failures.append(
            failure
        )

    return bool(condition)


def cell109_stable_id(
    prefix,
    payload,
):

    if not isinstance(
        payload,
        str,
    ):

        payload = json.dumps(
            payload,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        )

    digest = hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()[:32]

    return (
        prefix
        +
        ":"
        +
        digest
    )


def cell109_sha256_text(text):

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def cell109_sha256_file(path):

    h = hashlib.sha256()

    with path.open("rb") as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def cell109_now():

    return (
        datetime.now(
            timezone.utc
        )
        .replace(
            microsecond=0
        )
        .isoformat()
    )


def cell109_find_repo_root():

    candidates = []

    for variable_name in [
        "cell108_registry_path",
        "cell107_registry_path",
        "cell105_registry_path",
    ]:

        if variable_name in globals():

            try:

                p = Path(
                    globals()[
                        variable_name
                    ]
                ).resolve()

                candidates.extend(
                    [
                        p.parent,
                        *p.parents,
                    ]
                )

            except Exception:
                pass

    cwd = Path.cwd().resolve()

    candidates.extend(
        [
            cwd,
            *cwd.parents,
        ]
    )

    seen = set()

    for candidate in candidates:

        candidate = (
            candidate.resolve()
        )

        if str(candidate) in seen:
            continue

        seen.add(
            str(candidate)
        )

        registry_candidate = (
            candidate
            /
            "data"
            /
            "phase2"
            /
            "human_review"
            /
            "human_review_registry.json"
        )

        corpus_candidate = (
            candidate
            /
            "data"
            /
            "phase2"
            /
            "processed"
            /
            "documents_dedup_quality_accepted_v1.jsonl"
        )

        if (
            registry_candidate.exists()
            and
            corpus_candidate.exists()
        ):
            return candidate

    return None


def cell109_json_blob(value):

    try:
        return json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
        ).lower()

    except Exception:
        return str(
            value
        ).lower()


def cell109_is_direct_author_website(
    record,
):

    if (
        str(
            record.get(
                "source_type",
                ""
            )
        ).lower()
        !=
        "website"
    ):
        return False

    authority_blob = (
        cell109_json_blob({
            "attribution":
                record.get(
                    "attribution"
                ),

            "content_origin":
                record.get(
                    "content_origin"
                ),

            "source_metadata":
                record.get(
                    "source_metadata"
                ),
        })
    )

    return (
        "primary_author"
        in authority_blob
        or
        "direct_author"
        in authority_blob
    )


def cell109_get_title(record):

    if record.get(
        "title"
    ):
        return str(
            record[
                "title"
            ]
        )

    metadata = (
        record.get(
            "source_metadata"
        )
        or
        {}
    )

    if isinstance(
        metadata,
        dict,
    ):

        for key in [
            "title",
            "article_title",
            "page_title",
        ]:

            if metadata.get(
                key
            ):
                return str(
                    metadata[
                        key
                    ]
                )

    return "NO_TITLE"


def cell109_get_url(record):

    candidates = []

    locator = (
        record.get(
            "locator"
        )
        or
        {}
    )

    metadata = (
        record.get(
            "source_metadata"
        )
        or
        {}
    )

    if isinstance(
        locator,
        dict,
    ):

        for key in [
            "canonical_url",
            "url",
            "source_url",
        ]:

            if locator.get(
                key
            ):
                candidates.append(
                    locator[
                        key
                    ]
                )

    if isinstance(
        metadata,
        dict,
    ):

        for key in [
            "canonical_url",
            "url",
            "source_url",
        ]:

            if metadata.get(
                key
            ):
                candidates.append(
                    metadata[
                        key
                    ]
                )

    return (
        candidates[0]
        if candidates
        else None
    )


# ============================================================
# 4) Resolve Registry + frozen Corpus
# ============================================================

cell109_repo_root = (
    cell109_find_repo_root()
)


cell109_check(
    "REPO_ROOT_NOT_FOUND",
    cell109_repo_root
    is not None,
)


if cell109_failures:
    raise AssertionError(
        cell109_failures
    )


cell109_registry_path = (
    cell109_repo_root
    /
    "data"
    /
    "phase2"
    /
    "human_review"
    /
    "human_review_registry.json"
)


cell109_corpus_path = (
    cell109_repo_root
    /
    "data"
    /
    "phase2"
    /
    "processed"
    /
    "documents_dedup_quality_accepted_v1.jsonl"
)


cell109_check(
    "REGISTRY_NOT_FOUND",
    cell109_registry_path.exists(),
)


cell109_check(
    "CORPUS_NOT_FOUND",
    cell109_corpus_path.exists(),
)


cell109_corpus_sha256 = (
    cell109_sha256_file(
        cell109_corpus_path
    )
)


cell109_check(
    "FROZEN_CORPUS_SHA256_MISMATCH",
    cell109_corpus_sha256
    ==
    CELL109_EXPECTED_CORPUS_SHA256,
    {
        "expected":
            CELL109_EXPECTED_CORPUS_SHA256,

        "actual":
            cell109_corpus_sha256,
    },
)


if cell109_failures:
    raise AssertionError(
        cell109_failures
    )


# ============================================================
# 5) Load only the exact required Corpus documents
# ============================================================

cell109_required_document_ids = sorted({
    source[
        "document_id"
    ]

    for decision
    in CELL109_DECISIONS.values()

    for source
    in decision[
        "sources"
    ]
})


cell109_source_documents = {}


with cell109_corpus_path.open(
    "r",
    encoding="utf-8",
) as f:

    for line_number, line in enumerate(
        f,
        start=1,
    ):

        line = line.strip()

        if not line:
            continue

        record = json.loads(
            line
        )

        document_id = (
            record.get(
                "document_id"
            )
        )

        if (
            document_id
            in
            cell109_required_document_ids
        ):

            cell109_source_documents[
                document_id
            ] = record


cell109_check(
    "REQUIRED_SOURCE_DOCUMENTS_MISSING",
    sorted(
        cell109_source_documents.keys()
    )
    ==
    cell109_required_document_ids,
    {
        "required":
            cell109_required_document_ids,

        "found":
            sorted(
                cell109_source_documents.keys()
            ),
    },
)


for document_id, record in (
    cell109_source_documents.items()
):

    cell109_check(
        (
            "SOURCE_NOT_DIRECT_AUTHOR_WEBSITE:"
            +
            document_id
        ),
        cell109_is_direct_author_website(
            record
        ),
    )


if cell109_failures:
    raise AssertionError(
        cell109_failures
    )


# ============================================================
# 6) Load Registry fresh
# ============================================================

cell109_registry_before_text = (
    cell109_registry_path.read_text(
        encoding="utf-8"
    )
)


cell109_registry = json.loads(
    cell109_registry_before_text
)


cell109_registry_before_sha256 = (
    cell109_sha256_text(
        cell109_registry_before_text
    )
)


cell109_completed_cases = (
    cell109_registry.get(
        "completed_human_review_cases",
        []
    )
    or
    []
)


cell109_author_review = (
    cell109_registry.get(
        "author_review",
        {}
    )
    or
    {}
)


cell109_author_cases = (
    cell109_author_review.get(
        "cases_ready_for_review",
        []
    )
    or
    []
)


cell109_second_review = (
    cell109_registry.get(
        "second_human_review",
        {}
    )
    or
    {}
)


cell109_feedback_batch = (
    cell109_second_review.get(
        CELL109_BATCH_KEY,
        {}
    )
    or
    {}
)


cell109_feedback_cases = (
    cell109_feedback_batch.get(
        "cases",
        []
    )
    or
    []
)


# ============================================================
# 7) Preconditions
# ============================================================

cell109_check(
    "COMPLETED_CASE_COUNT_NOT_22",
    len(
        cell109_completed_cases
    )
    ==
    CELL109_EXPECTED_COMPLETED_CASES,
)


cell109_check(
    "AUTHOR_QUEUE_COUNT_NOT_28",
    len(
        cell109_author_cases
    )
    ==
    CELL109_EXPECTED_AUTHOR_QUEUE,
)


cell109_check(
    "FEEDBACK_BATCH_COUNT_NOT_6",
    len(
        cell109_feedback_cases
    )
    ==
    CELL109_EXPECTED_NEW_CASES,
)


cell109_batch_by_number = {
    case.get(
        "reference_pilot_number"
    ):
        case

    for case
    in cell109_feedback_cases
}


cell109_author_new_by_number = {
    case.get(
        "reference_pilot_number"
    ):
        case

    for case
    in cell109_author_cases[22:]
}


cell109_check(
    "BATCH_REFERENCE_NUMBERS_MISMATCH",
    sorted(
        cell109_batch_by_number.keys()
    )
    ==
    CELL109_REFERENCE_NUMBERS,
)


cell109_check(
    "AUTHOR_REFERENCE_NUMBERS_MISMATCH",
    sorted(
        cell109_author_new_by_number.keys()
    )
    ==
    CELL109_REFERENCE_NUMBERS,
)


for number in (
    CELL109_REFERENCE_NUMBERS
):

    batch_id = (
        cell109_batch_by_number[
            number
        ].get(
            "pilot_case_id"
        )
    )

    author_id = (
        cell109_author_new_by_number[
            number
        ].get(
            "pilot_case_id"
        )
    )

    cell109_check(
        (
            "PILOT_ID_COPY_MISMATCH:"
            +
            str(number)
        ),
        batch_id
        ==
        author_id,
    )


    for case_copy in [
        cell109_batch_by_number[
            number
        ],
        cell109_author_new_by_number[
            number
        ],
    ]:

        cell109_check(
            (
                "AUTHOR_REVIEW_ALREADY_STARTED:"
                +
                str(number)
            ),
            (
                case_copy.get(
                    "author_final_review",
                    {}
                )
                or
                {}
            ).get(
                "status"
            )
            ==
            "NOT_STARTED",
        )


if cell109_failures:
    raise AssertionError(
        cell109_failures
    )


# ============================================================
# 8) Protect original 22 cases + raw historical data
# ============================================================

cell109_completed_snapshot = (
    deepcopy(
        cell109_completed_cases
    )
)


cell109_original_22_author_snapshot = (
    deepcopy(
        cell109_author_cases[:22]
    )
)


cell109_raw_history_before = {}


for number in (
    CELL109_REFERENCE_NUMBERS
):

    cell109_raw_history_before[
        number
    ] = {
        "batch":
            deepcopy(
                cell109_batch_by_number[
                    number
                ].get(
                    "historical_phase1_demo"
                )
            ),

        "author":
            deepcopy(
                cell109_author_new_by_number[
                    number
                ].get(
                    "historical_phase1_demo"
                )
            ),
    }


# ============================================================
# 9) Build verified source objects
# ============================================================

def cell109_verified_source(
    source_config,
):

    document_id = (
        source_config[
            "document_id"
        ]
    )

    record = (
        cell109_source_documents[
            document_id
        ]
    )

    text = (
        record.get(
            "text"
        )
        or
        ""
    )

    return {
        "document_id":
            document_id,

        "source_id":
            record.get(
                "source_id"
            ),

        "source_type":
            "website",

        "authority_class":
            "PRIMARY_DIRECT_AUTHOR",

        "title":
            cell109_get_title(
                record
            ),

        "url":
            cell109_get_url(
                record
            ),

        "role":
            source_config[
                "role"
            ],

        "coverage":
            source_config[
                "coverage"
            ],

        "review_note":
            source_config[
                "review_note"
            ],

        "document_text_sha256":
            cell109_sha256_text(
                text
            ),
    }


# ============================================================
# 10) Apply review to one case copy
# ============================================================

def cell109_apply_review(
    case,
    number,
):

    decision_config = (
        CELL109_DECISIONS[
            number
        ]
    )

    pilot_id = case[
        "pilot_case_id"
    ]


    # --------------------------------------------------------
    # Stable review event
    # --------------------------------------------------------

    event_id = (
        cell109_stable_id(
            "second-source-review",
            {
                "pilot_case_id":
                    pilot_id,

                "review_version":
                    CELL109_REVIEW_VERSION,

                "decision":
                    decision_config[
                        "decision"
                    ],

                "revision_scope":
                    decision_config[
                        "revision_scope"
                    ],

                "sources":
                    [
                        source[
                            "document_id"
                        ]

                        for source
                        in decision_config[
                            "sources"
                        ]
                    ],
            },
        )
    )


    history = case.setdefault(
        "source_review_history",
        [],
    )


    existing_event = None


    for event in history:

        if (
            isinstance(
                event,
                dict,
            )
            and
            event.get(
                "review_event_id"
            )
            ==
            event_id
        ):

            existing_event = event
            break


    if existing_event is not None:

        review_timestamp = (
            existing_event[
                "review_timestamp_utc"
            ]
        )

    else:

        review_timestamp = (
            cell109_now()
        )


    # --------------------------------------------------------
    # Preserve superseded Pilot 25 state BEFORE revision
    # --------------------------------------------------------

    superseded_snapshot = None


    if number == 25:

        previous_question = (
            (
                case.get(
                    "derived_question",
                    {}
                )
                or
                {}
            ).get(
                "question_text"
            )
        )

        previous_behavior = (
            (
                case.get(
                    "expected_behavior_review",
                    {}
                )
                or
                {}
            ).get(
                "expected_behavior"
            )
        )

        superseded_snapshot = {
            "previous_pilot_type":
                case.get(
                    "pilot_type"
                ),

            "previous_golden_content_case":
                case.get(
                    "golden_content_case"
                ),

            "previous_question":
                previous_question,

            "previous_expected_behavior":
                previous_behavior,

            "previous_regression_role":
                case.get(
                    "regression_role"
                ),

            "superseded_reason":
                (
                    "The historical demo turns did not provide "
                    "reliable connected-chat context. The old "
                    "continuity interpretation is preserved only "
                    "as history, while the active Pilot is revised "
                    "to a source-grounded content case."
                ),
        }


        new_question = (
            decision_config[
                "new_question"
            ]
        )

        new_answer = (
            decision_config[
                "new_expected_answer"
            ]
        )


        new_question_id = (
            cell109_stable_id(
                "question-draft",
                {
                    "pilot_case_id":
                        pilot_id,

                    "question":
                        new_question,

                    "review_version":
                        CELL109_REVIEW_VERSION,
                },
            )
        )


        new_answer_id = (
            cell109_stable_id(
                "expected-answer-draft",
                {
                    "pilot_case_id":
                        pilot_id,

                    "question_draft_id":
                        new_question_id,

                    "answer":
                        new_answer,

                    "review_version":
                        CELL109_REVIEW_VERSION,
                },
            )
        )


        case[
            "pilot_type"
        ] = (
            decision_config[
                "new_pilot_type"
            ]
        )


        case[
            "golden_content_case"
        ] = True


        case[
            "derived_question"
        ] = {
            "question_draft_id":
                new_question_id,

            "question_text":
                new_question,

            "review_origin":
                "SECOND_HUMAN_SOURCE_REVIEW",

            "human_decision":
                "REVISE",

            "external_knowledge_used":
                False,
        }


        case[
            "expected_answer_review"
        ] = {
            "expected_answer_draft_id":
                new_answer_id,

            "expected_answer_text":
                new_answer,

            "human_decision":
                "REVISE",

            "grounding_source":
                "PHASE2_ACCEPTED_DIRECT_AUTHOR_CORPUS",

            "external_knowledge_used":
                False,
        }


        # Do not delete old behavior evidence.
        old_behavior = (
            case.get(
                "expected_behavior_review"
            )
        )

        if isinstance(
            old_behavior,
            dict,
        ):

            old_behavior[
                "status"
            ] = (
                "SUPERSEDED_BY_SOURCE_GROUNDED_CONTENT_REVISION"
            )


        if (
            "regression_role"
            in case
        ):

            case[
                "regression_role_status"
            ] = (
                "SUPERSEDED"
            )


    # --------------------------------------------------------
    # Verified Corpus sources
    # --------------------------------------------------------

    verified_sources = [
        cell109_verified_source(
            source
        )

        for source
        in decision_config[
            "sources"
        ]
    ]


    source_mapping_id = (
        cell109_stable_id(
            "source-mapping",
            {
                "pilot_case_id":
                    pilot_id,

                "review_version":
                    CELL109_REVIEW_VERSION,

                "document_ids":
                    [
                        source[
                            "document_id"
                        ]

                        for source
                        in verified_sources
                    ],

                "grounding_mode":
                    decision_config[
                        "grounding_mode"
                    ],
            },
        )
    )


    review_event = {
        "review_event_id":
            event_id,

        "review_version":
            CELL109_REVIEW_VERSION,

        "review_timestamp_utc":
            review_timestamp,

        "decision":
            decision_config[
                "decision"
            ],

        "revision_scope":
            decision_config[
                "revision_scope"
            ],

        "grounding_mode":
            decision_config[
                "grounding_mode"
            ],

        "source_mapping_id":
            source_mapping_id,

        "verified_corpus_sources":
            deepcopy(
                verified_sources
            ),

        "direct_author_feedback_preserved":
            True,

        "external_knowledge_used":
            False,

        "author_final_review_recorded":
            False,
    }


    if (
        "grounding_constraint"
        in decision_config
    ):

        review_event[
            "grounding_constraint"
        ] = (
            decision_config[
                "grounding_constraint"
            ]
        )


    if superseded_snapshot is not None:

        review_event[
            "superseded_state"
        ] = (
            superseded_snapshot
        )


    # --------------------------------------------------------
    # Idempotent history append / exact-match validation
    # --------------------------------------------------------

    if existing_event is None:

        history.append(
            deepcopy(
                review_event
            )
        )

    else:

        cell109_check(
            (
                "EXISTING_REVIEW_EVENT_MISMATCH:"
                +
                pilot_id
            ),
            existing_event
            ==
            review_event,
        )


    # --------------------------------------------------------
    # Current Source Review state
    # --------------------------------------------------------

    case[
        "second_source_review"
    ] = {
        "status":
            "COMPLETE",

        "review_event_id":
            event_id,

        "review_version":
            CELL109_REVIEW_VERSION,

        "decision":
            decision_config[
                "decision"
            ],

        "revision_scope":
            decision_config[
                "revision_scope"
            ],

        "grounding_mode":
            decision_config[
                "grounding_mode"
            ],

        "source_mapping_id":
            source_mapping_id,

        "verified_corpus_sources":
            deepcopy(
                verified_sources
            ),

        "human_source_decision":
            decision_config[
                "decision"
            ],

        "review_timestamp_utc":
            review_timestamp,

        "direct_author_feedback_preserved":
            True,

        "author_final_review_status":
            "NOT_STARTED",
    }


    if (
        "grounding_constraint"
        in decision_config
    ):

        case[
            "second_source_review"
        ][
            "grounding_constraint"
        ] = (
            decision_config[
                "grounding_constraint"
            ]
        )


    # --------------------------------------------------------
    # Rebuild citation review WITHOUT deleting Author Feedback
    # --------------------------------------------------------

    old_citation_entries = (
        (
            case.get(
                "citation_review",
                {}
            )
            or
            {}
        ).get(
            "citation_entries",
            []
        )
        or
        []
    )


    direct_feedback_entries = [
        deepcopy(
            entry
        )

        for entry
        in old_citation_entries

        if (
            isinstance(
                entry,
                dict,
            )
            and
            entry.get(
                "source_type"
            )
            ==
            "DIRECT_AUTHOR_FEEDBACK"
        )
    ]


    if not direct_feedback_entries:

        historical = (
            case.get(
                "historical_phase1_demo",
                {}
            )
            or
            {}
        )

        raw_feedback = (
            historical.get(
                "raw_author_feedback"
            )
            or
            ""
        )

        feedback_source = (
            case.get(
                "feedback_source",
                {}
            )
            or
            {}
        )

        direct_feedback_entries = [
            {
                "source_type":
                    "DIRECT_AUTHOR_FEEDBACK",

                "feedback_source_id":
                    feedback_source.get(
                        "feedback_source_id"
                    ),

                "raw_author_feedback_sha256":
                    cell109_sha256_text(
                        raw_feedback
                    ),

                "authority_class":
                    "PRIMARY_DIRECT_AUTHOR_FEEDBACK",

                "role":
                    "PRIMARY_AUTHOR_EVIDENCE",
            },
        ]


    website_entries = []


    for source in verified_sources:

        website_entries.append({
            "source_type":
                "WEBSITE_DIRECT_AUTHOR",

            "document_id":
                source[
                    "document_id"
                ],

            "source_id":
                source[
                    "source_id"
                ],

            "title":
                source[
                    "title"
                ],

            "url":
                source[
                    "url"
                ],

            "authority_class":
                source[
                    "authority_class"
                ],

            "role":
                source[
                    "role"
                ],

            "coverage":
                source[
                    "coverage"
                ],

            "document_text_sha256":
                source[
                    "document_text_sha256"
                ],
        })


    citation_payload = {
        "pilot_case_id":
            pilot_id,

        "review_version":
            CELL109_REVIEW_VERSION,

        "direct_feedback":
            direct_feedback_entries,

        "website_sources":
            website_entries,
    }


    citation_mapping_id = (
        cell109_stable_id(
            "citation-mapping",
            citation_payload,
        )
    )


    citation_entries = []


    for entry in (
        direct_feedback_entries
        +
        website_entries
    ):

        entry = deepcopy(
            entry
        )

        entry[
            "citation_index"
        ] = len(
            citation_entries
        )

        citation_entries.append(
            entry
        )


    case[
        "citation_review"
    ] = {
        "citation_mapping_id":
            citation_mapping_id,

        "human_decision":
            "ACCEPT",

        "citation_entries":
            citation_entries,

        "grounding_mode":
            decision_config[
                "grounding_mode"
            ],

        "source_review_event_id":
            event_id,
    }


    # --------------------------------------------------------
    # Keep Author Review untouched
    # --------------------------------------------------------

    author_final_review = (
        case.setdefault(
            "author_final_review",
            {},
        )
    )


    author_final_review.update({
        "status":
            "NOT_STARTED",

        "reviewer":
            "AUTHOR",

        "decision":
            None,

        "notes":
            None,
    })


    # Promotion remains valid; source review is now completed.
    promotion = (
        case.setdefault(
            "promotion",
            {},
        )
    )

    promotion[
        "source_review_status"
    ] = (
        "COMPLETE"
    )


    return case


# ============================================================
# 11) Apply to BOTH copies
# ============================================================

for number in (
    CELL109_REFERENCE_NUMBERS
):

    cell109_apply_review(
        cell109_batch_by_number[
            number
        ],
        number,
    )


    cell109_apply_review(
        cell109_author_new_by_number[
            number
        ],
        number,
    )


if cell109_failures:

    print(
        cell109_failures
    )

    raise AssertionError(
        "Cell 109 source-review application failed."
    )


# ============================================================
# 12) Sync Batch-level status
# ============================================================

cell109_feedback_batch.update({
    "source_review_status":
        "COMPLETE",

    "source_review_version":
        CELL109_REVIEW_VERSION,

    "source_review_case_count":
        6,

    "source_review_decisions": {
        "ACCEPT":
            3,

        "REVISE":
            3,

        "REJECT":
            0,

        "RETARGET":
            0,
    },

    "content_pilot_count_after_source_review":
        6,

    "continuity_regression_pilot_count_after_source_review":
        0,

    "author_final_review_status":
        "NOT_STARTED",
})


# ============================================================
# 13) Strong in-memory validation
# ============================================================

cell109_check(
    "ORIGINAL_COMPLETED_22_MUTATED",
    cell109_registry.get(
        "completed_human_review_cases",
        []
    )
    ==
    cell109_completed_snapshot,
)


cell109_check(
    "ORIGINAL_AUTHOR_22_MUTATED",
    cell109_author_cases[:22]
    ==
    cell109_original_22_author_snapshot,
)


for number in (
    CELL109_REFERENCE_NUMBERS
):

    batch_case = (
        cell109_batch_by_number[
            number
        ]
    )

    author_case = (
        cell109_author_new_by_number[
            number
        ]
    )


    # Raw historical Phase-1 material must be untouched.
    cell109_check(
        (
            "RAW_BATCH_HISTORY_MUTATED:"
            +
            str(number)
        ),
        batch_case.get(
            "historical_phase1_demo"
        )
        ==
        cell109_raw_history_before[
            number
        ][
            "batch"
        ],
    )


    cell109_check(
        (
            "RAW_AUTHOR_HISTORY_MUTATED:"
            +
            str(number)
        ),
        author_case.get(
            "historical_phase1_demo"
        )
        ==
        cell109_raw_history_before[
            number
        ][
            "author"
        ],
    )


    # Both copies must agree on active content fields.
    for field_name in [
        "pilot_type",
        "golden_content_case",
        "derived_question",
        "expected_answer_review",
        "second_source_review",
        "citation_review",
    ]:

        cell109_check(
            (
                "COPY_SYNC_MISMATCH:"
                +
                str(number)
                +
                ":"
                +
                field_name
            ),
            batch_case.get(
                field_name
            )
            ==
            author_case.get(
                field_name
            ),
        )


    cell109_check(
        (
            "SOURCE_REVIEW_NOT_COMPLETE:"
            +
            str(number)
        ),
        (
            batch_case.get(
                "second_source_review",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "COMPLETE",
    )


    expected_decision = (
        CELL109_DECISIONS[
            number
        ][
            "decision"
        ]
    )


    cell109_check(
        (
            "SOURCE_DECISION_MISMATCH:"
            +
            str(number)
        ),
        (
            batch_case.get(
                "second_source_review",
                {}
            )
            or
            {}
        ).get(
            "decision"
        )
        ==
        expected_decision,
    )


    cell109_check(
        (
            "AUTHOR_REVIEW_STARTED:"
            +
            str(number)
        ),
        (
            author_case.get(
                "author_final_review",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "NOT_STARTED",
    )


# Pilot 25 special validation.
cell109_case25 = (
    cell109_batch_by_number[
        25
    ]
)


cell109_check(
    "PILOT25_NOT_CONVERTED_TO_CONTENT",
    cell109_case25.get(
        "pilot_type"
    )
    ==
    "AUTHOR_FEEDBACK_CONTENT",
)


cell109_check(
    "PILOT25_GOLDEN_CONTENT_FALSE",
    cell109_case25.get(
        "golden_content_case"
    )
    is True,
)


cell109_check(
    "PILOT25_NEW_QUESTION_MISMATCH",
    (
        cell109_case25.get(
            "derived_question",
            {}
        )
        or
        {}
    ).get(
        "question_text"
    )
    ==
    CELL109_DECISIONS[
        25
    ][
        "new_question"
    ],
)


cell109_check(
    "PILOT25_NEW_ANSWER_MISMATCH",
    (
        cell109_case25.get(
            "expected_answer_review",
            {}
        )
        or
        {}
    ).get(
        "expected_answer_text"
    )
    ==
    CELL109_DECISIONS[
        25
    ][
        "new_expected_answer"
    ],
)


cell109_check(
    "PILOT25_OLD_BEHAVIOR_HISTORY_NOT_PRESERVED",
    bool(
        (
            cell109_case25.get(
                "expected_behavior_review",
                {}
            )
            or
            {}
        ).get(
            "expected_behavior"
        )
    ),
)


if cell109_failures:

    print(
        "\n=== IN-MEMORY VALIDATION FAILURES ==="
    )

    print(
        cell109_failures
    )

    raise AssertionError(
        "Cell 109 failed before Registry write."
    )


# ============================================================
# 14) Serialize + backup + write
# ============================================================

cell109_registry_after_text = (
    json.dumps(
        cell109_registry,
        ensure_ascii=False,
        indent=2,
    )
    +
    "\n"
)


cell109_change_required = (
    cell109_registry_after_text
    !=
    cell109_registry_before_text
)


cell109_backup_path = None


if cell109_change_required:

    timestamp = (
        datetime.now(
            timezone.utc
        ).strftime(
            "%Y%m%dT%H%M%SZ"
        )
    )


    cell109_backup_path = (
        cell109_registry_path.parent
        /
        (
            cell109_registry_path.stem
            +
            ".before_pilots23_28_source_review_"
            +
            timestamp
            +
            cell109_registry_path.suffix
        )
    )


    cell109_backup_path.write_text(
        cell109_registry_before_text,
        encoding="utf-8",
    )


    temp_path = (
        cell109_registry_path.parent
        /
        (
            cell109_registry_path.name
            +
            ".cell109.tmp"
        )
    )


    temp_path.write_text(
        cell109_registry_after_text,
        encoding="utf-8",
    )


    # Parse temp before replacement.
    json.loads(
        temp_path.read_text(
            encoding="utf-8"
        )
    )


    temp_path.replace(
        cell109_registry_path
    )


# ============================================================
# 15) Reload exact persisted Registry
# ============================================================

cell109_persisted_text = (
    cell109_registry_path.read_text(
        encoding="utf-8"
    )
)


cell109_persisted_registry = (
    json.loads(
        cell109_persisted_text
    )
)


cell109_registry_after_sha256 = (
    cell109_sha256_text(
        cell109_persisted_text
    )
)


cell109_persisted_completed = (
    cell109_persisted_registry.get(
        "completed_human_review_cases",
        []
    )
    or
    []
)


cell109_persisted_author_cases = (
    (
        cell109_persisted_registry.get(
            "author_review",
            {}
        )
        or
        {}
    ).get(
        "cases_ready_for_review",
        []
    )
    or
    []
)


cell109_persisted_batch_cases = (
    (
        (
            cell109_persisted_registry.get(
                "second_human_review",
                {}
            )
            or
            {}
        ).get(
            CELL109_BATCH_KEY,
            {}
        )
        or
        {}
    ).get(
        "cases",
        []
    )
    or
    []
)


# ============================================================
# 16) Persisted final validation
# ============================================================

cell109_check(
    "PERSISTED_COMPLETED_COUNT_CHANGED",
    len(
        cell109_persisted_completed
    )
    ==
    22,
)


cell109_check(
    "PERSISTED_AUTHOR_QUEUE_NOT_28",
    len(
        cell109_persisted_author_cases
    )
    ==
    28,
)


cell109_check(
    "PERSISTED_BATCH_NOT_6",
    len(
        cell109_persisted_batch_cases
    )
    ==
    6,
)


cell109_persisted_batch_by_number = {
    case.get(
        "reference_pilot_number"
    ):
        case

    for case
    in cell109_persisted_batch_cases
}


cell109_persisted_author_by_number = {
    case.get(
        "reference_pilot_number"
    ):
        case

    for case
    in cell109_persisted_author_cases[22:]
}


for number in (
    CELL109_REFERENCE_NUMBERS
):

    batch_case = (
        cell109_persisted_batch_by_number[
            number
        ]
    )

    author_case = (
        cell109_persisted_author_by_number[
            number
        ]
    )


    cell109_check(
        (
            "PERSISTED_COPY_SOURCE_REVIEW_MISMATCH:"
            +
            str(number)
        ),
        batch_case.get(
            "second_source_review"
        )
        ==
        author_case.get(
            "second_source_review"
        ),
    )


    cell109_check(
        (
            "PERSISTED_COPY_QUESTION_MISMATCH:"
            +
            str(number)
        ),
        batch_case.get(
            "derived_question"
        )
        ==
        author_case.get(
            "derived_question"
        ),
    )


    cell109_check(
        (
            "PERSISTED_AUTHOR_NOT_NOT_STARTED:"
            +
            str(number)
        ),
        (
            author_case.get(
                "author_final_review",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "NOT_STARTED",
    )


# Final Pilot 25 persisted state.
cell109_persisted_case25 = (
    cell109_persisted_batch_by_number[
        25
    ]
)


cell109_check(
    "PERSISTED_PILOT25_TYPE_WRONG",
    cell109_persisted_case25.get(
        "pilot_type"
    )
    ==
    "AUTHOR_FEEDBACK_CONTENT",
)


cell109_check(
    "PERSISTED_PILOT25_ANSWER_MISSING",
    bool(
        (
            cell109_persisted_case25.get(
                "expected_answer_review",
                {}
            )
            or
            {}
        ).get(
            "expected_answer_text"
        )
    ),
)


# ============================================================
# 17) Final Gate / Human-readable summary
# ============================================================

cell109_gate = (
    len(
        cell109_failures
    )
    ==
    0
)


print(
    "=== CELL 109 — APPLY SOURCE REVIEW: PILOTS 23-28 ==="
)


print(
    "\nFrozen Corpus SHA256:",
    cell109_corpus_sha256,
)


print(
    "Verified Corpus Source Documents:",
    len(
        cell109_source_documents
    ),
)


print(
    "\n=== HUMAN SOURCE REVIEW DECISIONS ==="
)


for number in (
    CELL109_REFERENCE_NUMBERS
):

    case = (
        cell109_persisted_batch_by_number[
            number
        ]
    )

    review = (
        case.get(
            "second_source_review",
            {}
        )
        or
        {}
    )

    print(
        "\n"
        +
        "="
        *
        90
    )

    print(
        "Pilot:",
        number,
    )

    print(
        "Pilot ID:",
        case.get(
            "pilot_case_id"
        ),
    )

    print(
        "Decision:",
        review.get(
            "decision"
        ),
    )

    print(
        "Revision scope:",
        review.get(
            "revision_scope"
        ),
    )

    print(
        "Pilot type:",
        case.get(
            "pilot_type"
        ),
    )

    print(
        "Golden content case:",
        case.get(
            "golden_content_case"
        ),
    )

    print(
        "Question:"
    )

    print(
        (
            case.get(
                "derived_question",
                {}
            )
            or
            {}
        ).get(
            "question_text"
        )
    )

    if case.get(
        "golden_content_case"
    ):

        print(
            "Expected Answer:"
        )

        print(
            (
                case.get(
                    "expected_answer_review",
                    {}
                )
                or
                {}
            ).get(
                "expected_answer_text"
            )
        )


    print(
        "Grounding mode:",
        review.get(
            "grounding_mode"
        ),
    )


    print(
        "Verified Corpus Sources:"
    )

    for source in (
        review.get(
            "verified_corpus_sources",
            []
        )
    ):

        print(
            " -",
            source.get(
                "document_id"
            ),
            "|",
            source.get(
                "title"
            ),
            "|",
            source.get(
                "coverage"
            ),
        )


    print(
        "Author Final Review:",
        (
            case.get(
                "author_final_review",
                {}
            )
            or
            {}
        ).get(
            "status"
        ),
    )


print(
    "\n=== EXPECTED DECISION COUNTS ==="
)


print(
    "ACCEPT:",
    sum(
        1

        for number
        in CELL109_REFERENCE_NUMBERS

        if (
            CELL109_DECISIONS[
                number
            ][
                "decision"
            ]
            ==
            "ACCEPT"
        )
    ),
)


print(
    "REVISE:",
    sum(
        1

        for number
        in CELL109_REFERENCE_NUMBERS

        if (
            CELL109_DECISIONS[
                number
            ][
                "decision"
            ]
            ==
            "REVISE"
        )
    ),
)


print(
    "REJECT:",
    0,
)


print(
    "RETARGET:",
    0,
)


print(
    "\n=== POST-REVIEW ACTIVE PILOT TYPES ==="
)


print(
    "AUTHOR_FEEDBACK_CONTENT:",
    sum(
        1

        for case
        in cell109_persisted_batch_cases

        if (
            case.get(
                "pilot_type"
            )
            ==
            "AUTHOR_FEEDBACK_CONTENT"
        )
    ),
)


print(
    "MULTI_TURN_CONTEXT_CONTINUITY:",
    sum(
        1

        for case
        in cell109_persisted_batch_cases

        if (
            case.get(
                "pilot_type"
            )
            ==
            "MULTI_TURN_CONTEXT_CONTINUITY"
        )
    ),
)


print(
    "\n=== PROTECTION / SAFETY ==="
)


print(
    "Original completed 22 cases changed:",
    False,
)


print(
    "Original first 22 Author Review cases changed:",
    False,
)


print(
    "Raw Phase-1 conversations changed:",
    False,
)


print(
    "Raw Dr. Roudi feedback changed:",
    False,
)


print(
    "Pilot 25 old continuity history preserved:",
    True,
)


print(
    "Author decisions recorded:",
    False,
)


print(
    "Author Review queue count:",
    len(
        cell109_persisted_author_cases
    ),
)


print(
    "Markdown changed:",
    False,
)


print(
    "Commit / Push:",
    False,
)


print(
    "\n=== WRITE ==="
)


print(
    "Registry write required:",
    cell109_change_required,
)


print(
    "Backup:",
    (
        str(
            cell109_backup_path
        )
        if cell109_backup_path
        is not None
        else
        "NOT_NEEDED_ALREADY_APPLIED"
    ),
)


print(
    "Registry SHA256 before:",
    cell109_registry_before_sha256,
)


print(
    "Registry SHA256 after:",
    cell109_registry_after_sha256,
)


print(
    "\n=== FAILURES ==="
)


print(
    cell109_failures
)


print(
    "\nCODE CELL 109 GATE:",
    (
        "PASS"
        if cell109_gate
        else
        "FAIL"
    ),
)


assert cell109_gate, (
    "Pilots 23-28 Source Review application failed. "
    "Repair this SAME Code Cell 109 before updating the "
    "28-case Author Review Markdown."
)

=== CELL 109 — APPLY SOURCE REVIEW: PILOTS 23-28 ===

Frozen Corpus SHA256: 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40
Verified Corpus Source Documents: 9

=== HUMAN SOURCE REVIEW DECISIONS ===

Pilot: 23
Pilot ID: question-pilot:a0f61a7f0c946fe1d79095dbb96a9497
Decision: ACCEPT
Revision scope: NONE
Pilot type: AUTHOR_FEEDBACK_CONTENT
Golden content case: True
Question:
سواد مالی درباره میزان پولی است که داریم یا درباره مدیریت پول؟
Expected Answer:
سواد مالی درباره مدیریت پول است، نه درباره میزان پول.
Grounding mode: DIRECT_AUTHOR_FEEDBACK_PLUS_CORPUS_DIRECT_SUPPORT
Verified Corpus Sources:
 - website:7C71884B | ابزارهای سواد مالی برای بهبود مدیریت هزینه | DIRECT_EXPLICIT
Author Final Review: NOT_STARTED

Pilot: 24
Pilot ID: question-pilot:70628caeefebbfba48300176fafddc96
Decision: ACCEPT
Revision scope: NONE
Pilot type: AUTHOR_FEEDBACK_CONTENT
Golden content case: True
Question:
برای اینکه بفهمیم از نظر دانش، نگرش و مهارت‌های مالی وضعیت قابل قبولی داریم، چه باید 

<div style="direction: rtl; text-align: right; white-space: normal; line-height: 1.8;">

## ارتقای پایلوت‌های 23 تا 28 به صف بازبینی مؤلف

در مرحله قبل، ۶ پایلوت جدید بر اساس بازخورد واقعی دکتر کمیل رودی در دموی فاز یک ساخته و در یک Batch مستقل ثبت شدند.

وضعیت قبل از این مرحله:

- پایلوت‌های 1 تا 22: آماده بازبینی مؤلف
- پایلوت‌های 23 تا 28: ساخته‌شده و آماده Promotion
- تعداد کل فعلی در صف Author Review: 22
- Author Final Review: هنوز شروع نشده است

در این مرحله، ۶ پایلوت جدید بدون تغییر ۲۲ مورد قبلی به انتهای صف بازبینی مؤلف اضافه می‌شوند.

ترتیب جدید:

- موارد 1 تا 22: پایلوت‌های قبلی فاز دو
- مورد 23: تعریف سواد مالی
- مورد 24: ارزیابی دانش، نگرش و مهارت مالی
- مورد 25: تداوم Context در سؤال پیگیری
- مورد 26: الگوهای خرید و هدررفت پول
- مورد 27: روش‌های افزایش درآمد
- مورد 28: احساس فقر با وجود درآمد

پس از اجرای موفق Cell بعدی:

**Author Review Queue = 28 Cases**

توجه:

- ۲۲ مورد قبلی نباید تغییر کنند.
- `completed_human_review_cases` همچنان ۲۲ مورد باقی می‌ماند.
- ۶ مورد جدید از Batch بازخورد فاز یک وارد صف بازبینی دکتر می‌شوند.
- هیچ تصمیمی از طرف دکتر ثبت نمی‌شود.
- `Author Final Review = NOT_STARTED`
- Raw Feedback دکتر تغییر نمی‌کند.
- هیچ Commit یا Push انجام نمی‌شود.

</div>

In [9]:
# Code Cell 110
# ============================================================
# PROMOTE PILOTS 23-28 TO THE AUTHOR REVIEW QUEUE
#
# Self-contained:
# - reads Registry directly from disk;
# - does NOT depend on Cell 107 variables in memory;
# - preserves the existing 22 Author Review cases byte-for-byte
#   at the object level;
# - appends the six Phase-1 Author Feedback derived Pilots;
# - updates Author Review queue count from 22 to 28;
# - updates promotion metadata for the six new Pilots;
# - preserves completed_human_review_cases at 22;
# - keeps all Author decisions NOT_STARTED;
# - creates backup before write;
# - reloads and verifies persisted state;
# - is idempotent.
# ============================================================

from pathlib import Path
from copy import deepcopy
from datetime import datetime, timezone
import hashlib
import json


# ============================================================
# 1) Constants
# ============================================================

CELL110_EXPECTED_OLD_AUTHOR_COUNT = 22
CELL110_EXPECTED_NEW_AUTHOR_COUNT = 28
CELL110_EXPECTED_NEW_PILOTS = 6

CELL110_BATCH_KEY = (
    "phase1_author_feedback_derived_pilots"
)

CELL110_EXPECTED_REFERENCE_NUMBERS = [
    23,
    24,
    25,
    26,
    27,
    28,
]


# ============================================================
# 2) Utilities
# ============================================================

cell110_failures = []


def cell110_check(
    name,
    condition,
    details=None,
):

    if not bool(condition):

        failure = {
            "reason": name,
        }

        if details is not None:
            failure["details"] = details

        cell110_failures.append(
            failure
        )

    return bool(condition)


def cell110_now():

    return (
        datetime.now(
            timezone.utc
        )
        .replace(
            microsecond=0
        )
        .isoformat()
    )


def cell110_sha256_text(text):

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def cell110_stable_id(
    prefix,
    payload,
):

    if not isinstance(
        payload,
        str,
    ):

        payload = json.dumps(
            payload,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        )

    digest = hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()[:32]

    return (
        prefix
        +
        ":"
        +
        digest
    )


def cell110_find_repo_root():

    candidates = []

    # Reuse known path only as a convenience if available.
    if "cell105_registry_path" in globals():

        known_registry = Path(
            cell105_registry_path
        ).resolve()

        candidates.extend(
            [
                known_registry.parent,
                *known_registry.parents,
            ]
        )

    cwd = Path.cwd().resolve()

    candidates.extend(
        [
            cwd,
            *cwd.parents,
        ]
    )

    seen = set()

    for candidate in candidates:

        candidate = candidate.resolve()

        if str(candidate) in seen:
            continue

        seen.add(
            str(candidate)
        )

        registry_candidate = (
            candidate
            /
            "data"
            /
            "phase2"
            /
            "human_review"
            /
            "human_review_registry.json"
        )

        if registry_candidate.exists():
            return candidate

    return None


# ============================================================
# 3) Resolve Registry independently
# ============================================================

cell110_repo_root = (
    cell110_find_repo_root()
)


cell110_check(
    "REPO_ROOT_NOT_FOUND",
    cell110_repo_root is not None,
)


if cell110_failures:
    raise AssertionError(
        cell110_failures
    )


cell110_registry_path = (
    cell110_repo_root
    /
    "data"
    /
    "phase2"
    /
    "human_review"
    /
    "human_review_registry.json"
)


cell110_check(
    "REGISTRY_NOT_FOUND",
    cell110_registry_path.exists(),
    str(cell110_registry_path),
)


if cell110_failures:
    raise AssertionError(
        cell110_failures
    )


# ============================================================
# 4) Load Registry fresh from disk
# ============================================================

cell110_registry_before_text = (
    cell110_registry_path.read_text(
        encoding="utf-8"
    )
)


cell110_registry = json.loads(
    cell110_registry_before_text
)


cell110_registry_before_sha256 = (
    cell110_sha256_text(
        cell110_registry_before_text
    )
)


# ============================================================
# 5) Resolve protected existing collections
# ============================================================

cell110_completed_cases = (
    cell110_registry.get(
        "completed_human_review_cases",
        []
    )
    or
    []
)


cell110_author_review = (
    cell110_registry.get(
        "author_review",
        {}
    )
    or
    {}
)


cell110_author_cases = (
    cell110_author_review.get(
        "cases_ready_for_review",
        []
    )
    or
    []
)


cell110_second_review = (
    cell110_registry.get(
        "second_human_review",
        {}
    )
    or
    {}
)


cell110_feedback_batch = (
    cell110_second_review.get(
        CELL110_BATCH_KEY,
        {}
    )
    or
    {}
)


cell110_feedback_cases = (
    cell110_feedback_batch.get(
        "cases",
        []
    )
    or
    []
)


# ============================================================
# 6) Validate source Batch
# ============================================================

cell110_check(
    "COMPLETED_HUMAN_REVIEW_COUNT_NOT_22",
    len(cell110_completed_cases) == 22,
    len(cell110_completed_cases),
)


cell110_check(
    "PHASE1_FEEDBACK_BATCH_NOT_FOUND",
    bool(cell110_feedback_batch),
)


cell110_check(
    "PHASE1_FEEDBACK_BATCH_CASE_COUNT_NOT_6",
    len(cell110_feedback_cases)
    ==
    CELL110_EXPECTED_NEW_PILOTS,
    len(cell110_feedback_cases),
)


cell110_reference_numbers = [
    case.get(
        "reference_pilot_number"
    )

    for case
    in cell110_feedback_cases
]


cell110_check(
    "PHASE1_FEEDBACK_REFERENCE_NUMBERS_MISMATCH",
    cell110_reference_numbers
    ==
    CELL110_EXPECTED_REFERENCE_NUMBERS,
    cell110_reference_numbers,
)


cell110_feedback_ids = [
    case.get(
        "pilot_case_id"
    )

    for case
    in cell110_feedback_cases
]


cell110_check(
    "PHASE1_FEEDBACK_IDS_NOT_UNIQUE",
    len(
        set(cell110_feedback_ids)
    )
    ==
    6,
)


for case in cell110_feedback_cases:

    pilot_id = case.get(
        "pilot_case_id",
        "UNKNOWN",
    )

    cell110_check(
        f"AUTHOR_REVIEW_ALREADY_STARTED:{pilot_id}",
        (
            case.get(
                "author_final_review",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "NOT_STARTED",
    )

    promotion_status = (
        case.get(
            "promotion",
            {}
        )
        or
        {}
    ).get(
        "status"
    )

    cell110_check(
        f"UNEXPECTED_PROMOTION_STATUS:{pilot_id}",
        promotion_status
        in
        {
            "READY_FOR_AUTHOR_REVIEW",
            "PROMOTED_TO_AUTHOR_REVIEW",
        },
        promotion_status,
    )


if cell110_failures:

    print(
        "\n=== SOURCE BATCH VALIDATION FAILURES ==="
    )

    print(
        cell110_failures
    )

    raise AssertionError(
        "Cell 110 blocked: source Batch is not valid."
    )


# ============================================================
# 7) Determine whether promotion was already completed
# ============================================================

cell110_existing_author_ids = [
    case.get(
        "pilot_case_id"
    )

    for case
    in cell110_author_cases
]


cell110_new_ids_already_present = [
    pilot_id

    for pilot_id
    in cell110_feedback_ids

    if pilot_id
    in
    cell110_existing_author_ids
]


cell110_already_promoted = (
    len(cell110_author_cases)
    ==
    CELL110_EXPECTED_NEW_AUTHOR_COUNT
    and
    len(cell110_new_ids_already_present)
    ==
    6
)


cell110_fresh_promotion = (
    len(cell110_author_cases)
    ==
    CELL110_EXPECTED_OLD_AUTHOR_COUNT
    and
    len(cell110_new_ids_already_present)
    ==
    0
)


cell110_check(
    "AUTHOR_QUEUE_IS_IN_PARTIAL_OR_UNKNOWN_STATE",
    (
        cell110_fresh_promotion
        or
        cell110_already_promoted
    ),
    {
        "current_author_queue_count":
            len(cell110_author_cases),

        "new_ids_already_present":
            len(
                cell110_new_ids_already_present
            ),
    },
)


if cell110_failures:

    print(
        cell110_failures
    )

    raise AssertionError(
        "Author Review queue is neither clean 22-state nor valid 28-state."
    )


# ============================================================
# 8) Freeze exact original 22 cases
# ============================================================

cell110_original_22_author_cases = deepcopy(
    cell110_author_cases[:22]
)


cell110_completed_cases_snapshot = deepcopy(
    cell110_completed_cases
)


# ============================================================
# 9) Promotion
# ============================================================

cell110_promotion_timestamp = (
    cell110_feedback_batch.get(
        "promotion_timestamp_utc"
    )
    or
    cell110_now()
)


if cell110_fresh_promotion:

    cell110_promoted_cases = []

    for expected_position, source_case in enumerate(
        cell110_feedback_cases,
        start=23,
    ):

        promoted_case = deepcopy(
            source_case
        )

        reference_number = (
            promoted_case.get(
                "reference_pilot_number"
            )
        )

        cell110_check(
            (
                "REFERENCE_NUMBER_POSITION_MISMATCH:"
                +
                str(reference_number)
            ),
            reference_number
            ==
            expected_position,
        )


        promotion = (
            promoted_case.setdefault(
                "promotion",
                {},
            )
        )

        promotion.update({
            "status":
                "PROMOTED_TO_AUTHOR_REVIEW",

            "promoted_to_author_review":
                True,

            "author_review_queue_position":
                expected_position,

            "promotion_timestamp_utc":
                cell110_promotion_timestamp,

            "future_target_total":
                28,
        })


        promoted_case[
            "author_review_queue_position"
        ] = (
            expected_position
        )


        author_final_review = (
            promoted_case.setdefault(
                "author_final_review",
                {},
            )
        )

        author_final_review.update({
            "status":
                "NOT_STARTED",

            "reviewer":
                "AUTHOR",

            "decision":
                None,

            "notes":
                None,
        })


        cell110_promoted_cases.append(
            promoted_case
        )


    if cell110_failures:
        raise AssertionError(
            cell110_failures
        )


    cell110_author_cases.extend(
        cell110_promoted_cases
    )


# ============================================================
# 10) Update source Batch promotion metadata
# ============================================================

for source_case in cell110_feedback_cases:

    reference_number = (
        source_case.get(
            "reference_pilot_number"
        )
    )

    promotion = source_case.setdefault(
        "promotion",
        {},
    )

    promotion.update({
        "status":
            "PROMOTED_TO_AUTHOR_REVIEW",

        "promoted_to_author_review":
            True,

        "author_review_queue_position":
            reference_number,

        "promotion_timestamp_utc":
            cell110_promotion_timestamp,

        "future_target_total":
            28,
    })


cell110_feedback_batch.update({
    "status":
        "COMPLETE_PROMOTED_TO_AUTHOR_REVIEW",

    "currently_promoted_to_author_review":
        True,

    "promotion_timestamp_utc":
        cell110_promotion_timestamp,

    "author_review_queue_count_before_promotion":
        22,

    "author_review_queue_count_after_promotion":
        28,
})


# ============================================================
# 11) Update Author Review queue metadata
# ============================================================

cell110_author_review[
    "cases_ready_for_review"
] = (
    cell110_author_cases
)


cell110_author_review[
    "case_count"
] = (
    len(
        cell110_author_cases
    )
)


# Keep review state untouched / not started.
if (
    "status"
    in
    cell110_author_review
):

    cell110_check(
        "AUTHOR_REVIEW_STATUS_ALREADY_CHANGED",
        cell110_author_review[
            "status"
        ]
        in
        {
            "NOT_STARTED",
            "READY",
            "READY_FOR_REVIEW",
        },
        cell110_author_review[
            "status"
        ],
    )


# ============================================================
# 12) Promotion event record
# ============================================================

cell110_promotion_event_id = (
    cell110_stable_id(
        "author-review-promotion",
        {
            "source_batch":
                CELL110_BATCH_KEY,

            "pilot_ids":
                cell110_feedback_ids,

            "from_count":
                22,

            "to_count":
                28,
        },
    )
)


cell110_promotion_history = (
    cell110_author_review.setdefault(
        "promotion_history",
        [],
    )
)


cell110_existing_event_ids = {
    event.get(
        "promotion_event_id"
    )

    for event
    in cell110_promotion_history

    if isinstance(
        event,
        dict,
    )
}


if (
    cell110_promotion_event_id
    not in
    cell110_existing_event_ids
):

    cell110_promotion_history.append({
        "promotion_event_id":
            cell110_promotion_event_id,

        "promotion_timestamp_utc":
            cell110_promotion_timestamp,

        "source":
            "PHASE1_DEMO_AUTHOR_FEEDBACK",

        "source_batch":
            CELL110_BATCH_KEY,

        "previous_author_review_case_count":
            22,

        "added_case_count":
            6,

        "new_author_review_case_count":
            28,

        "added_reference_pilot_numbers":
            deepcopy(
                CELL110_EXPECTED_REFERENCE_NUMBERS
            ),

        "added_pilot_case_ids":
            deepcopy(
                cell110_feedback_ids
            ),

        "author_decisions_recorded":
            False,
    })


# ============================================================
# 13) Strong in-memory validation
# ============================================================

cell110_check(
    "AUTHOR_QUEUE_COUNT_NOT_28",
    len(
        cell110_author_cases
    )
    ==
    28,
    len(
        cell110_author_cases
    ),
)


cell110_check(
    "AUTHOR_REVIEW_CASE_COUNT_FIELD_NOT_28",
    cell110_author_review.get(
        "case_count"
    )
    ==
    28,
    cell110_author_review.get(
        "case_count"
    ),
)


# First 22 must be exactly unchanged.
cell110_check(
    "ORIGINAL_22_AUTHOR_CASES_MUTATED",
    cell110_author_cases[:22]
    ==
    cell110_original_22_author_cases,
)


# completed_human_review_cases must remain exactly unchanged.
cell110_check(
    "COMPLETED_HUMAN_REVIEW_CASES_MUTATED",
    cell110_registry.get(
        "completed_human_review_cases",
        []
    )
    ==
    cell110_completed_cases_snapshot,
)


cell110_new_author_cases = (
    cell110_author_cases[22:]
)


cell110_check(
    "NEW_AUTHOR_CASE_COUNT_NOT_6",
    len(
        cell110_new_author_cases
    )
    ==
    6,
)


cell110_check(
    "NEW_AUTHOR_REFERENCE_ORDER_MISMATCH",
    [
        case.get(
            "reference_pilot_number"
        )

        for case
        in cell110_new_author_cases
    ]
    ==
    CELL110_EXPECTED_REFERENCE_NUMBERS,
)


cell110_check(
    "NEW_AUTHOR_PILOT_IDS_MISMATCH",
    [
        case.get(
            "pilot_case_id"
        )

        for case
        in cell110_new_author_cases
    ]
    ==
    cell110_feedback_ids,
)


for case in cell110_new_author_cases:

    pilot_id = case.get(
        "pilot_case_id"
    )

    cell110_check(
        f"NEW_CASE_AUTHOR_REVIEW_STARTED:{pilot_id}",
        (
            case.get(
                "author_final_review",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "NOT_STARTED",
    )

    cell110_check(
        f"NEW_CASE_NOT_MARKED_PROMOTED:{pilot_id}",
        (
            case.get(
                "promotion",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "PROMOTED_TO_AUTHOR_REVIEW",
    )


cell110_check(
    "SOURCE_BATCH_NOT_MARKED_PROMOTED",
    cell110_feedback_batch.get(
        "currently_promoted_to_author_review"
    )
    is True,
)


if cell110_failures:

    print(
        "\n=== IN-MEMORY FAILURES ==="
    )

    print(
        cell110_failures
    )

    raise AssertionError(
        "Cell 110 promotion failed before Registry write."
    )


# ============================================================
# 14) Serialize
# ============================================================

cell110_registry_after_text = (
    json.dumps(
        cell110_registry,
        ensure_ascii=False,
        indent=2,
    )
    +
    "\n"
)


cell110_change_required = (
    cell110_registry_after_text
    !=
    cell110_registry_before_text
)


cell110_backup_path = None


# ============================================================
# 15) Backup + atomic-ish write
# ============================================================

if cell110_change_required:

    filename_timestamp = (
        datetime.now(
            timezone.utc
        ).strftime(
            "%Y%m%dT%H%M%SZ"
        )
    )


    cell110_backup_path = (
        cell110_registry_path.parent
        /
        (
            cell110_registry_path.stem
            +
            ".before_author_review_28_promotion_"
            +
            filename_timestamp
            +
            cell110_registry_path.suffix
        )
    )


    cell110_backup_path.write_text(
        cell110_registry_before_text,
        encoding="utf-8",
    )


    cell110_temp_path = (
        cell110_registry_path.parent
        /
        (
            cell110_registry_path.name
            +
            ".cell110.tmp"
        )
    )


    cell110_temp_path.write_text(
        cell110_registry_after_text,
        encoding="utf-8",
    )


    # Ensure temp file parses before replacement.
    json.loads(
        cell110_temp_path.read_text(
            encoding="utf-8"
        )
    )


    cell110_temp_path.replace(
        cell110_registry_path
    )


# ============================================================
# 16) Reload persisted Registry
# ============================================================

cell110_persisted_text = (
    cell110_registry_path.read_text(
        encoding="utf-8"
    )
)


cell110_persisted_registry = (
    json.loads(
        cell110_persisted_text
    )
)


cell110_registry_after_sha256 = (
    cell110_sha256_text(
        cell110_persisted_text
    )
)


# ============================================================
# 17) Persisted validation
# ============================================================

cell110_persisted_completed = (
    cell110_persisted_registry.get(
        "completed_human_review_cases",
        []
    )
    or
    []
)


cell110_persisted_author_review = (
    cell110_persisted_registry.get(
        "author_review",
        {}
    )
    or
    {}
)


cell110_persisted_author_cases = (
    cell110_persisted_author_review.get(
        "cases_ready_for_review",
        []
    )
    or
    []
)


cell110_persisted_batch = (
    (
        cell110_persisted_registry.get(
            "second_human_review",
            {}
        )
        or
        {}
    ).get(
        CELL110_BATCH_KEY,
        {}
    )
    or
    {}
)


cell110_check(
    "PERSISTED_COMPLETED_CASES_CHANGED",
    cell110_persisted_completed
    ==
    cell110_completed_cases_snapshot,
)


cell110_check(
    "PERSISTED_AUTHOR_QUEUE_COUNT_NOT_28",
    len(
        cell110_persisted_author_cases
    )
    ==
    28,
    len(
        cell110_persisted_author_cases
    ),
)


cell110_check(
    "PERSISTED_AUTHOR_CASE_COUNT_FIELD_NOT_28",
    cell110_persisted_author_review.get(
        "case_count"
    )
    ==
    28,
)


cell110_check(
    "PERSISTED_FIRST_22_CHANGED",
    cell110_persisted_author_cases[:22]
    ==
    cell110_original_22_author_cases,
)


cell110_check(
    "PERSISTED_NEW_SIX_IDS_MISMATCH",
    [
        case.get(
            "pilot_case_id"
        )

        for case
        in
        cell110_persisted_author_cases[22:]
    ]
    ==
    cell110_feedback_ids,
)


cell110_check(
    "PERSISTED_BATCH_NOT_PROMOTED",
    cell110_persisted_batch.get(
        "currently_promoted_to_author_review"
    )
    is True,
)


cell110_check(
    "PERSISTED_BATCH_STATUS_MISMATCH",
    cell110_persisted_batch.get(
        "status"
    )
    ==
    "COMPLETE_PROMOTED_TO_AUTHOR_REVIEW",
)


for case in (
    cell110_persisted_author_cases[22:]
):

    pilot_id = case.get(
        "pilot_case_id"
    )

    cell110_check(
        f"PERSISTED_AUTHOR_REVIEW_STARTED:{pilot_id}",
        (
            case.get(
                "author_final_review",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "NOT_STARTED",
    )


# ============================================================
# 18) Final Gate
# ============================================================

cell110_gate = (
    len(
        cell110_failures
    )
    ==
    0
)


print(
    "=== AUTHOR REVIEW QUEUE PROMOTION: 22 -> 28 ==="
)


print(
    "\nExisting completed Human Review cases:",
    len(
        cell110_persisted_completed
    ),
)


print(
    "Author Review queue:",
    len(
        cell110_persisted_author_cases
    ),
)


print(
    "Newly promoted Pilots:",
    6,
)


print(
    "Reference Pilot range added:",
    "23-28",
)


print(
    "\n=== NEW AUTHOR REVIEW CASES ==="
)


for case in (
    cell110_persisted_author_cases[22:]
):

    print(
        "\n"
        +
        "="
        *
        90
    )

    print(
        "Reference Pilot:",
        case.get(
            "reference_pilot_number"
        ),
    )

    print(
        "Pilot ID:",
        case.get(
            "pilot_case_id"
        ),
    )

    print(
        "Type:",
        case.get(
            "pilot_type"
        ),
    )

    print(
        "Question:"
    )

    print(
        (
            case.get(
                "derived_question"
            )
            or
            {}
        ).get(
            "question_text"
        )
    )

    if case.get(
        "golden_content_case"
    ):

        print(
            "Expected Answer:"
        )

        print(
            (
                case.get(
                    "expected_answer_review"
                )
                or
                {}
            ).get(
                "expected_answer_text"
            )
        )

    else:

        print(
            "Expected Behavior:"
        )

        print(
            (
                case.get(
                    "expected_behavior_review"
                )
                or
                {}
            ).get(
                "expected_behavior"
            )
        )

    print(
        "Author Review:",
        (
            case.get(
                "author_final_review"
            )
            or
            {}
        ).get(
            "status"
        ),
    )


print(
    "\n=== PROTECTION ==="
)


print(
    "Original 22 Author Review cases changed:",
    False,
)


print(
    "Completed Human Review cases changed:",
    False,
)


print(
    "Completed Human Review case count:",
    len(
        cell110_persisted_completed
    ),
)


print(
    "Author Review case count:",
    len(
        cell110_persisted_author_cases
    ),
)


print(
    "Author decisions recorded:",
    False,
)


print(
    "\n=== WRITE / SAFETY ==="
)


print(
    "Registry write required:",
    cell110_change_required,
)


print(
    "Backup path:",
    (
        str(
            cell110_backup_path
        )
        if cell110_backup_path
        is not None
        else
        "NOT_NEEDED_ALREADY_PROMOTED"
    ),
)


print(
    "Registry SHA256 before:",
    cell110_registry_before_sha256,
)


print(
    "Registry SHA256 after:",
    cell110_registry_after_sha256,
)


print(
    "Raw Author Feedback mutated:",
    False,
)


print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)


print(
    cell110_failures
)


print(
    "\nCODE CELL 110 GATE:",
    (
        "PASS"
        if cell110_gate
        else
        "FAIL"
    ),
)


assert cell110_gate, (
    "Author Review Queue promotion failed. "
    "Repair this SAME Code Cell 110 before generating the "
    "28-case Author Review document."
)

=== AUTHOR REVIEW QUEUE PROMOTION: 22 -> 28 ===

Existing completed Human Review cases: 22
Author Review queue: 28
Newly promoted Pilots: 6
Reference Pilot range added: 23-28

=== NEW AUTHOR REVIEW CASES ===

Reference Pilot: 23
Pilot ID: question-pilot:a0f61a7f0c946fe1d79095dbb96a9497
Type: AUTHOR_FEEDBACK_CONTENT
Question:
سواد مالی درباره میزان پولی است که داریم یا درباره مدیریت پول؟
Expected Answer:
سواد مالی درباره مدیریت پول است، نه درباره میزان پول.
Author Review: NOT_STARTED

Reference Pilot: 24
Pilot ID: question-pilot:70628caeefebbfba48300176fafddc96
Type: AUTHOR_FEEDBACK_CONTENT
Question:
برای اینکه بفهمیم از نظر دانش، نگرش و مهارت‌های مالی وضعیت قابل قبولی داریم، چه باید کرد؟
Expected Answer:
مدیریت مالی شخصی به دانش، نگرش و مهارت‌های خاص نیاز دارد. برای اینکه بدانیم وضعیت مالی ما از این جنبه‌ها به اندازه کافی مناسب است، باید پرسشنامه مربوط را تکمیل کنیم.
Author Review: NOT_STARTED

Reference Pilot: 25
Pilot ID: question-pilot:77d8b06bc6bccdbbfca4faab99051fe0
Type: AUTHOR_F

<div style="direction: rtl; text-align: right; white-space: normal; line-height: 1.8;">

## تولید بسته ۲۸ موردی بازبینی نهایی مؤلف

در این مرحله، نسخه بازبینی مؤلف از ۲۲ مورد به ۲۸ مورد ارتقا داده می‌شود.

### منبع ساخت

- همان فایل Markdown تأییدشده ۲۲ موردی برای موارد 1 تا 22؛
- Registry فعلی برای پایلوت‌های 23 تا 28؛
- تاریخچه واقعی دموی فاز یک و بازخورد مستقیم دکتر برای موارد 23 تا 28؛
- نتایج Source Mapping و Second Human Source Review انجام‌شده برای پایلوت‌های 23 تا 28.

### وضعیت پایلوت‌های جدید پس از Source Review

هر ۶ پایلوت جدید اکنون از نوع:

`AUTHOR_FEEDBACK_CONTENT`

هستند.

تصمیم‌های Human Source Review:

- Pilot 23 → `ACCEPT`
- Pilot 24 → `ACCEPT`
- Pilot 25 → `REVISE`
- Pilot 26 → `REVISE / GROUNDING_ONLY`
- Pilot 27 → `REVISE / GROUNDING_ONLY`
- Pilot 28 → `ACCEPT`

Pilot 25 دیگر یک پایلوت فعال `MULTI_TURN_CONTEXT_CONTINUITY` نیست.

نسخه قبلی آن به‌عنوان History حفظ شده، اما سؤال و پاسخ فعال جدید آن بر اساس Source واقعی Corpus ساخته شده است.

### قواعد این مرحله

- محتوای تأییدشده موارد 1 تا 22 دوباره از صفر تولید نمی‌شود.
- همان Markdown تأییدشده قبلی مبنا قرار می‌گیرد.
- فقط شمارنده کل از 22 به 28 تغییر می‌کند.
- موارد 23 تا 28 به انتهای همان سند اضافه می‌شوند.
- سؤال‌ها، جواب‌های مدل فاز یک و بازخورد قبلی دکتر باید در History قابل مشاهده باشند.
- Feedback خام دکتر بازنویسی نمی‌شود.
- Sourceهای واقعی تأییدشده در Source Review باید برای موارد 23 تا 28 نمایش داده شوند.
- تصمیم Human Source Review شامل `ACCEPT` یا `REVISE` باید در روند بازبینی هر مورد نمایش داده شود.
- Pilot 25 باید با سؤال و پاسخ جدید فعال نمایش داده شود و History نسخه قبلی آن نیز حذف نشود.
- برای Pilots 26 و 27 باید روشن باشد که Direct Author Feedback منبع اصلی Grounding است و Website Corpus فقط Partial Corroboration ارائه می‌دهد.
- هیچ تصمیمی به نام دکتر ثبت نمی‌شود.
- بخش نظر مؤلف برای تمام موارد همچنان خالی باقی می‌ماند.
- اگر فایل ۲۲ موردی تأییدشده پیدا نشود، Build باید Fail-Closed متوقف شود.
- هیچ Commit یا Push انجام نمی‌شود.

### History مورد 27

جواب کامل مدل فاز یک برای مورد 27 قبلاً بازیابی و در هر دو Copy Registry همگام شده است.

در این مرحله فقط باید صحت باقی‌ماندن آن Verify شود و نباید دوباره به‌عنوان Repair جدید ثبت شود.

### خروجی مورد انتظار

`notebooks/phase2/author_review_28_pilots.md`

در پایان:

- دقیقاً 28 مورد وجود داشته باشد.
- موارد 1 تا 22 همان محتوای تأییدشده قبلی باشند.
- موارد 23 تا 28 دارای History کامل باشند.
- هر ۶ مورد جدید دارای Source Review واقعی باشند.
- هر ۶ مورد جدید `AUTHOR_FEEDBACK_CONTENT` باشند.
- Pilot 25 سؤال و پاسخ اصلاح‌شده فعلی را نمایش دهد.
- Sourceهای Corpus تأییدشده در موارد 23 تا 28 قابل مشاهده باشند.
- تصمیم‌های Human Source Review قابل مشاهده باشند.
- Author Review برای همه موارد جدید `NOT_STARTED` باقی بماند.
- Content Coverage برای هر ۲۸ مورد PASS باشد.

</div>

In [13]:
# Code Cell 111
# ============================================================
# BUILD FINAL 28-CASE AUTHOR REVIEW MARKDOWN
#
# FINAL STRATEGY
#
# Cases 1-22:
#   - reuse exact approved/frozen 22-case Markdown
#   - verify exact SHA256
#   - DO NOT regenerate
#   - DO NOT infer/revalidate their internal Markdown layout
#
# Cases 23-28:
#   - read current persisted Registry
#   - require completed Source Review
#   - require all 6 active cases = AUTHOR_FEEDBACK_CONTENT
#   - include Phase-1 history
#   - include raw Dr. Roudi feedback
#   - include verified Corpus evidence
#   - include Source Review decisions
#   - keep Author Final Review blank / NOT_STARTED
#
# Pilot 25:
#   - active case = content
#   - old continuity interpretation preserved as history
#
# Pilot 26 / 27:
#   - Direct Author Feedback remains primary grounding
#   - Website Corpus is partial corroboration
#
# Registry is READ-ONLY.
# No Author decision.
# No Commit / Push.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import unicodedata


# ============================================================
# 1) Constants
# ============================================================

CELL111_EXPECTED_AUTHOR_CASES = 28
CELL111_EXPECTED_NEW_CASES = 6

CELL111_BATCH_KEY = (
    "phase1_author_feedback_derived_pilots"
)

CELL111_NEW_NUMBERS = [
    23,
    24,
    25,
    26,
    27,
    28,
]

CELL111_EXPECTED_DECISIONS = {
    23: "ACCEPT",
    24: "ACCEPT",
    25: "REVISE",
    26: "REVISE",
    27: "REVISE",
    28: "ACCEPT",
}

CELL111_EXPECTED_REVISION_SCOPES = {
    23: "NONE",
    24: "NONE",
    25: "QUESTION_ANSWER_AND_PILOT_TYPE",
    26: "GROUNDING_ONLY",
    27: "GROUNDING_ONLY",
    28: "NONE",
}

CELL111_EXPECTED_ACTIVE_TYPE = (
    "AUTHOR_FEEDBACK_CONTENT"
)

CELL111_EXPECTED_CORPUS_SHA256 = (
    "9d72c709f81b1218c5aac064052c7ed1"
    "da34869f671e6e32762864e0ad70ea40"
)

CELL111_EXPECTED_SOURCE22_SHA256 = (
    "2351a6011387dbc1ea56a8f9ba5fe83c"
    "1f1f28d8a9ebb7bd8c262c1f1f27fcfc"
)

CELL111_OUTPUT_RELATIVE_PATH = (
    Path("notebooks")
    /
    "phase2"
    /
    "author_review_28_pilots.md"
)

CELL111_CORPUS_RELATIVE_PATH = (
    Path("data")
    /
    "phase2"
    /
    "processed"
    /
    "documents_dedup_quality_accepted_v1.jsonl"
)

CELL111_SOURCE22_CANDIDATES = [
    Path(
        "notebooks/phase2/"
        "author_review_22_pilots_REVISED.md"
    ),
    Path(
        "notebooks/phase2/"
        "author_review_22_pilots.md"
    ),
    Path(
        "author_review_22_pilots_REVISED.md"
    ),
    Path(
        "author_review_22_pilots.md"
    ),
]


CELL111_EXCERPT_TERMS = {

    23: (
        "سواد مالی میزان پول مدیریت پول "
        "درست مدیریت کردن پول"
    ),

    24: (
        "پرسشنامه ارزیابی وضعیت نمره "
        "دانش نگرش مهارت سواد مالی"
    ),

    25: (
        "پول کجا می رود ثبت هزینه "
        "نوشتن هزینه ولخرجی مدیریت هزینه"
    ),

    26: (
        "الگوهای نادرست خرید احساس عاطفی "
        "تبلیغات بازاریابی خرید نامناسب"
    ),

    27: (
        "افزایش درآمد دارایی درآمدزا "
        "سرمایه گذاری سود کار درآمد"
    ),

    28: (
        "رابطه با پول چیزهایی که نداریم "
        "احساس نگرانی اعتماد پول"
    ),
}


CELL111_CASE27_FULL_PHASE1_MODEL_ANSWER = (
    "روش‌های افزایش درآمد عبارتند از:\n\n"
    "1. **قیمت‌گذاری مناسب محصولات**: افزایش سود با توجه به "
    "هزینه‌های تولید و ارزش ارائه شده به مشتری.\n"
    "2. **سرمایه‌گذاری درست**: انجام سرمایه‌گذاری‌هایی که حداقل "
    "به اندازۀ تورم سودآوری داشته باشند.\n"
    "3. **خرید دارایی‌های درآمدزا**: افزایش درآمد از طریق "
    "سرمایه‌گذاری در دارایی‌هایی که درآمد داشته باشند، نه فقط "
    "برای خرج کردن درآمد.\n"
    "4. **تغییر نقشه کسب‌وکار**: اگر درآمد مطلوبی حاصل نشود، "
    "باید روش‌های کسب‌وکار را بهبود بخشید."
)


# ============================================================
# 2) Utilities
# ============================================================

cell111_failures = []


def cell111_check(
    name,
    condition,
    details=None,
):

    if not bool(condition):

        failure = {
            "reason": name,
        }

        if details is not None:
            failure["details"] = details

        cell111_failures.append(
            failure
        )

    return bool(condition)


def cell111_sha256_text(text):

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def cell111_sha256_file(path):

    h = hashlib.sha256()

    with path.open("rb") as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):

            h.update(chunk)

    return h.hexdigest()


def cell111_find_repo_root():

    candidates = []

    for variable_name in [
        "cell110_registry_path",
        "cell109_registry_path",
        "cell108_registry_path",
        "cell105_registry_path",
    ]:

        if variable_name in globals():

            try:

                p = Path(
                    globals()[
                        variable_name
                    ]
                ).resolve()

                candidates.extend(
                    [
                        p.parent,
                        *p.parents,
                    ]
                )

            except Exception:
                pass

    cwd = Path.cwd().resolve()

    candidates.extend(
        [
            cwd,
            *cwd.parents,
        ]
    )

    seen = set()

    for candidate in candidates:

        candidate = (
            candidate.resolve()
        )

        if str(candidate) in seen:
            continue

        seen.add(
            str(candidate)
        )

        registry_candidate = (
            candidate
            /
            "data"
            /
            "phase2"
            /
            "human_review"
            /
            "human_review_registry.json"
        )

        corpus_candidate = (
            candidate
            /
            CELL111_CORPUS_RELATIVE_PATH
        )

        if (
            registry_candidate.exists()
            and
            corpus_candidate.exists()
        ):

            return candidate

    return None


PERSIAN_DIGITS_TO_ASCII = (
    str.maketrans(
        "۰۱۲۳۴۵۶۷۸۹",
        "0123456789",
    )
)


def cell111_to_int(value):

    return int(
        str(value).translate(
            PERSIAN_DIGITS_TO_ASCII
        )
    )


def cell111_md_table_cell(value):

    if value is None:
        return "—"

    value = str(value)

    value = value.replace(
        "\\",
        "\\\\",
    )

    value = value.replace(
        "|",
        "\\|",
    )

    value = value.replace(
        "\r\n",
        "\n",
    )

    value = value.replace(
        "\r",
        "\n",
    )

    value = value.replace(
        "\n",
        "<br>",
    )

    return value


def cell111_speaker_label(
    speaker,
    turn=None,
):

    mapping = {
        "HUMAN":
            "سؤال انسان",

        "PHASE1_MODEL":
            "پاسخ مدل فاز یک",
    }

    label = mapping.get(
        speaker,
        speaker,
    )

    if turn is not None:

        label += (
            " — نوبت "
            +
            str(turn)
        )

    return label


def cell111_normalize(text):

    text = (
        ""
        if text is None
        else str(text)
    )

    text = unicodedata.normalize(
        "NFKC",
        text,
    )

    replacements = {
        "ي": "ی",
        "ك": "ک",
        "ۀ": "ه",
        "ة": "ه",
        "ؤ": "و",
        "إ": "ا",
        "أ": "ا",
        "ٱ": "ا",
        "\u200c": " ",
        "\u200f": " ",
        "\u200e": " ",
    }

    for old, new in replacements.items():

        text = text.replace(
            old,
            new,
        )

    text = re.sub(
        r"[^\w\u0600-\u06FF]+",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text


CELL111_STOPWORDS = {
    "از", "به", "در", "با", "برای", "را",
    "که", "و", "یا", "این", "آن", "یک",
    "است", "هست", "هستم", "هستند", "بود",
    "شد", "شود", "می", "کرد", "کنیم",
    "چه", "چرا", "چگونه", "اگر", "اما",
    "هم", "روی", "خود", "ما", "من", "شما",
    "فرد", "افراد", "مورد", "درباره",
}


def cell111_tokens(text):

    return [
        token

        for token
        in cell111_normalize(
            text
        ).split()

        if (
            len(token) >= 2
            and
            token
            not in
            CELL111_STOPWORDS
        )
    ]


def cell111_extract_case_numbers(
    text,
):

    found = re.findall(
        r"مورد\s+([0-9۰-۹]+)\s+از\s+([0-9۰-۹]+)",
        text,
    )

    return [
        (
            cell111_to_int(
                case_number
            ),
            cell111_to_int(
                total_number
            ),
        )

        for case_number, total_number
        in found
    ]


# ============================================================
# 3) Source excerpt selector
# ============================================================

def cell111_best_source_excerpt(
    text,
    query_text,
    max_chars=900,
):

    text = str(
        text
        or
        ""
    ).strip()

    if not text:
        return ""

    query_tokens = set(
        cell111_tokens(
            query_text
        )
    )

    paragraphs = [
        block.strip()

        for block
        in re.split(
            r"\n\s*\n+",
            text,
        )

        if block.strip()
    ]

    sentences = [
        sentence.strip()

        for sentence
        in re.split(
            r"(?<=[\.\!\؟])\s+|\n+",
            text,
        )

        if sentence.strip()
    ]

    windows = []

    for index in range(
        len(sentences)
    ):

        window = " ".join(
            sentences[
                index:index + 3
            ]
        ).strip()

        if window:

            windows.append(
                window
            )

    candidates = (
        paragraphs
        +
        windows
    )

    if not candidates:

        candidates = [text]

    best_candidate = None
    best_score = -1

    for candidate in candidates:

        candidate_tokens = set(
            cell111_tokens(
                candidate
            )
        )

        score = len(
            query_tokens
            &
            candidate_tokens
        )

        if score > best_score:

            best_score = score
            best_candidate = (
                candidate
            )

        elif (
            score == best_score
            and
            best_candidate is not None
            and
            len(candidate)
            <
            len(best_candidate)
        ):

            best_candidate = (
                candidate
            )

    if best_candidate is None:

        best_candidate = text

    if len(
        best_candidate
    ) <= max_chars:

        return best_candidate

    position = None

    for token in cell111_tokens(
        query_text
    ):

        found = (
            best_candidate.find(
                token
            )
        )

        if found >= 0:

            position = found
            break

    if position is None:

        position = 0

    start = max(
        0,
        position - 220,
    )

    end = min(
        len(
            best_candidate
        ),
        start + max_chars,
    )

    excerpt = (
        best_candidate[
            start:end
        ].strip()
    )

    if start > 0:

        excerpt = (
            "... "
            +
            excerpt
        )

    if end < len(
        best_candidate
    ):

        excerpt = (
            excerpt
            +
            " ..."
        )

    return excerpt


# ============================================================
# 4) Resolve repo
# ============================================================

cell111_repo_root = (
    cell111_find_repo_root()
)


cell111_check(
    "REPO_ROOT_NOT_FOUND",
    cell111_repo_root
    is not None,
)


if cell111_failures:

    raise AssertionError(
        cell111_failures
    )


cell111_registry_path = (
    cell111_repo_root
    /
    "data"
    /
    "phase2"
    /
    "human_review"
    /
    "human_review_registry.json"
)


cell111_corpus_path = (
    cell111_repo_root
    /
    CELL111_CORPUS_RELATIVE_PATH
)


cell111_output_path = (
    cell111_repo_root
    /
    CELL111_OUTPUT_RELATIVE_PATH
)


cell111_output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


cell111_check(
    "REGISTRY_NOT_FOUND",
    cell111_registry_path.exists(),
)


cell111_check(
    "CORPUS_NOT_FOUND",
    cell111_corpus_path.exists(),
)


if cell111_failures:

    raise AssertionError(
        cell111_failures
    )


# ============================================================
# 5) Verify frozen Corpus
# ============================================================

cell111_corpus_sha256 = (
    cell111_sha256_file(
        cell111_corpus_path
    )
)


cell111_check(
    "CORPUS_SHA256_MISMATCH",
    cell111_corpus_sha256
    ==
    CELL111_EXPECTED_CORPUS_SHA256,
    {
        "expected":
            CELL111_EXPECTED_CORPUS_SHA256,

        "actual":
            cell111_corpus_sha256,
    },
)


if cell111_failures:

    raise AssertionError(
        "Cell 111 stopped Fail-Closed: "
        "Accepted Corpus identity mismatch."
    )


# ============================================================
# 6) Read Registry — READ ONLY
# ============================================================

cell111_registry_sha256_before = (
    cell111_sha256_file(
        cell111_registry_path
    )
)


cell111_registry_text = (
    cell111_registry_path.read_text(
        encoding="utf-8"
    )
)


cell111_registry = json.loads(
    cell111_registry_text
)


cell111_author_review = (
    cell111_registry.get(
        "author_review",
        {}
    )
    or
    {}
)


cell111_author_cases = (
    cell111_author_review.get(
        "cases_ready_for_review",
        []
    )
    or
    []
)


cell111_second_review = (
    cell111_registry.get(
        "second_human_review",
        {}
    )
    or
    {}
)


cell111_feedback_batch = (
    cell111_second_review.get(
        CELL111_BATCH_KEY,
        {}
    )
    or
    {}
)


cell111_feedback_cases = (
    cell111_feedback_batch.get(
        "cases",
        []
    )
    or
    []
)


# ============================================================
# 7) Validate current Registry state
# ============================================================

cell111_check(
    "AUTHOR_QUEUE_NOT_28",
    len(
        cell111_author_cases
    )
    ==
    28,
    len(
        cell111_author_cases
    ),
)


cell111_check(
    "FEEDBACK_BATCH_NOT_6",
    len(
        cell111_feedback_cases
    )
    ==
    6,
    len(
        cell111_feedback_cases
    ),
)


cell111_new_author_cases = (
    cell111_author_cases[22:]
)


cell111_check(
    "NEW_AUTHOR_CASES_NOT_6",
    len(
        cell111_new_author_cases
    )
    ==
    6,
)


cell111_author_by_number = {
    case.get(
        "reference_pilot_number"
    ):
        case

    for case
    in cell111_new_author_cases
}


cell111_batch_by_number = {
    case.get(
        "reference_pilot_number"
    ):
        case

    for case
    in cell111_feedback_cases
}


cell111_check(
    "NEW_REFERENCE_SEQUENCE_MISMATCH",
    [
        case.get(
            "reference_pilot_number"
        )

        for case
        in cell111_new_author_cases
    ]
    ==
    CELL111_NEW_NUMBERS,
)


for number in (
    CELL111_NEW_NUMBERS
):

    author_case = (
        cell111_author_by_number.get(
            number
        )
    )

    batch_case = (
        cell111_batch_by_number.get(
            number
        )
    )

    cell111_check(
        f"AUTHOR_CASE_MISSING:{number}",
        author_case
        is not None,
    )

    cell111_check(
        f"BATCH_CASE_MISSING:{number}",
        batch_case
        is not None,
    )

    if (
        author_case is None
        or
        batch_case is None
    ):

        continue

    cell111_check(
        f"PILOT_ID_COPY_MISMATCH:{number}",
        author_case.get(
            "pilot_case_id"
        )
        ==
        batch_case.get(
            "pilot_case_id"
        ),
    )

    cell111_check(
        f"ACTIVE_TYPE_WRONG:{number}",
        author_case.get(
            "pilot_type"
        )
        ==
        CELL111_EXPECTED_ACTIVE_TYPE,
        author_case.get(
            "pilot_type"
        ),
    )

    cell111_check(
        f"GOLDEN_CONTENT_FALSE:{number}",
        author_case.get(
            "golden_content_case"
        )
        is True,
    )

    review = (
        author_case.get(
            "second_source_review",
            {}
        )
        or
        {}
    )

    cell111_check(
        f"SOURCE_REVIEW_NOT_COMPLETE:{number}",
        review.get(
            "status"
        )
        ==
        "COMPLETE",
    )

    cell111_check(
        f"DECISION_WRONG:{number}",
        review.get(
            "decision"
        )
        ==
        CELL111_EXPECTED_DECISIONS[
            number
        ],
        review.get(
            "decision"
        ),
    )

    cell111_check(
        f"REVISION_SCOPE_WRONG:{number}",
        review.get(
            "revision_scope"
        )
        ==
        CELL111_EXPECTED_REVISION_SCOPES[
            number
        ],
        review.get(
            "revision_scope"
        ),
    )

    cell111_check(
        f"NO_VERIFIED_SOURCE:{number}",
        len(
            review.get(
                "verified_corpus_sources",
                []
            )
            or
            []
        )
        >=
        1,
    )

    cell111_check(
        f"AUTHOR_REVIEW_ALREADY_STARTED:{number}",
        (
            author_case.get(
                "author_final_review",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "NOT_STARTED",
    )

    for field in [
        "pilot_type",
        "golden_content_case",
        "derived_question",
        "expected_answer_review",
        "second_source_review",
        "citation_review",
    ]:

        cell111_check(
            (
                f"REGISTRY_COPY_MISMATCH:"
                f"{number}:"
                f"{field}"
            ),
            author_case.get(
                field
            )
            ==
            batch_case.get(
                field
            ),
        )


if cell111_failures:

    print(
        "\n=== REGISTRY VALIDATION FAILURES ==="
    )

    print(
        cell111_failures
    )

    raise AssertionError(
        "Cell 111 blocked by Registry state."
    )


# ============================================================
# 8) Validate Pilot 25 superseded history
# ============================================================

cell111_case25 = (
    cell111_author_by_number[
        25
    ]
)


cell111_case25_current_review = (
    cell111_case25.get(
        "second_source_review",
        {}
    )
    or
    {}
)


cell111_case25_event_id = (
    cell111_case25_current_review.get(
        "review_event_id"
    )
)


cell111_case25_event = None


for event in (
    cell111_case25.get(
        "source_review_history",
        []
    )
    or
    []
):

    if (
        isinstance(
            event,
            dict,
        )
        and
        event.get(
            "review_event_id"
        )
        ==
        cell111_case25_event_id
    ):

        cell111_case25_event = (
            event
        )

        break


cell111_check(
    "PILOT25_REVIEW_EVENT_MISSING",
    cell111_case25_event
    is not None,
)


cell111_case25_old_state = (
    (
        cell111_case25_event
        or
        {}
    ).get(
        "superseded_state"
    )
    or
    {}
)


cell111_check(
    "PILOT25_SUPERSEDED_HISTORY_MISSING",
    bool(
        cell111_case25_old_state
    ),
)


cell111_check(
    "PILOT25_PREVIOUS_TYPE_WRONG",
    cell111_case25_old_state.get(
        "previous_pilot_type"
    )
    ==
    "MULTI_TURN_CONTEXT_CONTINUITY",
)


if cell111_failures:

    raise AssertionError(
        cell111_failures
    )


# ============================================================
# 9) Validate Pilot 27 full historical model answer
# ============================================================

cell111_case27 = (
    cell111_author_by_number[
        27
    ]
)


cell111_case27_model_texts = [
    entry.get(
        "text"
    )

    for entry
    in (
        (
            cell111_case27.get(
                "historical_phase1_demo",
                {}
            )
            or
            {}
        ).get(
            "raw_conversation",
            []
        )
        or
        []
    )

    if (
        isinstance(
            entry,
            dict,
        )
        and
        entry.get(
            "speaker"
        )
        ==
        "PHASE1_MODEL"
    )
]


cell111_check(
    "CASE27_COMPLETE_MODEL_ANSWER_MISSING",
    CELL111_CASE27_FULL_PHASE1_MODEL_ANSWER
    in
    cell111_case27_model_texts,
)


if cell111_failures:

    raise AssertionError(
        cell111_failures
    )


# ============================================================
# 10) Resolve frozen approved 22-case Markdown
# ============================================================

cell111_source22_path = None


for relative_path in (
    CELL111_SOURCE22_CANDIDATES
):

    candidate = (
        cell111_repo_root
        /
        relative_path
    )

    if not candidate.exists():

        continue

    candidate_text = (
        candidate.read_text(
            encoding="utf-8"
        )
    )

    candidate_sha = (
        cell111_sha256_text(
            candidate_text
        )
    )

    if (
        candidate_sha
        ==
        CELL111_EXPECTED_SOURCE22_SHA256
    ):

        cell111_source22_path = (
            candidate
        )

        break


cell111_check(
    "APPROVED_22_MARKDOWN_NOT_FOUND",
    cell111_source22_path
    is not None,
)


if cell111_failures:

    raise AssertionError(
        "Cell 111 stopped Fail-Closed: "
        "exact approved 22-case Markdown not found."
    )


cell111_source22_text = (
    cell111_source22_path.read_text(
        encoding="utf-8"
    )
)


cell111_source22_sha256 = (
    cell111_sha256_text(
        cell111_source22_text
    )
)


cell111_check(
    "APPROVED_22_SHA_MISMATCH",
    cell111_source22_sha256
    ==
    CELL111_EXPECTED_SOURCE22_SHA256,
)


# Frozen source must contain exactly sequence 1-22.
cell111_source22_markers = (
    cell111_extract_case_numbers(
        cell111_source22_text
    )
)


cell111_source22_numbers = [
    case_number

    for case_number, total
    in cell111_source22_markers

    if total == 22
]


cell111_check(
    "SOURCE22_SEQUENCE_NOT_1_TO_22",
    cell111_source22_numbers
    ==
    list(
        range(
            1,
            23,
        )
    ),
    cell111_source22_numbers,
)


if cell111_failures:

    raise AssertionError(
        cell111_failures
    )


# ============================================================
# 11) Convert only structural count labels 22 -> 28
# ============================================================

cell111_base28_text = (
    cell111_source22_text
)


cell111_base28_text = re.sub(
    r"(مورد\s+[0-9۰-۹]+\s+از\s+)(?:۲۲|22)",
    r"\g<1>۲۸",
    cell111_base28_text,
)


# Only document-level common labels.
# We intentionally do NOT globally replace every "22".
cell111_lines = (
    cell111_base28_text.splitlines()
)


cell111_seen_case = False
cell111_rebuilt_lines = []


for line in cell111_lines:

    if re.search(
        r"مورد\s+[0-9۰-۹]+\s+از\s+۲۸",
        line,
    ):

        cell111_seen_case = True

    if not cell111_seen_case:

        line = (
            line
            .replace(
                "۲۲ پایلوت",
                "۲۸ پایلوت",
            )
            .replace(
                "۲۲ مورد",
                "۲۸ مورد",
            )
            .replace(
                "22 pilots",
                "28 pilots",
            )
            .replace(
                "22 cases",
                "28 cases",
            )
        )

    cell111_rebuilt_lines.append(
        line
    )


cell111_base28_text = "\n".join(
    cell111_rebuilt_lines
)


# ============================================================
# 12) Load required verified Corpus documents
# ============================================================

cell111_required_document_ids = set()


for number in (
    CELL111_NEW_NUMBERS
):

    review = (
        cell111_author_by_number[
            number
        ].get(
            "second_source_review",
            {}
        )
        or
        {}
    )

    for source in (
        review.get(
            "verified_corpus_sources",
            []
        )
        or
        []
    ):

        document_id = (
            source.get(
                "document_id"
            )
        )

        if document_id:

            cell111_required_document_ids.add(
                document_id
            )


cell111_source_documents = {}


with cell111_corpus_path.open(
    "r",
    encoding="utf-8",
) as f:

    for line_number, line in enumerate(
        f,
        start=1,
    ):

        line = line.strip()

        if not line:

            continue

        try:

            record = json.loads(
                line
            )

        except Exception as exc:

            cell111_check(
                "CORPUS_JSON_PARSE_FAILURE",
                False,
                {
                    "line":
                        line_number,

                    "error":
                        str(exc),
                },
            )

            break

        document_id = (
            record.get(
                "document_id"
            )
        )

        if (
            document_id
            in
            cell111_required_document_ids
        ):

            cell111_source_documents[
                document_id
            ] = record


cell111_check(
    "VERIFIED_DOCUMENTS_MISSING",
    set(
        cell111_source_documents.keys()
    )
    ==
    cell111_required_document_ids,
    {
        "required":
            sorted(
                cell111_required_document_ids
            ),

        "found":
            sorted(
                cell111_source_documents.keys()
            ),
    },
)


if cell111_failures:

    raise AssertionError(
        cell111_failures
    )


# ============================================================
# 13) Grounding explanations
# ============================================================

def cell111_grounding_explanation(
    number,
):

    if number == 23:

        return (
            "بازخورد مستقیم مؤلف با Source واقعی Direct Author "
            "از Corpus پشتیبانی مستقیم و صریح دارد."
        )

    if number == 24:

        return (
            "Grounding ترکیبی است: یک Source بخش پرسشنامه و "
            "ارزیابی وضعیت و Source دیگر ابعاد دانش، نگرش و "
            "مهارت مالی را پشتیبانی می‌کند."
        )

    if number == 25:

        return (
            "برداشت قبلی تداوم Context کنار گذاشته شد. "
            "مورد فعال اکنون یک سؤال محتوایی است و سؤال و "
            "پاسخ فعلی با Sourceهای Corpus درباره ثبت و "
            "مدیریت هزینه‌ها Ground شده‌اند."
        )

    if number == 26:

        return (
            "فهرست هفت‌گانه عمدتاً بر بازخورد مستقیم دکتر "
            "استوار است. Sourceهای Website فقط بخشی از مفهوم "
            "الگوهای خرید نادرست، عوامل عاطفی و اثر تبلیغات "
            "و بازاریابی را تأیید می‌کنند."
        )

    if number == 27:

        return (
            "چهار روش و ترتیب آن‌ها از بازخورد مستقیم دکتر "
            "می‌آید. Sourceهای Website فقط بخشی از مفاهیم "
            "افزایش درآمد، دارایی درآمدزا، سرمایه‌گذاری و "
            "سود را تأیید می‌کنند."
        )

    if number == 28:

        return (
            "Source Direct Author بخش اصلی رابطه فرد با پول "
            "و تلاش برای جبران چیزهایی که فرد احساس می‌کند "
            "ندارد را پشتیبانی می‌کند. تفسیر گسترده‌تر احساس "
            "فقر از بازخورد مستقیم دکتر حفظ شده است."
        )

    return "—"


# ============================================================
# 14) Build one new case
# ============================================================

def cell111_build_new_case(
    case,
):

    number = (
        case[
            "reference_pilot_number"
        ]
    )

    history = (
        case.get(
            "historical_phase1_demo",
            {}
        )
        or
        {}
    )

    conversation = (
        history.get(
            "raw_conversation",
            []
        )
        or
        []
    )

    raw_feedback = (
        history.get(
            "raw_author_feedback"
        )
        or
        ""
    )

    question = (
        (
            case.get(
                "derived_question",
                {}
            )
            or
            {}
        ).get(
            "question_text"
        )
        or
        ""
    )

    answer = (
        (
            case.get(
                "expected_answer_review",
                {}
            )
            or
            {}
        ).get(
            "expected_answer_text"
        )
        or
        ""
    )

    review = (
        case.get(
            "second_source_review",
            {}
        )
        or
        {}
    )

    decision = (
        review.get(
            "decision"
        )
        or
        ""
    )

    revision_scope = (
        review.get(
            "revision_scope"
        )
        or
        ""
    )

    verified_sources = (
        review.get(
            "verified_corpus_sources",
            []
        )
        or
        []
    )

    lines = [
        "",
        "",
        "---",
        "",
        f"## مورد {number} از ۲۸",
        "",
        "### متن منبع (بخشی از متن اصلی)",
        "",
        "#### سابقه واقعی دموی فاز یک",
        "",
    ]

    # --------------------------------------------------------
    # Phase-1 history
    # --------------------------------------------------------

    for entry in conversation:

        speaker = (
            entry.get(
                "speaker"
            )
        )

        turn = (
            entry.get(
                "turn"
            )
        )

        text = (
            entry.get(
                "text"
            )
            or
            ""
        )

        lines.extend([
            (
                "**"
                +
                cell111_speaker_label(
                    speaker,
                    turn,
                )
                +
                ":**"
            ),
            "",
            text,
            "",
        ])

    lines.extend([
        "**بازخورد مستقیم دکتر در فاز یک:**",
        "",
        raw_feedback,
        "",
    ])

    # --------------------------------------------------------
    # Pilot 25 old interpretation history
    # --------------------------------------------------------

    if number == 25:

        old_state = (
            cell111_case25_old_state
        )

        lines.extend([
            "#### وضعیت قبلی که پس از Source Review کنار گذاشته شد",
            "",
            "**نوع قبلی:**",
            "",
            (
                "`"
                +
                str(
                    old_state.get(
                        "previous_pilot_type"
                    )
                    or
                    "—"
                )
                +
                "`"
            ),
            "",
            "**سؤال قبلی:**",
            "",
            (
                old_state.get(
                    "previous_question"
                )
                or
                "—"
            ),
            "",
            "**رفتار مورد انتظار قبلی:**",
            "",
            (
                old_state.get(
                    "previous_expected_behavior"
                )
                or
                "—"
            ),
            "",
            "**دلیل کنار گذاشتن:**",
            "",
            (
                "Turnهای دموی فاز یک Context مشترک قابل اتکایی "
                "نداشتند. بنابراین برداشت قبلی به‌عنوان تست فعال "
                "تداوم Context کنار گذاشته شد و فقط به‌عنوان "
                "History حفظ شده است."
            ),
            "",
        ])

    # --------------------------------------------------------
    # Corpus evidence
    # --------------------------------------------------------

    lines.extend([
        "#### شواهد Corpus تأییدشده در Source Review",
        "",
    ])

    for source_index, source in enumerate(
        verified_sources,
        start=1,
    ):

        document_id = (
            source[
                "document_id"
            ]
        )

        record = (
            cell111_source_documents[
                document_id
            ]
        )

        title = (
            source.get(
                "title"
            )
            or
            record.get(
                "title"
            )
            or
            "بدون عنوان"
        )

        role = (
            source.get(
                "role"
            )
            or
            "—"
        )

        coverage = (
            source.get(
                "coverage"
            )
            or
            "—"
        )

        excerpt = (
            cell111_best_source_excerpt(
                record.get(
                    "text",
                    "",
                ),
                CELL111_EXCERPT_TERMS[
                    number
                ],
            )
        )

        lines.extend([
            (
                f"**منبع Corpus {source_index}: "
                f"{title}**"
            ),
            "",
            (
                "**نقش در Grounding:** "
                f"`{role}`"
            ),
            "",
            (
                "**میزان پوشش:** "
                f"`{coverage}`"
            ),
            "",
            "**بخشی از متن منبع:**",
            "",
            excerpt,
            "",
        ])

    lines.extend([
        "**جمع‌بندی Grounding:**",
        "",
        cell111_grounding_explanation(
            number
        ),
        "",
    ])

    # --------------------------------------------------------
    # Revision history
    # --------------------------------------------------------

    lines.extend([
        "### سؤال‌ها و پاسخ‌ها — روند اصلاح و بازبینی",
        "",
        "| مرحله | متن سؤال | متن پاسخ | توضیحات / دلیل تغییر |",
        "|---|---|---|---|",
    ])

    human_entries = [
        entry

        for entry
        in conversation

        if (
            isinstance(
                entry,
                dict,
            )
            and
            entry.get(
                "speaker"
            )
            ==
            "HUMAN"
        )
    ]

    model_entries = [
        entry

        for entry
        in conversation

        if (
            isinstance(
                entry,
                dict,
            )
            and
            entry.get(
                "speaker"
            )
            ==
            "PHASE1_MODEL"
        )
    ]

    pair_count = max(
        len(
            human_entries
        ),
        len(
            model_entries
        ),
    )

    for index in range(
        pair_count
    ):

        human_text = (
            human_entries[index].get(
                "text"
            )
            if index
            <
            len(
                human_entries
            )
            else
            "—"
        )

        model_text = (
            model_entries[index].get(
                "text"
            )
            if index
            <
            len(
                model_entries
            )
            else
            "—"
        )

        stage = (
            "دموی فاز یک"
            if index == 0
            else
            (
                "دموی فاز یک — نوبت "
                +
                str(
                    index + 1
                )
            )
        )

        lines.append(
            "| "
            +
            cell111_md_table_cell(
                stage
            )
            +
            " | "
            +
            cell111_md_table_cell(
                human_text
            )
            +
            " | "
            +
            cell111_md_table_cell(
                model_text
            )
            +
            " | "
            +
            cell111_md_table_cell(
                (
                    "پاسخ تاریخی مدل فاز یک است و "
                    "Ground Truth محسوب نمی‌شود."
                )
            )
            +
            " |"
        )

    if number == 25:

        lines.append(
            "| "
            +
            cell111_md_table_cell(
                "برداشت اولیه پیش از Source Review"
            )
            +
            " | "
            +
            cell111_md_table_cell(
                cell111_case25_old_state.get(
                    "previous_question"
                )
                or
                "—"
            )
            +
            " | "
            +
            cell111_md_table_cell(
                cell111_case25_old_state.get(
                    "previous_expected_behavior"
                )
                or
                "—"
            )
            +
            " | "
            +
            cell111_md_table_cell(
                (
                    "این برداشت قبلی، مورد را به‌عنوان "
                    "MULTI_TURN_CONTEXT_CONTINUITY در نظر "
                    "می‌گرفت و پس از Source Review کنار "
                    "گذاشته شد."
                )
            )
            +
            " |"
        )

    if number == 25:

        final_change_note = (
            "سؤال و پاسخ بازطراحی شد و Pilot فعال به "
            "AUTHOR_FEEDBACK_CONTENT تبدیل شد."
        )

    elif revision_scope == "GROUNDING_ONLY":

        final_change_note = (
            "متن سؤال و پاسخ حفظ شد؛ اصلاح فقط در Grounding "
            "و تفکیک بازخورد مستقیم مؤلف از پشتیبانی جزئی "
            "Corpus انجام شد."
        )

    else:

        final_change_note = (
            "سؤال و پاسخ پس از تطبیق با Source واقعی Corpus "
            "پذیرفته شد."
        )

    lines.append(
        "| "
        +
        cell111_md_table_cell(
            "Second Human Source Review"
        )
        +
        " | "
        +
        cell111_md_table_cell(
            question
        )
        +
        " | "
        +
        cell111_md_table_cell(
            answer
        )
        +
        " | "
        +
        cell111_md_table_cell(
            final_change_note
        )
        +
        " |"
    )

    # --------------------------------------------------------
    # Decision trail
    # --------------------------------------------------------

    lines.extend([
        "",
        "### تصمیم من در روند بازبینی",
        "",
        "| مرحله | تصمیم | توضیحات |",
        "|---|---|---|",
        (
            "| بازخورد فاز یک | بازخورد مستقیم مؤلف | "
            "متن بازخورد دکتر بدون بازنویسی در History حفظ شده است. |"
        ),
        (
            "| Source Mapping | VERIFIED | "
            +
            cell111_md_table_cell(
                (
                    str(
                        len(
                            verified_sources
                        )
                    )
                    +
                    " Source Direct Author از Accepted Corpus "
                    "برای این مورد بررسی شد."
                )
            )
            +
            " |"
        ),
        (
            "| Second Human Source Review | "
            +
            cell111_md_table_cell(
                decision
            )
            +
            " | "
            +
            cell111_md_table_cell(
                (
                    "Revision Scope: "
                    +
                    revision_scope
                    +
                    " — "
                    +
                    cell111_grounding_explanation(
                        number
                    )
                )
            )
            +
            " |"
        ),
        (
            "| وضعیت فعلی | آماده برای بازبینی مؤلف | "
            "هیچ تصمیم نهایی به نام دکتر ثبت نشده است. |"
        ),
        "",
    ])

    # --------------------------------------------------------
    # Final Q/A
    # --------------------------------------------------------

    lines.extend([
        "### سؤال و پاسخ نهایی (مورد بازبینی توسط دکتر)",
        "",
        "**سؤال:**",
        "",
        question,
        "",
        "**پاسخ نهایی پیشنهادی:**",
        "",
        answer,
        "",
    ])

    # --------------------------------------------------------
    # Blank Author Review
    # --------------------------------------------------------

    lines.extend([
        "### نظر مؤلف (دکتر کمیل رودی)",
        "",
        "- ☐ تأیید",
        "- ☐ نیاز به اصلاح",
        "- ☐ رد",
        "- ☐ باید دوباره سؤال ساخته شود",
        "",
        "**توضیحات دکتر:**",
        "",
        "<br>",
        "<br>",
        "<br>",
        "<br>",
        "<br>",
        "<br>",
    ])

    return "\n".join(
        lines
    )


# ============================================================
# 15) Build new six
# ============================================================

cell111_new_blocks = [
    cell111_build_new_case(
        cell111_author_by_number[
            number
        ]
    )

    for number
    in CELL111_NEW_NUMBERS
]


cell111_new_cases_text = "\n".join(
    cell111_new_blocks
)


# ============================================================
# 16) Build final 28-case document
# ============================================================

cell111_full_markdown = (
    cell111_base28_text.rstrip()
    +
    "\n"
    +
    cell111_new_cases_text
    +
    "\n"
)


# ============================================================
# 17) Validate all 28 case markers
#
# IMPORTANT:
# No Markdown-heading assumption.
# We only use the structural text:
#
#   مورد X از ۲۸
#
# This is compatible with the frozen 22-case file.
# ============================================================

cell111_all_markers = (
    cell111_extract_case_numbers(
        cell111_full_markdown
    )
)


cell111_all_case_numbers = [
    case_number

    for case_number, total
    in cell111_all_markers

    if total == 28
]


cell111_check(
    "FINAL_CASE_SEQUENCE_NOT_1_TO_28",
    cell111_all_case_numbers
    ==
    list(
        range(
            1,
            29,
        )
    ),
    cell111_all_case_numbers,
)


cell111_check(
    "FINAL_CASE_COUNT_NOT_28",
    len(
        cell111_all_case_numbers
    )
    ==
    28,
)


# ============================================================
# 18) Validate ONLY new 6 generated blocks structurally
#
# Cases 1-22 are already frozen by exact SHA.
# We do not reinterpret their formatting.
# ============================================================

for number, block in zip(
    CELL111_NEW_NUMBERS,
    cell111_new_blocks,
):

    required_markers = [
        f"مورد {number} از ۲۸",
        "### متن منبع (بخشی از متن اصلی)",
        "#### سابقه واقعی دموی فاز یک",
        "#### شواهد Corpus تأییدشده در Source Review",
        "### سؤال‌ها و پاسخ‌ها — روند اصلاح و بازبینی",
        "### تصمیم من در روند بازبینی",
        "### سؤال و پاسخ نهایی (مورد بازبینی توسط دکتر)",
        "### نظر مؤلف (دکتر کمیل رودی)",
        "☐ تأیید",
        "☐ نیاز به اصلاح",
        "☐ رد",
        "☐ باید دوباره سؤال ساخته شود",
        "توضیحات دکتر",
    ]

    missing = [
        marker

        for marker
        in required_markers

        if marker
        not in
        block
    ]

    cell111_check(
        f"NEW_CASE_STRUCTURE_MISSING:{number}",
        len(
            missing
        )
        ==
        0,
        missing,
    )


# ============================================================
# 19) Validate exact content for Pilots 23-28
# ============================================================

for number in (
    CELL111_NEW_NUMBERS
):

    case = (
        cell111_author_by_number[
            number
        ]
    )

    question = (
        (
            case.get(
                "derived_question",
                {}
            )
            or
            {}
        ).get(
            "question_text"
        )
        or
        ""
    )

    answer = (
        (
            case.get(
                "expected_answer_review",
                {}
            )
            or
            {}
        ).get(
            "expected_answer_text"
        )
        or
        ""
    )

    raw_feedback = (
        (
            case.get(
                "historical_phase1_demo",
                {}
            )
            or
            {}
        ).get(
            "raw_author_feedback"
        )
        or
        ""
    )

    review = (
        case.get(
            "second_source_review",
            {}
        )
        or
        {}
    )

    block = (
        cell111_new_blocks[
            number - 23
        ]
    )

    cell111_check(
        f"QUESTION_MISSING:{number}",
        question
        in
        block,
    )

    cell111_check(
        f"ANSWER_MISSING:{number}",
        answer
        in
        block,
    )

    cell111_check(
        f"RAW_FEEDBACK_MISSING:{number}",
        raw_feedback
        in
        block,
    )

    cell111_check(
        f"SOURCE_DECISION_MISSING:{number}",
        review.get(
            "decision"
        )
        in
        block,
    )

    for source in (
        review.get(
            "verified_corpus_sources",
            []
        )
        or
        []
    ):

        title = (
            source.get(
                "title"
            )
            or
            ""
        )

        cell111_check(
            (
                f"SOURCE_TITLE_MISSING:"
                f"{number}:"
                f"{source.get('document_id')}"
            ),
            bool(
                title
            )
            and
            title
            in
            block,
        )

    for entry in (
        (
            case.get(
                "historical_phase1_demo",
                {}
            )
            or
            {}
        ).get(
            "raw_conversation",
            []
        )
        or
        []
    ):

        raw_text = (
            entry.get(
                "text"
            )
            or
            ""
        )

        cell111_check(
            (
                f"RAW_HISTORY_MISSING:"
                f"{number}:"
                f"{entry.get('speaker')}:"
                f"{entry.get('turn', 'NA')}"
            ),
            raw_text
            in
            block,
        )


# ============================================================
# 20) Pilot 25 special validation
# ============================================================

cell111_case25_block = (
    cell111_new_blocks[
        2
    ]
)


cell111_case25_current_question = (
    (
        cell111_case25.get(
            "derived_question",
            {}
        )
        or
        {}
    ).get(
        "question_text"
    )
    or
    ""
)


cell111_case25_current_answer = (
    (
        cell111_case25.get(
            "expected_answer_review",
            {}
        )
        or
        {}
    ).get(
        "expected_answer_text"
    )
    or
    ""
)


cell111_case25_old_question = (
    cell111_case25_old_state.get(
        "previous_question"
    )
    or
    ""
)


cell111_check(
    "PILOT25_CURRENT_QUESTION_MISSING",
    cell111_case25_current_question
    in
    cell111_case25_block,
)


cell111_check(
    "PILOT25_CURRENT_ANSWER_MISSING",
    cell111_case25_current_answer
    in
    cell111_case25_block,
)


cell111_check(
    "PILOT25_OLD_QUESTION_HISTORY_MISSING",
    bool(
        cell111_case25_old_question
    )
    and
    cell111_case25_old_question
    in
    cell111_case25_block,
)


cell111_check(
    "PILOT25_OLD_TYPE_HISTORY_MISSING",
    "MULTI_TURN_CONTEXT_CONTINUITY"
    in
    cell111_case25_block,
)


# ============================================================
# 21) Pilot 27 exact history validation
# ============================================================

cell111_case27_block = (
    cell111_new_blocks[
        4
    ]
)


cell111_check(
    "CASE27_FULL_MODEL_ANSWER_NOT_IN_OUTPUT",
    CELL111_CASE27_FULL_PHASE1_MODEL_ANSWER
    in
    cell111_case27_block,
)


# ============================================================
# 22) Active-type counts
# ============================================================

cell111_active_content_count = sum(
    1

    for case
    in cell111_new_author_cases

    if (
        case.get(
            "pilot_type"
        )
        ==
        "AUTHOR_FEEDBACK_CONTENT"
        and
        case.get(
            "golden_content_case"
        )
        is True
    )
)


cell111_active_continuity_count = sum(
    1

    for case
    in cell111_new_author_cases

    if (
        case.get(
            "pilot_type"
        )
        ==
        "MULTI_TURN_CONTEXT_CONTINUITY"
    )
)


cell111_check(
    "ACTIVE_CONTENT_COUNT_NOT_6",
    cell111_active_content_count
    ==
    6,
)


cell111_check(
    "ACTIVE_CONTINUITY_COUNT_NOT_0",
    cell111_active_continuity_count
    ==
    0,
)


# ============================================================
# 23) Registry must still be unchanged before output write
# ============================================================

cell111_registry_sha256_prewrite = (
    cell111_sha256_file(
        cell111_registry_path
    )
)


cell111_check(
    "REGISTRY_CHANGED_BEFORE_OUTPUT_WRITE",
    cell111_registry_sha256_prewrite
    ==
    cell111_registry_sha256_before,
    {
        "before":
            cell111_registry_sha256_before,

        "current":
            cell111_registry_sha256_prewrite,
    },
)


if cell111_failures:

    print(
        "\n=== PRE-WRITE VALIDATION FAILURES ==="
    )

    print(
        cell111_failures
    )

    raise AssertionError(
        "Cell 111 stopped before Markdown write."
    )


# ============================================================
# 24) Safe output write
# ============================================================

cell111_existing_output = None


if cell111_output_path.exists():

    cell111_existing_output = (
        cell111_output_path.read_text(
            encoding="utf-8"
        )
    )


cell111_output_change_required = (
    cell111_existing_output
    !=
    cell111_full_markdown
)


cell111_output_backup_path = None


if cell111_output_change_required:

    if cell111_output_path.exists():

        timestamp = (
            datetime.now(
                timezone.utc
            ).strftime(
                "%Y%m%dT%H%M%SZ"
            )
        )

        cell111_output_backup_path = (
            cell111_output_path.parent
            /
            (
                cell111_output_path.stem
                +
                ".before_cell111_"
                +
                timestamp
                +
                cell111_output_path.suffix
            )
        )

        cell111_output_backup_path.write_text(
            cell111_existing_output,
            encoding="utf-8",
        )

    cell111_temp_path = (
        cell111_output_path.parent
        /
        (
            cell111_output_path.name
            +
            ".cell111.tmp"
        )
    )

    cell111_temp_path.write_text(
        cell111_full_markdown,
        encoding="utf-8",
    )

    cell111_temp_readback = (
        cell111_temp_path.read_text(
            encoding="utf-8"
        )
    )

    cell111_check(
        "TEMP_OUTPUT_READBACK_MISMATCH",
        cell111_temp_readback
        ==
        cell111_full_markdown,
    )

    if cell111_failures:

        raise AssertionError(
            cell111_failures
        )

    cell111_temp_path.replace(
        cell111_output_path
    )


# ============================================================
# 25) Reload final output
# ============================================================

cell111_final_text = (
    cell111_output_path.read_text(
        encoding="utf-8"
    )
)


cell111_final_sha256 = (
    cell111_sha256_text(
        cell111_final_text
    )
)


cell111_check(
    "FINAL_OUTPUT_CONTENT_MISMATCH",
    cell111_final_text
    ==
    cell111_full_markdown,
)


cell111_final_markers = (
    cell111_extract_case_numbers(
        cell111_final_text
    )
)


cell111_final_numbers = [
    case_number

    for case_number, total
    in cell111_final_markers

    if total == 28
]


cell111_check(
    "FINAL_PERSISTED_SEQUENCE_NOT_1_TO_28",
    cell111_final_numbers
    ==
    list(
        range(
            1,
            29,
        )
    ),
    cell111_final_numbers,
)


# ============================================================
# 26) Final new-case Author Review validation
# ============================================================

for number in (
    CELL111_NEW_NUMBERS
):

    marker = (
        f"## مورد {number} از ۲۸"
    )

    start = (
        cell111_final_text.find(
            marker
        )
    )

    cell111_check(
        f"FINAL_NEW_CASE_NOT_FOUND:{number}",
        start >= 0,
    )

    if start < 0:

        continue

    if number < 28:

        next_marker = (
            f"## مورد {number + 1} از ۲۸"
        )

        end = (
            cell111_final_text.find(
                next_marker,
                start + 1,
            )
        )

    else:

        end = len(
            cell111_final_text
        )

    if end < 0:

        end = len(
            cell111_final_text
        )

    case_block = (
        cell111_final_text[
            start:end
        ]
    )

    author_position = (
        case_block.rfind(
            "### نظر مؤلف (دکتر کمیل رودی)"
        )
    )

    cell111_check(
        f"FINAL_AUTHOR_SECTION_MISSING:{number}",
        author_position
        >=
        0,
    )

    if author_position < 0:

        continue

    author_tail = (
        case_block[
            author_position:
        ]
    )

    required_author_texts = [
        "☐ تأیید",
        "☐ نیاز به اصلاح",
        "☐ رد",
        "☐ باید دوباره سؤال ساخته شود",
        "توضیحات دکتر",
    ]

    missing = [
        text

        for text
        in required_author_texts

        if text
        not in
        author_tail
    ]

    cell111_check(
        f"FINAL_AUTHOR_STRUCTURE_MISSING:{number}",
        len(
            missing
        )
        ==
        0,
        missing,
    )


# ============================================================
# 27) Final Registry safety
# ============================================================

cell111_registry_sha256_final = (
    cell111_sha256_file(
        cell111_registry_path
    )
)


cell111_check(
    "FINAL_REGISTRY_SHA_CHANGED",
    cell111_registry_sha256_final
    ==
    cell111_registry_sha256_before,
    {
        "before":
            cell111_registry_sha256_before,

        "after":
            cell111_registry_sha256_final,
    },
)


# ============================================================
# 28) Final Gate
# ============================================================

cell111_gate = (
    len(
        cell111_failures
    )
    ==
    0
)


print(
    "=== BUILD FINAL 28-CASE AUTHOR REVIEW MARKDOWN ==="
)


print(
    "\n=== FROZEN INPUTS ==="
)


print(
    "Approved 22-case source:",
    str(
        cell111_source22_path
    ),
)


print(
    "Approved 22-case SHA256:",
    cell111_source22_sha256,
)


print(
    "Accepted Corpus SHA256:",
    cell111_corpus_sha256,
)


print(
    "Registry SHA256:",
    cell111_registry_sha256_before,
)


print(
    "\n=== CURRENT AUTHOR REVIEW STATE ==="
)


print(
    "Author Review queue:",
    len(
        cell111_author_cases
    ),
)


print(
    "Frozen approved cases:",
    22,
)


print(
    "New Source-reviewed cases:",
    6,
)


print(
    "Active AUTHOR_FEEDBACK_CONTENT cases:",
    cell111_active_content_count,
)


print(
    "Active MULTI_TURN_CONTEXT_CONTINUITY cases:",
    cell111_active_continuity_count,
)


print(
    "\n=== SOURCE REVIEW DECISIONS ==="
)


for number in (
    CELL111_NEW_NUMBERS
):

    review = (
        cell111_author_by_number[
            number
        ].get(
            "second_source_review",
            {}
        )
        or
        {}
    )

    print(
        (
            f"Pilot {number}: "
            f"{review.get('decision')} "
            f"| {review.get('revision_scope')} "
            f"| Sources: "
            f"{len(review.get('verified_corpus_sources', []) or [])}"
        )
    )


print(
    "\n=== AUTHOR REVIEW DOCUMENT ==="
)


print(
    "Output:",
    str(
        cell111_output_path
    ),
)


print(
    "Cases:",
    len(
        cell111_final_numbers
    ),
)


print(
    "Case sequence:",
    (
        f"{cell111_final_numbers[0]}"
        f"-"
        f"{cell111_final_numbers[-1]}"
    ),
)


print(
    "Cases 1-22 regenerated from Registry:",
    False,
)


print(
    "Cases 1-22 protected by frozen SHA:",
    True,
)


print(
    "Cases 23-28 include Phase-1 history:",
    True,
)


print(
    "Cases 23-28 include raw Author Feedback:",
    True,
)


print(
    "Cases 23-28 include verified Corpus evidence:",
    True,
)


print(
    "Cases 23-28 include Source Review decision:",
    True,
)


print(
    "New Author Review sections validated:",
    6,
)


print(
    "Pilot 25 old continuity interpretation preserved:",
    True,
)


print(
    "Pilot 25 active case is content:",
    True,
)


print(
    "Pilot 27 complete historical model answer preserved:",
    True,
)


print(
    "\n=== WRITE / SAFETY ==="
)


print(
    "Registry mutated:",
    False,
)


print(
    "Registry SHA256 after build:",
    cell111_registry_sha256_final,
)


print(
    "Author decisions recorded:",
    False,
)


print(
    "Author Review status for Pilots 23-28:",
    "NOT_STARTED",
)


print(
    "Output write required:",
    cell111_output_change_required,
)


print(
    "Output backup:",
    (
        str(
            cell111_output_backup_path
        )
        if cell111_output_backup_path
        is not None
        else
        "NOT_NEEDED"
    ),
)


print(
    "Output SHA256:",
    cell111_final_sha256,
)


print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)


print(
    cell111_failures
)


print(
    "\nCODE CELL 111 GATE:",
    (
        "PASS"
        if cell111_gate
        else
        "FAIL"
    ),
)


assert cell111_gate, (
    "Final 28-case Author Review Markdown build failed. "
    "Repair this SAME Code Cell 111 before Author Review."
)

=== BUILD FINAL 28-CASE AUTHOR REVIEW MARKDOWN ===

=== FROZEN INPUTS ===
Approved 22-case source: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/author_review_22_pilots_REVISED.md
Approved 22-case SHA256: 2351a6011387dbc1ea56a8f9ba5fe83c1f1f28d8a9ebb7bd8c262c1f1f27fcfc
Accepted Corpus SHA256: 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40
Registry SHA256: 10933065b1d4598dba1534964b8a6837f723cf79082d1af2bc8a4964e2a6766e

=== CURRENT AUTHOR REVIEW STATE ===
Author Review queue: 28
Frozen approved cases: 22
New Source-reviewed cases: 6
Active AUTHOR_FEEDBACK_CONTENT cases: 6
Active MULTI_TURN_CONTEXT_CONTINUITY cases: 0

=== SOURCE REVIEW DECISIONS ===
Pilot 23: ACCEPT | NONE | Sources: 1
Pilot 24: ACCEPT | NONE | Sources: 2
Pilot 25: REVISE | QUESTION_ANSWER_AND_PILOT_TYPE | Sources: 2
Pilot 26: REVISE | GROUNDING_ONLY | Sources: 2
Pilot 27: REVISE | GROUNDING_ONLY | Sources: 2
Pilot 28: ACCEPT | NONE | Sources: 1

=== AUTHOR